# Large-FOV Nuclear Pipeline v18.1 — Vulcan 2.5.3 baseline

Start with a fresh kernel and execute in order. Set configuration before run bootstrap.
The model SHA256 and complete configuration determine a new run directory; prior results remain available.

Inputs: five edge-replicated Z planes, each ordered NLS/NPC/membrane, globally normalized per timepoint.
Inference: Hann overlap blending and eight dihedral TTA views. All eight output heads are retained;
the four-channel probability TIFF is a compatibility view for existing analysis code.
Target planes: Z >= 6, with no focus-window cropping. Lower planes may supply input context.
Nuclei: edge-guided watershed; instance IDs survive measurement and mask recovery.
Z selection: mean predicted Z-offset minimum, with maximum-area comparison saved in QC.
Legacy NPC/membrane rejection and raw-image fragmentation repair are disabled by default.
Exceptions stop the run before committing the affected timepoint. Resume uses completed timepoints only.

The complete model output is `segmentation/vulcan_all_heads.tif` (actual directory follows the path resolver),
with channel names and Z-offset units in its JSON sidecar. Z-offset is in micrometres, not probability.
Old four-channel re-extraction is disabled for this baseline; run fresh inference.


## 1. Imports

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import time
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import shutil

from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage import filters, measure, morphology

try:
    import tensorflow as tf
except Exception:
    tf = None

try:
    import tifffile as tiff
except Exception:
    tiff = None

try:
    from sklearn.cluster import DBSCAN
except Exception:
    DBSCAN = None


## 2. Configuration

In [ ]:
def get_n_workers() -> int:
    """"Return the number of CPU Cores Avalible for Paralell Processing"""
    return int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))


@dataclass
class PipelineConfig:
    # ── Root paths ────────────────────────────────────────────────────────
    model_name: str = "Vulcan_2.5.3.keras"
    input_image_name: str = "control_extract_1.1.tif"
    image_subdir: str = "Condition/Control"
    # Vulcan 2.5.3 is a 2.5D model: five Z planes x three channels.
    n_z_context: int = 5
    normalization_mode: str = "global_t"
    norm_stats_stride: int = 4
    norm_stats_z_stride: int = 4

    # Vulcan baseline; set all result-affecting options BEFORE run bootstrap.
    pipeline_revision: str = "v18.1-vulcan253-contract1"
    model_sha256: str = ""
    input_fingerprint: str = ""
    tta: bool = True
    best_z_mode: str = "argmin_z_offset"
    watershed_core_thresh: float = 0.7
    watershed_edge_weight: float = 1.0
    watershed_min_marker_dist_px: int = 20
    sharpness_window_px: int = 7
    repair_enabled: bool = False
    repair_flatten_um: float = 8.0
    repair_npc_slack: float = 1.10
    repair_min_solidity: float = 0.93
    repair_area_deficit: float = 0.50
    repair_z_pad: int = 6
    repair_max_frac_droplet: float = 0.60
    repair_min_signed_margin: float = 1.15
    repair_edge_margin_um: float = 2.0
    repair_npc_min_rays: int = 36
    repair_win_um: float = 45.0

    # ── Imaging ───────────────────────────────────────────────────────────
    pixel_size_um: float = 0.1625
    z_step_um: float = 2.0
    nuclear_channel_index: int = 1
    membrane_channel_index: int = 0
    npc_channel_index: int = 2

    # ── Halo analysis ─────────────────────────────────────────────────────
    halo_step_px: int = 4
    n_halos: int = 4
    erosion_px: int = 4

    # ── Radial sweep (v16) ───────────────────────────────────────
    # Rays cast from each tracked nucleus centroid to its droplet edge,
    # sampling the membrane / NE channel. One row per ray sample point.
    radial_channel_index: int = 0        # membrane / NE channel (matches membrane_channel_index)
    radial_n_angles: int = 180
    radial_step_size_px: float = 1.0
    radial_exclude_nucleus: bool = True  # drop ray points falling inside the nucleus
    radial_max_ray_steps: int = 4000

    # ── Grouping / tracking ───────────────────────────────────────────────
    z_group_tolerance_um: float = 1.0
    time_track_tolerance_um: float = 5.0
    multi_nucleus_exclusion_um: float = 1.0

    track_dbscan_eps_um: float = 6.0
    track_dbscan_min_samples: int = 1
    track_crowded_distance_scale: float = 0.65
    track_area_log_weight: float = 12.0
    track_area_ratio_max: float = 5.0

    # ── Stitched acquisition layout ───────────────────────────────────────
    tile_rows: int = 2
    tile_cols: int = 3
    minutes_per_tile: float = 1.0
    serpentine_scan: bool = True

    # ── Segmentation ──────────────────────────────────────────────────────
    # ── v9 per-channel sigmoid extraction thresholds ──────────────────────
    # v9 head = 4 INDEPENDENT sigmoids. Each class thresholded on its OWN
    # channel, never argmax (classes overlap: nucleus subset of droplet, NPC
    # ring on the envelope). These are the per-experiment knobs to sweep.
    droplet_threshold: float = 0.5
    npc_threshold:     float = 0.5
    nucleus_threshold: float = 0.5
    # ── Re-extraction source (layer 3) ────────────────────────────────────
    # None -> run inference from input_image_name, write a probability hyperstack.
    # str  -> skip the model; load this probability hyperstack and re-extract
    #         masks at the current thresholds (cheap threshold sweeps). Points
    #         at any dataset's saved probability map.
    probability_hyperstack_input: Optional[str] = None
    min_nucleus_area_px: int = 0
    nucleus_hole_size_px: int = 10647
    nucleus_opening_radius_px: int = 5
    min_nucleus_circularity: float = 0.65
    batch_size: int = 8   # tune down if GPU OOM, raise to 16-32 if VRAM permits
    # ── Stage-2 gate: NPC-shell organisation + membrane co-localisation ───
    # PORT OF THE TRAINING-SET GATE (vulcan_training_1.1, sections 5b + 6).
    # Field names and defaults are copied so a diff between the two configs is
    # meaningful. Change them in BOTH places or the model and the pipeline stop
    # agreeing on what a nucleus is.
    #
    # NOTE ON WHAT THIS ACTUALLY TESTS. The generator gate is NPC + MEMBRANE,
    # not NLS + NPC. NLS enters upstream, as the detector that proposes the
    # nucleus (detect_nucleus_adaptive); in v16 the U-Net's nucleus channel
    # plays that role. The gate then asks the independent question "is there an
    # organised envelope around this thing?" — which is exactly what a droplet
    # cap fails. NLS enrichment is recorded as a diagnostic column but is NOT
    # part of the pass/fail decision, matching the generator.
    require_stage2_gate: bool = False

    # Skip the gate for t < this (None = gate every timepoint). The generator
    # sets generation_min_timepoint = 2 and never gates t = 0-1: too few puncta
    # for a shell fit and the NE probe is too faint to co-localise. Applying the
    # gate there will therefore reject nearly everything. Leaving this at None
    # means unverified early objects are dropped rather than emitted — set it to
    # 2 to pass t = 0-1 through ungated instead.
    gate_min_timepoint: Optional[int] = None

    # NPC puncta (RAW channel, nucleus-boundary anchored)
    npc_margin_px: int = 5            # annular half-width around the nucleus edge
    npc_std_mult: float = 2.0         # thresh = mean + k*std of RAW NPC in droplet

    # Shell organisation (primary test)
    gate_min_puncta: int = 8
    gate_shell_tol_px: float = 6.0
    gate_shell_min_inlier_frac: float = 0.45
    ransac_n_iter: int = 200

    # Membrane co-localisation (secondary test)
    gate_membrane_nn_radius_px: int = 6
    gate_membrane_min_coloc_frac: float = 0.30
    gate_membrane_k_std: float = 1.0

    # Droplet-relative area window, from the generator's nucleus detector
    # (_clean_and_gate). v16 only ever applied an ABSOLUTE min_nucleus_area_px,
    # so a whole-droplet false positive (frac ~ 1.0) and a merged multi-droplet
    # blob both sailed through. This ratio is calibration-free — it does not
    # move when pixel_size_um is finally settled.
    gate_use_droplet_area_fraction: bool = True
    nucleus_min_frac: float = 0.01
    nucleus_max_frac: float = 0.50

    # ── Beyond generator parity — READ THE VERIFICATION NOTE IN SECTION 7a ──
    # The ported shell test is close to vacuous. Its puncta zone is ALREADY a
    # 10 px annulus (dilate 5 / erode 5) and the RANSAC tolerance is 6 px, so
    # every suprathreshold pixel in the zone is an inlier to a circle at the
    # nucleus boundary by construction. On a synthetic droplet cap with a FLAT
    # NPC channel the ported gate passes 8/8 trials on Gaussian noise alone.
    # It does not reject caps.
    #
    # These two knobs restore the discrimination the gate was designed to have.
    # Both default to exact generator parity (no effect). Turn them on
    # deliberately, and mirror the change in vulcan_training_1.1.
    gate_puncta_min_size_px: int = 0                 # drop noise puncta; try 4-9
    gate_min_ring_contrast: Optional[float] = None   # ring/core raw NPC; try 1.15

    # Optional shell-radius sanity check, absent from the generator. The
    # generator records shell_r and never tests it, so a ring of puncta on the
    # DROPLET wall fits a circle and passes. Bounds are fractions of the parent
    # droplet's equivalent radius; measured r/R ~ 0.53. None = generator parity.
    gate_shell_r_frac_min: Optional[float] = None
    gate_shell_r_frac_max: Optional[float] = None

    # Safety valve: if more than this fraction of a plane's pixels exceed the
    # nucleus_threshold the plane is almost certainly a bad Z (model
    # over-fired on bright droplet walls / out-of-focus texture).  Skip mask
    # cleaning entirely and return an empty mask.  7 ms to check vs 20 min to clean.
    max_nucleus_foreground_fraction: float = 0.25
    use_mixed_precision: bool = False
    patch_size: int = 512
    patch_stride: int = 384
    background_class_index: int = 0
    droplet_class_index: int = 1
    npc_class_index:     int = 2
    nucleus_class_index: int = 3
    num_classes: int = 4
    # Four-channel compatibility view; all eight heads are also saved separately.
    # Project the four semantic masks used by this pipeline into canonical order:
    # background, droplet_interior, npc, nucleus_interior.
    model_output_channels: int = 8
    model_probability_output_indices: Tuple[int, ...] = (0, 1, 3, 4)
    model_outputs_logits: bool = True

    # ── Focus-aware Z selection ───────────────────────────────────────────
    use_focus_z_selection: bool = False
    focus_window_radius: int = 6
    focus_edge_z_exclusion: int = 0
    focus_channel_index: Optional[int] = None
    focus_metric: str = "object_based_nuclear"
    # Droplet-resistant focus scorer tuning (passed to focus_score_object_based_nuclear)
    focus_min_nucleus_area_um2: float = 20.0    # lower nucleus size bound in µm²
    focus_max_nucleus_area_um2: float = 800.0   # upper bound — set BELOW droplet area
    focus_threshold_k: float = 1.5             # mean + k*std threshold selectivity
    focus_sharpness_weight: float = 0.5        # 0=off, 0.5=moderate, 1.0=strong

    # ── v13: Z floor/ceiling — exclude coverslip artefacts ──────────────
    # Bright round objects near the coverslip produce high nuclear channel
    # focus scores even when no nucleus is present. Setting focus_min_z
    # to the first artefact-free plane eliminates these false peaks.
    # Must match focus_min_z / focus_max_z used during training patch generation.
    focus_min_z: int = 6        # exclude Z < focus_min_z from focus scoring
    focus_max_z: Optional[int] = None  # None = only upper edge exclusion applies

    # ── v13: Adaptive channel selection ─────────────────────────────────
    # Nucleus channel Laplacian peak (across all valid Z) must exceed this
    # floor to use nuclear-channel scoring. Below it: NPC-ring scoring.
    focus_nucleus_score_floor: float = 0.008
    # Above floor but below this threshold: contrast-based nuclear scoring.
    # Above this threshold: Laplacian nuclear scoring.
    focus_nucleus_laplacian_threshold: float = 0.012
    # NPC-ring Laplacian variance minimum for a Z plane to be accepted.
    focus_npc_ring_min_score: float = 0.001
    # Erosion radius (px) applied to droplet mask before NPC-ring scoring.
    # Should match droplet_focus_erosion_px used in training.
    focus_npc_erosion_px: int = 20

    # ── HPC ───────────────────────────────────────────────────────────────
    n_workers: Optional[int] = None

    def __post_init__(self):
        if self.n_workers is None:
            self.n_workers = min(get_n_workers(), 16)
        self.paths = None   # set by the path-resolver bootstrap cell

    # ── Unit conversions (derived) ────────────────────────────────────────
    @property
    def z_group_tolerance_px(self) -> float:
        return self.z_group_tolerance_um / self.pixel_size_um

    @property
    def time_track_tolerance_px(self) -> float:
        return self.time_track_tolerance_um / self.pixel_size_um

    @property
    def multi_nucleus_exclusion_px(self) -> float:
        return self.multi_nucleus_exclusion_um / self.pixel_size_um

    @property
    def track_dbscan_eps_px(self) -> float:
        return self.track_dbscan_eps_um / self.pixel_size_um

    @property
    def effective_focus_channel_index(self) -> int:
        return self.nuclear_channel_index if self.focus_channel_index is None else self.focus_channel_index

    # ── Path layer — delegates to the run-scoped resolver ────────────────
    # cfg.paths (a RunPaths) is set once by the bootstrap cell after the run is
    # created. Nothing is hard-coded; every output is isolated under
    # Runs/<run_id>/, which is what makes stale-pickle mixing impossible.
    def _require_paths(self):
        if getattr(self, "paths", None) is None:
            raise RuntimeError(
                "cfg.paths is not set — run the path-resolver bootstrap cell "
                "before touching any path.")
        return self.paths

    # Immutable inputs (data_root/Inputs, via the parent ProjectPaths)
    @property
    def model_path(self) -> Path:
        return self._require_paths().project.models_dir / self.model_name

    @property
    def input_image_path(self) -> Path:
        return (self._require_paths().project.raw_images_dir /
                self.image_subdir / self.input_image_name)

    # Run-scoped output directories
    @property
    def output_dir(self) -> Path:
        return self._require_paths().run_dir

    @property
    def seg_dir(self) -> Path:
        return self._require_paths().seg_dir

    @property
    def obj_dir(self) -> Path:
        return self._require_paths().obj_dir

    @property
    def track_dir(self) -> Path:
        return self._require_paths().track_dir

    @property
    def analysis_dir(self) -> Path:
        return self._require_paths().analysis_dir

    @property
    def qc_dir(self) -> Path:
        return self._require_paths().qc_dir

    @property
    def mask_tif_dir(self) -> Path:
        return self._require_paths().mask_tif_dir

    @property
    def exports_dir(self) -> Path:
        return self._require_paths().exports_dir

    # ── Segmentation artefact paths ───────────────────────────────────────
    @property
    def segmentation_index_path(self) -> Path:
        return self.seg_dir / "segmentation_index.pkl"

    @property
    def segmentation_class_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_class_hyperstack.tif"

    @property
    def segmentation_label_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_label_hyperstack.tif"

    @property
    def nucleus_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "nucleus_instance_hyperstack.tif"

    @property
    def droplet_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "droplet_instance_hyperstack.tif"

    @property
    def segmentation_probability_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_probability_hyperstack.tif"

    @property
    def probability_input_path(self) -> Optional[Path]:
        """Layer 3: if set, re-extract from this probability hyperstack instead of
        running inference. Any dataset's saved probability map can be pointed at."""
        if self.probability_hyperstack_input is None:
            return None
        return Path(self.probability_hyperstack_input)

    @property
    def segmentation_roi_table_path(self) -> Path:
        return self.seg_dir / "segmentation_roi_table.pkl"

    def to_serializable_dict(self) -> Dict[str, object]:
        return asdict(self)

    # location-only fields excluded from the results signature (robust whether
    # or not the fields were deleted above)
    _LOCATION_FIELDS = ("project_root", "model_subdir", "image_subdir", "output_subdir")

    def to_run_signature(self) -> Dict[str, object]:
        """Config subset that defines a run's RESULTS (not where they're stored).
        Hashed into the run id and checked by the stale-guard."""
        d = asdict(self)
        for k in self._LOCATION_FIELDS:
            d.pop(k, None)
        return d

    def __repr__(self) -> str:
        keys = [
            "model_name", "input_image_name",
            "pixel_size_um", "nuclear_channel_index", "membrane_channel_index",
            "z_group_tolerance_um", "time_track_tolerance_um",
            "multi_nucleus_exclusion_um", "track_dbscan_eps_um",
            "min_nucleus_area_px", "patch_size", "patch_stride",
            "n_workers", "use_mixed_precision",
        ]
        return "\n".join(f"{k}: {getattr(self, k)}" for k in keys)



Calibration is defined once in PipelineConfig: 0.1625 µm/pixel and 2.0 µm acquisition Z step.


## 2b. Path resolver + run bootstrap

In [ ]:
# =====================================================================
# Path resolver — single source of truth for every filesystem location.
# Reads the logical layout in STRUCTURE and materialises it under two
# physical roots. On Cheaha: code_root=$HOME/Projects (small, git),
# data_root=/data/user/$USER/<project> (5 TB, all inputs + outputs).
# Every run writes under Runs/<run_id>/, so a stale pickle from a
# different config can never be loaded by accident. (Mirrors structure.json.)
# =====================================================================
import getpass, hashlib, datetime as _dt
import json as _json
from pathlib import Path as _Path

STRUCTURE = {
    "code_root": {"Python": {"Environments": None,
                             "Model_Training": {"Training_Scripts": None},
                             "Image_Segmentation": None, "Image_Analysis": None},
                  "R_Projects": None},
    "data_root": {"Inputs": {"Raw_Images": None, "Models": None, "Training_Data": None},
                  "Runs": None, "Legacy": None},
    "run_template": {"segmentation": None, "objects": None, "tracking": None,
                     "analysis": None, "qc": None, "mask_tifs": None, "exports": None},
}

_RUN_STAGE_ATTRS = {"segmentation": "seg_dir", "objects": "obj_dir",
                    "tracking": "track_dir", "analysis": "analysis_dir",
                    "qc": "qc_dir", "mask_tifs": "mask_tif_dir",
                    "exports": "exports_dir"}


def _build_tree(base, spec):
    """Recursively create a directory tree under base according to the spec dict."""
    base.mkdir(parents=True, exist_ok=True)
    if not spec:
        return
    for name, child in spec.items():
        if str(name).startswith("_"):
            continue
        _build_tree(base / name, child)


class ProjectPaths:
    """Resolve all filesystem locations for a project, given two roots:"""
    def __init__(self, code_root, data_root, structure=STRUCTURE, create=True):
        self.spec = structure
        self.code_root = _Path(code_root).expanduser().resolve()
        self.data_root = _Path(data_root).expanduser().resolve()
        self.run_template = structure.get("run_template", {})
        self.raw_images_dir    = self.data_root / "Inputs" / "Raw_Images"
        self.models_dir        = self.data_root / "Inputs" / "Models"
        self.training_data_dir = self.data_root / "Inputs" / "Training_Data"
        self.runs_dir          = self.data_root / "Runs"
        self.legacy_dir        = self.data_root / "Legacy"
        if create:
            _build_tree(self.code_root, self.spec.get("code_root"))
            _build_tree(self.data_root, self.spec.get("data_root"))

    @classmethod
    def for_cheaha(cls, project_name, code_subdir="Projects", create=True):
        """Resolve paths for a Cheaha HPC. The code root is $HOME/<code_subdir>,"""
        user = getpass.getuser()
        home = _Path(os.environ.get("HOME", f"/home/{user}"))
        user_data = _Path(os.environ.get("USER_DATA", f"/data/user/{user}")).expanduser()
        return cls(home / code_subdir, user_data / project_name, create=create)

    @staticmethod
    def make_run_id(label, config_dict, timestamp=False):
        """Stable per config: <label>__cfg-<hash>. Same config -> same dir, so
        RUN_*=False loads reuse the existing run. Any config change flips the
        hash -> a new dir -> no stale mixing. Set timestamp=True to force a
        distinct dir for same-config reruns."""
        h = hashlib.sha1(
            _json.dumps(config_dict, sort_keys=True, default=str).encode()).hexdigest()[:8]
        safe = "".join(c if (c.isalnum() or c in "-._") else "_" for c in str(label))
        if timestamp:
            return f"{safe}__{_dt.datetime.now():%Y%m%d_%H%M}__cfg-{h}"
        return f"{safe}__cfg-{h}"

    def for_run(self, run_id, create=True):
        run_dir = self.runs_dir / run_id
        if create:
            _build_tree(run_dir, self.run_template)
        return RunPaths(self, run_id, run_dir)

    def list_runs(self):
        if not self.runs_dir.exists():
            return []
        return sorted(p.name for p in self.runs_dir.iterdir() if p.is_dir())


class RunPaths:
    def __init__(self, project, run_id, run_dir):
        self.project, self.run_id, self.run_dir = project, run_id, run_dir
        for stage, attr in _RUN_STAGE_ATTRS.items():
            setattr(self, attr, run_dir / stage)
        self.config_snapshot_path = run_dir / "config.json"
        self.manifest_path = run_dir / "manifest.json"

    def snapshot_config(self, config_dict):
        self.config_snapshot_path.write_text(
            _json.dumps(config_dict, indent=2, sort_keys=True, default=str))

    def load_snapshot(self):
        if self.config_snapshot_path.exists():
            return _json.loads(self.config_snapshot_path.read_text())
        return None

    def assert_config_matches(self, config_dict, strict=True):
        """Refuse to trust cached stages if the live config drifted from the
        snapshot that produced this run's data. Call before any RUN_*=False load."""
        config_dict = _json.loads(_json.dumps(config_dict, default=str))
        snap = self.load_snapshot()
        if snap is None:
            return
        diffs = {k: (snap.get(k), config_dict.get(k))
                 for k in set(snap) | set(config_dict)
                 if snap.get(k) != config_dict.get(k)}
        if diffs:
            msg = ("[stale-guard] live config differs from snapshot in run '%s':\n%s"
                   % (self.run_id, "\n".join(f"    {k}: snapshot={s!r} live={l!r}"
                                             for k, (s, l) in diffs.items())))
            if strict:
                raise RuntimeError(msg)
            print("WARNING:", msg)

    def write_manifest(self, **fields):
        m = {}
        if self.manifest_path.exists():
            m = _json.loads(self.manifest_path.read_text())
        m.update(fields)
        m["updated_at"] = _dt.datetime.now().isoformat(timespec="seconds")
        self.manifest_path.write_text(_json.dumps(m, indent=2, default=str))


In [ ]:
# Configure all options before bootstrap; fingerprint the actual model bytes.
import hashlib
cfg = PipelineConfig()
paths = ProjectPaths.for_cheaha(project_name="Nuclear_Scaling")
_model_path = paths.models_dir / cfg.model_name
_image_path = paths.raw_images_dir / cfg.image_subdir / cfg.input_image_name
_sha = hashlib.sha256()
with _model_path.open("rb") as _stream:
    for _block in iter(lambda: _stream.read(8 * 1024**2), b""):
        _sha.update(_block)
cfg.model_sha256 = _sha.hexdigest()
_st = _image_path.stat()
cfg.input_fingerprint = f"{_image_path.resolve()}:{_st.st_size}:{_st.st_mtime_ns}"
run = paths.for_run(ProjectPaths.make_run_id(
    label=Path(cfg.input_image_name).stem, config_dict=cfg.to_run_signature()))
cfg.paths = run
run.assert_config_matches(cfg.to_run_signature())
run.snapshot_config(cfg.to_run_signature())
run.write_manifest(input_image=cfg.input_image_name, model=cfg.model_name,
                   model_sha256=cfg.model_sha256, run_id=run.run_id)
print("Run:", run.run_dir)
print("Model SHA256:", cfg.model_sha256)
cfg


In [ ]:
# Calibration and optional repair settings are now dataclass fields above bootstrap.
print(f"Pixel size={cfg.pixel_size_um} um; Z step={cfg.z_step_um} um; repair={cfg.repair_enabled}")


## 3. Save / load configuration

In [ ]:
def save_config(config: PipelineConfig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))


def load_config(in_path: Path) -> PipelineConfig:
    data = json.loads(in_path.read_text())
    return PipelineConfig(**data)


save_config(cfg, cfg.output_dir / "pipeline_config.json")
print("Configuration saved.")


## 4. GPU setup

In [ ]:
def configure_tensorflow_for_gpu(config: PipelineConfig) -> None:
    if tf is None:
        print("TensorFlow not available — skipping GPU setup.")
        return

    gpus = tf.config.list_physical_devices("GPU")
    print("GPUs found:", gpus)

    if getattr(config, "use_mixed_precision", False):
        try:
            from tensorflow.keras import mixed_precision
            mixed_precision.set_global_policy("mixed_float16")
            print("Mixed precision enabled.")
        except Exception as exc:
            print("Could not enable mixed precision:", exc)

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print("Could not set memory growth:", exc)


## 5. Image loading helpers

In [ ]:
def load_image_5d(image_path: Union[str, Path]) -> np.ndarray:
    if tiff is None:
        raise ImportError("tifffile is required to load TIFF data.")
    arr = tiff.imread(image_path)
    print("Loaded image shape:", arr.shape)
    return arr


def get_nuclear_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.nuclear_channel_index]


def get_membrane_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.membrane_channel_index]


def get_full_plane_yxc(
    img_5d: np.ndarray,
    t_idx: int,
    z_idx: int,
    config: "PipelineConfig" = None,
) -> np.ndarray:
    """
    Return a 3-channel plane in (Y, X, C) format ordered to match training input.

    Training channel order: (NLS/nucleus, NPC, Membrane)
    Hyperstack channel order: Ch0=Membrane, Ch1=NLS, Ch2=NPC

    If config is provided, uses config channel indices for explicit ordering.
    Falls back to default training order (1, 2, 0) if config is None.
    """
    plane_cyx = img_5d[t_idx, z_idx]  # shape: (C, Y, X)
    if config is not None:
        ch_order = [
            config.nuclear_channel_index,   # NLS first  (training ch0)
            config.npc_channel_index,        # NPC second (training ch1)
            config.membrane_channel_index,   # Mem third  (training ch2)
        ]
    else:
        ch_order = [1, 2, 0]  # default: NLS, NPC, Membrane
    return np.moveaxis(plane_cyx[ch_order], 0, -1)  # (Y, X, C)

def context_plane_indices(z_idx: int, n_z: int, n_context: int) -> List[int]:
    """Training-faithful Z context, edge-replicated at stack boundaries."""
    if n_context < 1 or n_context % 2 == 0:
        raise ValueError(f"n_z_context must be a positive odd integer, got {n_context}")
    half = n_context // 2
    return [int(np.clip(z_idx + dz, 0, n_z - 1))
            for dz in range(-half, half + 1)]


def get_full_context_yxc(
    img_5d: np.ndarray,
    t_idx: int,
    z_idx: int,
    config: "PipelineConfig",
) -> np.ndarray:
    """Return plane-major 2.5D input: [NLS, NPC, Mem] for each context Z."""
    planes = [get_full_plane_yxc(img_5d, t_idx, zz, config)
              for zz in context_plane_indices(z_idx, img_5d.shape[1],
                                              config.n_z_context)]
    return np.concatenate(planes, axis=-1)


def normalization_stats_for_timepoint(
    img_5d: np.ndarray,
    t_idx: int,
    config: "PipelineConfig",
) -> Optional[List[Tuple[float, float]]]:
    """Training-faithful p1/p99.8 stats pooled across eligible Z planes.

    Returns stats in the same plane-major channel order as get_full_context_yxc.
    With per_patch mode, returns None and _preprocess_patch computes each channel.
    """
    if config.normalization_mode == "per_patch":
        return None
    if config.normalization_mode != "global_t":
        raise ValueError(f"Unsupported normalization_mode: {config.normalization_mode!r}")
    n_z = img_5d.shape[1]
    z0 = max(0, int(config.focus_min_z or 0))
    zs = list(range(z0, n_z, int(config.norm_stats_z_stride))) or [n_z // 2]
    sy = sx = max(1, int(config.norm_stats_stride))
    base_order = [config.nuclear_channel_index, config.npc_channel_index,
                  config.membrane_channel_index]
    base_stats = []
    for channel_idx in base_order:
        pool = np.concatenate([
            np.asarray(img_5d[t_idx, z, channel_idx, ::sy, ::sx],
                       dtype=np.float32).ravel()
            for z in zs])
        lo, hi = np.percentile(pool, [1.0, 99.8])
        base_stats.append((float(lo), float(hi)))
    return base_stats * int(config.n_z_context)



## 6. Optional focus-scoring utilities

QC only for this baseline; target planes are all Z >= 6.


In [ ]:
def _normalize_2d(img2d: np.ndarray) -> np.ndarray:
    """Normalise a 2-D plane to [0, 1] for focus scoring."""
    arr = np.asarray(img2d, dtype=np.float32)
    lo, hi = np.nanmin(arr), np.nanmax(arr)
    if not (np.isfinite(lo) and np.isfinite(hi) and hi > lo):
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)


def _area_um2_to_px(area_um2: float, pixel_size_um: float) -> int:
    """Convert an area in µm² to pixels²"""
    return max(1, int(np.ceil(area_um2 / pixel_size_um ** 2)))


def focus_score_variance_laplacian(img2d: np.ndarray) -> float:
    """Whole-plane variance-of-Laplacian (fallback / debug)."""
    return float(np.nanvar(ndi.laplace(_normalize_2d(img2d))))


def focus_score_nucleus_weighted_laplacian(
    img2d: np.ndarray,
    candidate_percentile: float = 97.5,
    min_candidate_pixels: int = 50,
) -> float:
    """Bright-pixel-weighted Laplacian focus score (retained for comparison)."""
    img_f = _normalize_2d(img2d)
    finite = np.isfinite(img_f)
    if not finite.any():
        return 0.0
    cutoff = np.nanpercentile(img_f[finite], candidate_percentile)
    candidate_mask = (img_f >= cutoff) & finite
    if int(candidate_mask.sum()) < min_candidate_pixels:
        return 0.25 * focus_score_variance_laplacian(img2d)
    lap = ndi.laplace(img_f)
    weights = np.maximum(img_f[candidate_mask], 1e-6)
    weighted_energy = np.average(lap[candidate_mask] ** 2, weights=weights)
    signal_bonus = np.sqrt(max(float(candidate_mask.mean()), 1e-8))
    return float(weighted_energy * signal_bonus)


def focus_score_hybrid_nuclear_laplacian(
    img2d: np.ndarray,
    global_weight: float = 0.2,
    nuclear_weight: float = 0.8,
) -> float:
    """Hybrid score (retained for comparison / debugging)."""
    return float(
        global_weight * focus_score_variance_laplacian(img2d)
        + nuclear_weight * focus_score_nucleus_weighted_laplacian(img2d)
    )


def focus_score_object_based_nuclear(
    img2d: np.ndarray,
    pixel_size_um: float,
    min_nucleus_area_um2: float = 20.0,
    max_nucleus_area_um2: float = 500.0,
    blur_sigma: float = 1.5,
    threshold_k: float = 1.5,
    min_signal_fraction: float = 0.005,
    sharpness_weight: float = 0.5,
    min_object_count: int = 1,
) -> float:
    """Object-based nuclear focus score — droplet-resistant version."""
    nuc = _normalize_2d(img2d)
    if np.nanmax(nuc) <= 0:
        return 0.0
    mean_val = float(np.nanmean(nuc))
    if float((nuc > mean_val).mean()) < min_signal_fraction:
        return 0.0
    nuc_blur = filters.gaussian(nuc, sigma=blur_sigma, preserve_range=True)
    thr = mean_val + threshold_k * float(np.nanstd(nuc_blur))
    if thr >= 1.0:
        return 0.0
    min_area_px = _area_um2_to_px(min_nucleus_area_um2, pixel_size_um)
    max_area_px = _area_um2_to_px(max_nucleus_area_um2, pixel_size_um)
    mask = morphology.remove_small_objects(
        (nuc_blur > thr).astype(bool), min_size=min_area_px)
    lap = ndi.laplace(nuc)
    plane_lap_var = float(np.var(lap)) + 1e-12
    labeled = measure.label(mask)
    score = 0.0
    valid_count = 0
    for p in measure.regionprops(labeled, intensity_image=nuc):
        if p.area < min_area_px or p.area > max_area_px:
            continue
        roundness_weight = max(0.05, 1.0 - float(p.eccentricity))
        solidity_weight  = max(0.05, float(getattr(p, "solidity", 1.0)))
        base = float(p.area) * float(p.mean_intensity) * roundness_weight * solidity_weight
        if sharpness_weight > 0:
            obj_pixels = lap[labeled == p.label]
            sharpness = float(np.var(obj_pixels)) / plane_lap_var
            combined_weight = 1.0 + sharpness_weight * sharpness
        else:
            combined_weight = 1.0
        score += base * combined_weight
        valid_count += 1
    return float(score) if valid_count >= min_object_count else 0.0


_FOCUS_METRICS = {
    "object_based_nuclear":       lambda img, pxum: focus_score_object_based_nuclear(img, pixel_size_um=pxum),
    "variance_laplacian":         lambda img, pxum: focus_score_variance_laplacian(img),
    "nucleus_weighted_laplacian": lambda img, pxum: focus_score_nucleus_weighted_laplacian(img),
    "hybrid_nuclear_laplacian":   lambda img, pxum: focus_score_hybrid_nuclear_laplacian(img),
}


# ── v13: NPC-ring and adaptive focus scoring ──────────────────────────────

def focus_score_npc_ring(
    npc2d: np.ndarray,
    droplet_mask: Optional[np.ndarray] = None,
    erosion_px: int = 20,
) -> float:
    """
    Laplacian-variance focus score on the NPC boundary ring.
    If droplet_mask provided, restricts scoring to the annular ring region.
    Falls back to whole-plane Laplacian if no mask or ring is empty.
    """
    npc_f = _normalize_2d(npc2d)
    if droplet_mask is None:
        return float(np.nanvar(ndi.laplace(npc_f)))
    dist = ndi.distance_transform_edt(droplet_mask.astype(bool))
    interior = dist >= erosion_px
    ring = droplet_mask.astype(bool) & ~interior
    if not ring.any():
        return float(np.nanvar(ndi.laplace(npc_f)))
    lap_sq = ndi.laplace(npc_f) ** 2
    return float(np.mean(lap_sq[ring]))


def _z_in_focus_range(z: int, n_z: int, config: "PipelineConfig") -> bool:
    """Combined Z validity check: edge exclusion + floor + ceiling."""
    excl = config.focus_edge_z_exclusion
    if excl > 0 and (z < excl or z >= n_z - excl):
        return False
    if config.focus_min_z is not None and z < config.focus_min_z:
        return False
    if config.focus_max_z is not None and z > config.focus_max_z:
        return False
    return True


def _focus_channel_for_timepoint(
    img_5d: np.ndarray,
    t: int,
    config: "PipelineConfig",
    valid_zs: List[int],
) -> str:
    """
    Three-way adaptive channel selection for one timepoint.

    Tests the nuclear channel Laplacian peak across valid Z planes:
      peak < focus_nucleus_score_floor       -> 'npc_ring'
      floor <= peak < laplacian_threshold    -> 'nucleus_contrast'
      peak >= laplacian_threshold            -> 'nucleus_laplacian'
    """
    peak_scores = []
    for z in valid_zs:
        nuc2d = img_5d[t, z, config.nuclear_channel_index].astype(np.float32)
        nuc_f = _normalize_2d(nuc2d)
        lap_sq = ndi.laplace(nuc_f) ** 2
        peak_scores.append(float(np.mean(lap_sq)))

    peak_lap = float(np.max(peak_scores)) if peak_scores else 0.0

    if peak_lap < config.focus_nucleus_score_floor:
        return 'npc_ring'
    elif peak_lap < config.focus_nucleus_laplacian_threshold:
        return 'nucleus_contrast'
    else:
        return 'nucleus_laplacian'


def get_best_focus_z_indices_adaptive(
    img_5d: np.ndarray,
    t: int,
    config: "PipelineConfig",
) -> Tuple[List[int], np.ndarray, Optional[int], str]:
    """
    NPC-ring focus scoring with droplet-mask restriction.
    Universal across all timepoints — NPC ring is reliably sharp at the
    correct focal plane regardless of nuclear development stage.
    Reference droplet mask is computed from the Z with the highest droplet
    count in the valid range (matching training notebook get_reference_z logic).
    Returns (keep_z, scores, best_z, channel).
    """
    from skimage.filters import threshold_local

    n_z = img_5d.shape[1]
    valid_zs = [z for z in range(n_z) if _z_in_focus_range(z, n_z, config)]
    scores = np.full(n_z, np.nan, dtype=np.float32)
    channel = 'npc_ring'

    if not valid_zs:
        return [], scores, None, channel

    # Find reference Z with highest droplet count in valid range
    # Mirrors get_reference_z from training notebook
    best_ref_z = valid_zs[len(valid_zs) // 2]  # fallback
    best_ref_count = 0
    for z in valid_zs:
        npc_z = _normalize_2d(img_5d[t, z, config.npc_channel_index])
        npc_blur_z = filters.gaussian(npc_z, sigma=8.0, preserve_range=True)
        local_thresh_z = threshold_local(npc_blur_z, block_size=101, offset=-0.02)
        mask_z = morphology.remove_small_objects(
            (npc_blur_z > local_thresh_z).astype(bool), min_size=4000)
        n_droplets = len(np.unique(measure.label(mask_z))) - 1
        if n_droplets > best_ref_count:
            best_ref_count = n_droplets
            best_ref_z = z

    # Build reference droplet mask at the best-count Z
    npc_ref = _normalize_2d(img_5d[t, best_ref_z, config.npc_channel_index])
    npc_blur = filters.gaussian(npc_ref, sigma=8.0, preserve_range=True)
    local_thresh = threshold_local(npc_blur, block_size=101, offset=-0.02)
    droplet_mask_ref = morphology.remove_small_objects(
        (npc_blur > local_thresh).astype(bool), min_size=4000)

    print(f"    ref_z={best_ref_z} ({best_ref_count} droplets)")

    # Score each valid Z using ring-restricted NPC Laplacian
    for z in valid_zs:
        npc2d = img_5d[t, z, config.npc_channel_index]
        scores[z] = focus_score_npc_ring(
            npc2d,
            droplet_mask=droplet_mask_ref,
            erosion_px=config.focus_npc_erosion_px)

    # Apply minimum score threshold
    scores = np.where(
        scores < config.focus_npc_ring_min_score,
        np.nan, scores)

    if np.all(np.isnan(scores)):
        return [], scores, None, channel

    best_z = int(np.nanargmax(scores))
    excl = config.focus_edge_z_exclusion
    z_start = max(excl, best_z - config.focus_window_radius)
    z_stop  = min(n_z - excl, best_z + config.focus_window_radius + 1)
    keep_z  = [z for z in range(z_start, z_stop)
               if _z_in_focus_range(z, n_z, config)]
    return keep_z, scores, best_z, channel

def focus_score_plane(img2d: np.ndarray, metric: str = "object_based_nuclear",
                      pixel_size_um: float = 0.108,
                      config: Optional["PipelineConfig"] = None) -> float:
    """Dispatch focus metric."""
    if metric not in _FOCUS_METRICS:
        raise ValueError(f"Unsupported focus metric: {metric!r}. Choose from {list(_FOCUS_METRICS)}")
    if metric == "object_based_nuclear" and config is not None:
        return focus_score_object_based_nuclear(
            img2d, pixel_size_um=pixel_size_um,
            min_nucleus_area_um2=config.focus_min_nucleus_area_um2,
            max_nucleus_area_um2=config.focus_max_nucleus_area_um2,
            threshold_k=config.focus_threshold_k,
            sharpness_weight=config.focus_sharpness_weight,
        )
    return _FOCUS_METRICS[metric](img2d, pixel_size_um)


def get_focus_scores_for_timepoint(
    img_5d: np.ndarray,
    t: int,
    nuc_channel_idx: int,
    exclude_edge_z: int = 2,
    metric: str = "object_based_nuclear",
    pixel_size_um: float = 0.108,
    config: Optional["PipelineConfig"] = None,
) -> np.ndarray:
    """Score every Z plane; excluded planes set to NaN. Respects focus_min_z/max_z."""
    n_z = img_5d.shape[1]
    scores = np.full(n_z, np.nan, dtype=np.float32)
    for z in range(n_z):
        if config is not None:
            if not _z_in_focus_range(z, n_z, config):
                continue
        elif z < exclude_edge_z or z >= n_z - exclude_edge_z:
            continue
        scores[z] = focus_score_plane(img_5d[t, z, nuc_channel_idx],
                                      metric, pixel_size_um, config=config)
    return scores


def get_best_focus_z_indices(
    img_5d: np.ndarray,
    t: int,
    nuc_channel_idx: int,
    exclude_edge_z: int = 2,
    window_radius: int = 1,
    metric: str = "object_based_nuclear",
    pixel_size_um: float = 0.108,
    config: Optional["PipelineConfig"] = None,
) -> Tuple[List[int], np.ndarray, Optional[int]]:
    """Return (keep_z, scores, best_z) for the best-focus window at time t."""
    scores = get_focus_scores_for_timepoint(
        img_5d, t, nuc_channel_idx, exclude_edge_z, metric, pixel_size_um,
        config=config)
    if np.all(np.isnan(scores)):
        return [], scores, None
    best_z = int(np.nanargmax(scores))
    z_start = max(exclude_edge_z, best_z - window_radius)
    z_stop  = min(len(scores) - exclude_edge_z, best_z + window_radius + 1)
    n_z = len(scores)
    if config is not None:
        keep_z = [z for z in range(z_start, z_stop)
                  if _z_in_focus_range(z, n_z, config)]
    else:
        keep_z = list(range(z_start, z_stop))
    return keep_z, scores, best_z


def plot_focus_scores_for_timepoint(img_5d: np.ndarray, t: int,
                                    config: PipelineConfig) -> Tuple:
    """Quick QC plot of focus scores across Z for one time point."""
    keep_z, scores, best_z = get_best_focus_z_indices(
        img_5d, t,
        nuc_channel_idx=config.effective_focus_channel_index,
        exclude_edge_z=config.focus_edge_z_exclusion,
        window_radius=config.focus_window_radius,
        metric=config.focus_metric,
        pixel_size_um=config.pixel_size_um,
        config=config,
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(len(scores)), scores, marker="o")
    if best_z is not None:
        ax.axvline(best_z, linestyle="--", label=f"best z={best_z}")
    for z in keep_z:
        ax.axvspan(z - 0.45, z + 0.45, alpha=0.2)
    ax.set_xlabel("Z index")
    ax.set_ylabel("Focus score")
    ax.set_title(f"Focus scores t={t}  keep_z={keep_z}  metric={config.focus_metric}")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return keep_z, scores, best_z

## 7a. Optional legacy NPC/membrane gate

Retained for explicit experiments, disabled by default. These are Vulcan 1.1-era criteria and are not required by Vulcan 2.5.3.


## 7. Segmentation helpers

In [ ]:
def load_unet_model(model_path: Union[str, Path]):
    if tf is None:
        raise ImportError("TensorFlow is not available.")
    return tf.keras.models.load_model(model_path, compile=False)


def normalize_channel(
    ch2d: np.ndarray,
    stats: Optional[Tuple[float, float]] = None,
) -> np.ndarray:
    """Training-faithful p1/p99.8 normalization, optionally with fixed stats."""
    ch = ch2d.astype(np.float32)
    lo, hi = np.percentile(ch, [1.0, 99.8]) if stats is None else stats
    if hi <= lo:
        return np.zeros_like(ch)
    return np.clip((ch - lo) / (hi - lo), 0.0, 1.0)

def _preprocess_patch(
    patch_yxc: np.ndarray,
    channel_stats: Optional[List[Tuple[float, float]]] = None,
) -> np.ndarray:
    """Normalize every input channel with the same contract used in training.

    Vulcan 2.5.3 receives 15 plane-major channels: [NLS, NPC, Membrane]
    repeated over five Z planes. Fixed global_t statistics preserve relative
    intensity across Z; per_patch normalization remains supported.
    """
    x = patch_yxc.astype(np.float32)
    if channel_stats is not None and len(channel_stats) != x.shape[-1]:
        raise ValueError(
            f"normalization stats have {len(channel_stats)} channels, "
            f"but the patch has {x.shape[-1]}")
    return np.stack([
        normalize_channel(x[..., c], None if channel_stats is None else channel_stats[c])
        for c in range(x.shape[-1])
    ], axis=-1)

def collapse_label_map(droplet: np.ndarray, npc: np.ndarray,
                       nucleus: np.ndarray, config: PipelineConfig) -> np.ndarray:
    """Single-int DISPLAY label map. Priority Nucleus>NPC>Droplet>BG (v9
    collapse_to_integer_hwc). Display/QC only -- never an extraction source."""
    out = np.zeros(droplet.shape, dtype=np.uint8)
    out[droplet] = config.droplet_class_index
    out[npc]     = config.npc_class_index
    out[nucleus] = config.nucleus_class_index   # last write wins = highest priority
    return out


def extract_class_masks(class_prob_map: np.ndarray, config: PipelineConfig):
    """
    v9 contract: 4 INDEPENDENT sigmoid channels, each thresholded on its OWN
    channel. No argmax -- argmax forces mutual exclusivity and suppresses the
    faint NPC ring everywhere nucleus/droplet also fire.

    Returns (background, droplet, npc, nucleus_raw, label_map):
      droplet     = prob[...,1] > droplet_threshold, unioned with nucleus for a
                    SOLID FILLED disk (nucleus footprint included, as droplet|=nucleus
                    trained it). Union avoids fill_holes (documented segfault risk).
      npc         = prob[...,2] > npc_threshold       (envelope ring)
      nucleus_raw = prob[...,3] > nucleus_threshold   (caller cleans separately)
      background  = NOT droplet
      label_map   = display-only collapse (Nucleus>NPC>Droplet>BG)
    """
    droplet = class_prob_map[..., config.droplet_class_index] > config.droplet_threshold
    npc     = class_prob_map[..., config.npc_class_index]     > config.npc_threshold
    nucleus = class_prob_map[..., config.nucleus_class_index] > config.nucleus_threshold

    droplet = droplet | nucleus          # solid filled; nucleus subset of droplet
    background = ~droplet

    label_map = collapse_label_map(droplet, npc, nucleus, config)
    return background, droplet, npc, nucleus, label_map


def _generate_patch_starts(length: int, patch_size: int, stride: int) -> List[int]:
    """Generate patch start positions that always include the image end."""
    starts = list(range(0, max(length - patch_size + 1, 1), stride)) or [0]
    last = max(0, length - patch_size)
    if starts[-1] != last:
        starts.append(last)
    return sorted(set(starts))


def extract_overlapping_patches(
    image_yxc: np.ndarray,
    patch_size: int = 512,
    stride: int = 256,
) -> Tuple[List[np.ndarray], List[Tuple[int, int]]]:
    h, w, _ = image_yxc.shape
    y_starts = _generate_patch_starts(h, patch_size, stride)
    x_starts = _generate_patch_starts(w, patch_size, stride)
    patches, coords = [], []
    for y0 in y_starts:
        for x0 in x_starts:
            patches.append(image_yxc[y0:y0 + patch_size, x0:x0 + patch_size, :])
            coords.append((y0, x0))
    return patches, coords


def _project_model_probabilities(
    raw_predictions: np.ndarray,
    config: PipelineConfig,
) -> np.ndarray:
    """Convert Vulcan outputs to four canonical probability channels.

    Vulcan 2.5.3 emits seven mask logits plus a z_offset regression value.
    Only background, droplet interior, NPC and nucleus interior are consumed by
    this pipeline; edge/equatorial heads and z_offset remain model diagnostics.
    """
    raw = np.asarray(raw_predictions, dtype=np.float32)
    if raw.ndim != 4:
        raise ValueError(f"Unexpected prediction shape: {raw.shape}")
    expected = int(config.model_output_channels)
    if raw.shape[-1] != expected:
        raise ValueError(
            f"Model returned {raw.shape[-1]} outputs, expected {expected} for "
            "Vulcan 2.5.3 (7 mask logits + z_offset).")
    indices = tuple(int(i) for i in config.model_probability_output_indices)
    if len(indices) != int(config.num_classes) or max(indices) >= raw.shape[-1]:
        raise ValueError(
            f"Invalid model_probability_output_indices={indices} for "
            f"output shape {raw.shape} and num_classes={config.num_classes}")
    selected = raw[..., list(indices)]
    if config.model_outputs_logits:
        selected = 1.0 / (1.0 + np.exp(-np.clip(selected, -80.0, 80.0)))
    return selected.astype(np.float32)


def _predict_batch(
    model, batch_patches: np.ndarray, config: PipelineConfig
) -> np.ndarray:
    """
    Use model(x, training=False) instead of model.predict().
    model.predict() spins up a fresh TF dataset pipeline and progress-bar
    machinery on every call — at batch_size=1 this overhead dominates.
    Direct __call__ avoids all of that and runs the graph immediately.
    """
    raw = model(batch_patches, training=False).numpy()
    return _project_model_probabilities(raw, config)


def stitch_probability_patches(
    patch_probs: List[np.ndarray],
    coords: List[Tuple[int, int]],
    image_shape: Tuple[int, int],
    n_classes: int,
    patch_size: int = 512,
) -> np.ndarray:
    """
    Average overlapping patch probabilities into a full probability map (H, W, C).

    Accepts either a list of 2-D arrays or the stacked (N, pH, pW, C) array
    returned directly from the inference loop.
    """
    h, w = image_shape
    prob_sum   = np.zeros((h, w, n_classes), dtype=np.float32)
    prob_count = np.zeros((h, w),            dtype=np.float32)

    # Accept pre-stacked array or list
    probs_arr = (np.asarray(patch_probs) if not isinstance(patch_probs, np.ndarray)
                 else patch_probs)

    for idx, (y0, x0) in enumerate(coords):
        y1, x1 = y0 + patch_size, x0 + patch_size
        prob_sum  [y0:y1, x0:x1] += probs_arr[idx]
        prob_count[y0:y1, x0:x1] += 1.0

    return prob_sum / np.maximum(prob_count[:, :, np.newaxis], 1.0)


def _filter_by_area_and_circularity(
    binary_mask: np.ndarray,
    min_area_px: int = 50,
    min_circularity: float = 0.45,
) -> np.ndarray:
    """
    Filter objects by area and shape compactness.

    Uses SOLIDITY (area / convex_hull_area) instead of perimeter-based
    circularity.  Solidity is computed from the convex hull — a pure geometric
    operation with no C-level binary erosion — so it cannot segfault on large
    or irregular objects.  Nuclei are compact (solidity > 0.7); noise fragments,
    droplet-wall arcs, and merged blobs are not.

    The min_circularity parameter is reused as the solidity threshold so no
    config changes are needed (valid range and typical values are similar).
    """
    labeled = measure.label(binary_mask, connectivity=2)
    if labeled.max() == 0:
        return np.zeros_like(binary_mask, dtype=bool)

    tbl = measure.regionprops_table(
        labeled, properties=("label", "area", "solidity"))

    areas     = np.array(tbl["area"],     dtype=np.float64)
    solidities = np.array(tbl["solidity"], dtype=np.float64)
    labels    = np.array(tbl["label"],    dtype=np.int32)

    keep_mask   = (areas >= min_area_px) & (solidities >= min_circularity)
    keep_labels = labels[keep_mask]

    out = np.isin(labeled, keep_labels)
    return out


def clean_nucleus_mask(
    nucleus_mask: np.ndarray,
    min_size_px: int = 50,
    hole_size_px: int = 200,
    opening_radius: int = 1,
    min_circularity: float = 0.45,
    max_foreground_fraction: float = 0.25,
) -> np.ndarray:
    """
    Clean a raw nucleus probability mask.

    Operation order is intentional:
      1. Foreground-fraction guard  — bail out immediately on over-fired planes.
         A plane where >25% of pixels are foreground is almost always a bad Z
         (model firing on droplet walls / texture).  Returning empty here costs
         7 ms vs up to 20 min of morphology + regionprops on a near-solid mask.
      2. remove_small_objects first — reduces object count before any fill_holes,
         so subsequent operations work on a sparser mask.
      3. opening before fill_holes — breaks merged blobs before filling gaps,
         preventing fill_holes from joining large connected regions.
      4. fill_holes on the reduced mask — much cheaper than on the raw mask.
      5. Perimeter-based circularity filter last — only computed on surviving
         objects after area filtering, not on every initial fragment.
    """
    mask = nucleus_mask.astype(bool)

    # ── Guard: bail on over-fired planes ─────────────────────────────────
    if max_foreground_fraction < 1.0:
        fg_frac = float(mask.mean())
        if fg_frac > max_foreground_fraction:
            return np.zeros_like(mask, dtype=bool)

    # ── Step 1: remove small fragments early ─────────────────────────────
    mask = morphology.remove_small_objects(mask, min_size=min_size_px)
    if not mask.any():
        return mask

    # ── Step 2: opening before fill to break merged blobs ────────────────
    if opening_radius > 0:
        mask = morphology.opening(mask, footprint=morphology.disk(opening_radius))
    if not mask.any():
        return mask

    # ── Step 3: per-object hole fill (safe on large/irregular masks) ─────
    # ndi.binary_fill_holes on the full image flood-fills the entire background
    # which segfaults on complex maze-like backgrounds at mid-stack Z planes.
    # Filling per bounding-box is equivalent but operates on small arrays only.
    labeled_for_fill = measure.label(mask, connectivity=2)
    filled = np.zeros_like(mask, dtype=bool)
    for prop in measure.regionprops(labeled_for_fill):
        r0, c0, r1, c1 = prop.bbox
        r0m = max(0, r0 - 1); c0m = max(0, c0 - 1)
        r1m = min(mask.shape[0], r1 + 1); c1m = min(mask.shape[1], c1 + 1)
        patch = mask[r0m:r1m, c0m:c1m]
        filled[r0m:r1m, c0m:c1m] |= ndi.binary_fill_holes(patch)
    mask = filled
    del labeled_for_fill, filled

    # ── Step 4: circularity filter (perimeter only on survivors) ─────────
    return _filter_by_area_and_circularity(mask, min_area_px=min_size_px,
                                           min_circularity=min_circularity)


def _run_batched_inference(
    patch_arr: np.ndarray,
    model,
    batch_size: int,
    config: PipelineConfig,
) -> np.ndarray:
    """
    Run model inference on a (N, H, W, C) patch array in eager mode.

    tf.function has been intentionally removed.  With a large U-Net on GPU,
    inference FLOPS dominate completely — tf.function saves ~1-5 ms per batch
    in Python dispatch overhead but costs 60-300 s whenever it retraces
    (which it does on any new input shape or after gc/memory events).
    Eager mode via model(x, training=False) is consistent and deadlock-free.

    The model should be warmed up once before the segmentation loop with
    warm_up_model() so the first real batch does not pay the GPU JIT cost.
    """
    n_patches = patch_arr.shape[0]
    all_preds: List[np.ndarray] = []

    for i in range(0, n_patches, batch_size):
        chunk = patch_arr[i:i + batch_size].astype(np.float32)
        raw = model(chunk, training=False).numpy()
        all_preds.append(_project_model_probabilities(raw, config))

    return np.concatenate(all_preds, axis=0)


def warm_up_model(model, patch_size: int, n_channels: int, batch_size: int) -> None:
    """
    Run one dummy forward pass to initialise GPU kernels before the main loop.

    Without this, the very first model(x) call of the segmentation run pays
    the CUDA JIT compilation cost for every layer — typically 5-15 s on a
    large U-Net.  Running this once here means all subsequent calls hit the
    warm kernel cache immediately.
    """
    dummy = np.zeros((batch_size, patch_size, patch_size, n_channels), dtype=np.float32)
    _ = model(dummy, training=False)
    del dummy
    print(f"  Model warmed up (batch={batch_size}, patch={patch_size}, channels={n_channels})")



# =====================================================================
# Stage-2 gate — NPC-shell organisation + membrane co-localisation
# PORTED FROM vulcan_training_1.1.ipynb (sections 4b, 5b, 6)      [v16.2]
# ---------------------------------------------------------------------
# The training-set generator has always applied this gate. The segmentation
# block never did — it accepted any component surviving clean_nucleus_mask,
# which tests area, solidity and holes: all properties of the MASK, every one
# of which a droplet cap passes trivially (in a single Z a cap is a small,
# round, solid, filled disc). That asymmetry is the t = 0-3 contamination.
#
# The gate tests ORGANISATION, NOT PRESENCE:
#   1. Shell     — NPC puncta in an annulus around the nucleus edge must admit
#                  a coherent RANSAC circle. An envelope fits; scattered cap or
#                  antibody-clump signal does not.
#   2. Membrane  — each punctum needs membrane signal within R, and a minimum
#                  fraction must be supported. True NPC sits on the NE, which
#                  the dim membrane probe also marks.
# Both must fire.
#
# The functions below are transcribed from the generator so the two can be
# diffed. ONE deliberate deviation, marked inline: _punctum_centroids reshapes
# to (-1, 2) so the empty case has a 2-D shape. Behaviourally identical.
#
# Normalisation regions are the DROPLET INTERIOR in every case, exactly as in
# the generator, which is why the adapter pairs each candidate with its parent
# droplet instance rather than using a local annulus.
# =====================================================================


def _circle_from_3(p1, p2, p3):
    ax, ay = p1; bx, by = p2; cx, cy = p3
    d = 2.0 * (ax*(by-cy) + bx*(cy-ay) + cx*(ay-by))
    if abs(d) < 1e-9:
        return None
    a2, b2, c2 = ax*ax+ay*ay, bx*bx+by*by, cx*cx+cy*cy
    ux = (a2*(by-cy) + b2*(cy-ay) + c2*(ay-by)) / d
    uy = (a2*(cx-bx) + b2*(ax-cx) + c2*(bx-ax)) / d
    return ux, uy, float(np.hypot(ux-ax, uy-ay))


def _fit_circle_kasa(pts):
    x, y = pts[:, 0], pts[:, 1]
    A = np.c_[x, y, np.ones(len(x))]
    b = x*x + y*y
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = sol[0]/2.0, sol[1]/2.0
    r = np.sqrt(max(sol[2] + cx*cx + cy*cy, 0.0))
    return cx, cy, r


def fit_circle_ransac(pts, tol_px=4.0, n_iter=200, min_inlier_frac=0.5, rng=None):
    """
    RANSAC circle fit to Nx2 (x, y). Returns (cx, cy, r, inlier_mask) or None.
    Tolerant of missing arcs (touching neighbours) and stray points.

    rng defaults to seed 0, so the gate is DETERMINISTIC — re-running the same
    plane gives the same verdict. Do not pass a fresh rng per call.
    """
    rng = rng or np.random.default_rng(0)
    n = len(pts)
    if n < 3:
        return None
    best_inliers, best_circle = None, None
    for _ in range(n_iter):
        idx = rng.choice(n, 3, replace=False)
        circ = _circle_from_3(pts[idx[0]], pts[idx[1]], pts[idx[2]])
        if circ is None:
            continue
        cx, cy, r = circ
        resid = np.abs(np.hypot(pts[:, 0]-cx, pts[:, 1]-cy) - r)
        inliers = resid < tol_px
        if best_inliers is None or inliers.sum() > best_inliers.sum():
            best_inliers, best_circle = inliers, circ
    if best_inliers is None or best_inliers.sum() < max(3, int(min_inlier_frac*n)):
        return None
    cx, cy, r = _fit_circle_kasa(pts[best_inliers])      # refit on inliers
    resid = np.abs(np.hypot(pts[:, 0]-cx, pts[:, 1]-cy) - r)
    return cx, cy, r, resid < tol_px


def _punctum_centroids(npc_puncta_mask: np.ndarray) -> np.ndarray:
    """Centroids (x, y) of connected NPC puncta in crop coordinates."""
    lbl = measure.label(npc_puncta_mask)
    return np.array([[r.centroid[1], r.centroid[0]]
                     for r in measure.regionprops(lbl)],
                    dtype=float).reshape(-1, 2)   # <- the one deviation


def detect_npc_puncta(npc_crop_raw, nucleus_mask_crop, droplet_mask_crop,
                      config: PipelineConfig) -> np.ndarray:
    """
    NPC puncta in an annular zone straddling the nucleus edge.

    npc_crop_raw MUST be the RAW NPC plane — not a normalised patch and not a
    probability map. The threshold is mean + k*std of raw NPC over the DROPLET
    interior, so any per-patch contrast stretch invalidates it.
    """
    if nucleus_mask_crop.sum() == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)
    npc = np.asarray(npc_crop_raw, dtype=np.float32)
    selem = morphology.disk(config.npc_margin_px)
    outer = morphology.binary_dilation(nucleus_mask_crop, selem)
    inner = morphology.binary_erosion(nucleus_mask_crop, selem)
    zone = outer & ~inner
    interior_vals = npc[droplet_mask_crop]
    if interior_vals.size == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)
    thresh = interior_vals.mean() + config.npc_std_mult * interior_vals.std()
    puncta = (npc > thresh) & zone
    if config.gate_puncta_min_size_px > 0:
        # NOT in the generator (default 0 = parity). Without it a single noisy
        # pixel counts as a punctum, so gate_min_puncta=8 is met by noise.
        puncta = morphology.remove_small_objects(
            puncta, min_size=int(config.gate_puncta_min_size_px))
    return puncta


def gate_npc_shell(npc_puncta_mask, config: PipelineConfig):
    """
    Shell-organisation test. Returns (passed: bool, info: dict).
    Puncta must admit a coherent RANSAC circle fit (envelope), not scatter.
    """
    cents = _punctum_centroids(npc_puncta_mask)
    info = {"n_puncta": len(cents), "shell_r": np.nan, "shell_inliers": 0}
    if len(cents) < config.gate_min_puncta:
        return False, info
    fit = fit_circle_ransac(cents, tol_px=config.gate_shell_tol_px,
                            n_iter=config.ransac_n_iter,
                            min_inlier_frac=config.gate_shell_min_inlier_frac)
    if fit is None:
        return False, info
    cx, cy, r, inliers = fit
    info["shell_r"], info["shell_inliers"] = float(r), int(inliers.sum())
    passed = inliers.sum() >= max(3, int(config.gate_shell_min_inlier_frac * len(cents)))
    return passed, info


def gate_membrane_coloc(npc_puncta_mask, mem_crop_raw, droplet_mask_crop,
                        config: PipelineConfig):
    """
    Membrane co-localisation test. Returns (passed: bool, info: dict).
    Each NPC punctum must have membrane signal (mean + k*std, dim probe) within
    gate_membrane_nn_radius_px; a minimum fraction of puncta must be supported.
    """
    cents = _punctum_centroids(npc_puncta_mask)
    info = {"coloc_frac": 0.0}
    if len(cents) == 0:
        return False, info
    mem = np.asarray(mem_crop_raw, dtype=np.float32)
    interior = mem[droplet_mask_crop]
    if interior.size == 0:
        return False, info
    mem_thresh = interior.mean() + config.gate_membrane_k_std * interior.std()
    mem_present = mem > mem_thresh
    mem_dil = morphology.binary_dilation(
        mem_present, morphology.disk(config.gate_membrane_nn_radius_px))
    H, W = mem.shape
    supported = 0
    for x, y in cents:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= yi < H and 0 <= xi < W and mem_dil[yi, xi]:
            supported += 1
    frac = supported / len(cents)
    info["coloc_frac"] = float(frac)
    return frac >= config.gate_membrane_min_coloc_frac, info


def gate_droplet(npc_puncta_mask, mem_crop_raw, droplet_mask_crop,
                 config: PipelineConfig):
    """
    Full Stage-2 gate: shell-organisation AND membrane co-localisation.
    Returns (passed: bool, info: dict). A candidate passes only if both fire.
    """
    shell_ok, shell_info = gate_npc_shell(npc_puncta_mask, config)
    mem_ok, mem_info = gate_membrane_coloc(npc_puncta_mask, mem_crop_raw,
                                           droplet_mask_crop, config)
    info = {**shell_info, **mem_info, "shell_ok": shell_ok, "membrane_ok": mem_ok}
    return (shell_ok and mem_ok), info


# ---------------------------------------------------------------------
# v16 adapter — per-candidate, parent-droplet-scoped
# ---------------------------------------------------------------------
def apply_stage2_gate_to_plane(
    nucleus_mask: np.ndarray,
    npc_plane_raw: Optional[np.ndarray],
    mem_plane_raw: Optional[np.ndarray],
    droplet_labels: Optional[np.ndarray],
    config: PipelineConfig,
    t_idx: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray, Dict[int, dict], int, int]:
    """
    Run the generator's Stage-2 gate on every candidate in `nucleus_mask`.

    The generator gates per DROPLET; v16's unit is a nucleus candidate, so each
    candidate is paired with the droplet instance it sits in and that droplet's
    interior supplies the NPC and membrane thresholds. Since extract_class_masks
    does `droplet |= nucleus`, every candidate has a parent by construction.

    Parameters
    ----------
    npc_plane_raw, mem_plane_raw : RAW Ch2 / Ch0 for this (t, z). Not normalised,
        not probabilities — the thresholds are mean + k*std of raw intensity.
    droplet_labels : instance labels from measure.label(droplet_mask). Must be
        computed BEFORE this call.
    t_idx : used only for config.gate_min_timepoint.

    Returns
    -------
    (verified_mask, instance_labels, metrics_by_label, n_candidates, n_verified)

    `instance_labels` retains each survivor's ORIGINAL label, so metric keys stay
    valid and a rejected object leaves a gap in the numbering instead of silently
    renaming its neighbours. Labels are therefore non-contiguous; regionprops,
    napari and ImageJ all handle that.

    Inputs missing, gate disabled, or t below gate_min_timepoint -> everything
    passes through untouched, so the flag is a clean A/B switch.
    """
    mask_bool = nucleus_mask.astype(bool)
    if not mask_bool.any():
        return mask_bool, np.zeros(mask_bool.shape, np.uint16), {}, 0, 0

    lab = measure.label(mask_bool, connectivity=2)
    n_cand = int(lab.max())

    skip_t = (config.gate_min_timepoint is not None
              and t_idx is not None
              and t_idx < config.gate_min_timepoint)
    if (npc_plane_raw is None or mem_plane_raw is None
            or droplet_labels is None or skip_t):
        return mask_bool, lab.astype(np.uint16), {}, n_cand, n_cand

    npc = np.asarray(npc_plane_raw)
    mem = np.asarray(mem_plane_raw)
    H, W = lab.shape

    # ── Droplet bboxes: ONE vectorised pass, not one per candidate ────────
    # Doing `droplet_labels == parent` then regionprops inside the loop scans
    # the full 5732x3889 frame for every nucleus — thousands of full-frame
    # passes per timepoint. regionprops_table gets every droplet's bbox and
    # area in a single C-level pass.
    dtab = measure.regionprops_table(
        droplet_labels, properties=("label", "bbox", "area"))
    droplet_bbox = {
        int(l): (int(r0), int(c0), int(r1), int(c1), int(a))
        for l, r0, c0, r1, c1, a in zip(
            dtab["label"], dtab["bbox-0"], dtab["bbox-1"],
            dtab["bbox-2"], dtab["bbox-3"], dtab["area"])}

    metrics_by_label: Dict[int, dict] = {}
    keep_labels: List[int] = []

    for prop in measure.regionprops(lab):
        # ── parent droplet = the instance covering most of this candidate ──
        coords = prop.coords
        dvals = droplet_labels[coords[:, 0], coords[:, 1]]
        dvals = dvals[dvals > 0]
        if dvals.size and int(np.bincount(dvals).argmax()) in droplet_bbox:
            parent = int(np.bincount(dvals).argmax())
            r0, c0, r1, c1, droplet_area = droplet_bbox[parent]
            dmask_full = None          # crop is taken lazily below
        else:
            # no parent droplet — should not happen (droplet |= nucleus), but
            # fall back to a padded candidate bbox rather than crashing
            parent, droplet_area = 0, 0
            pad = int(config.npc_margin_px + config.gate_membrane_nn_radius_px + 4)
            r0, c0, r1, c1 = prop.bbox
            r0 = max(0, r0-pad); c0 = max(0, c0-pad)
            r1 = min(H, r1+pad); c1 = min(W, c1+pad)

        sl = (slice(r0, r1), slice(c0, c1))
        nuc_crop = lab[sl] == prop.label
        # Droplet mask cropped FIRST, compared second — the comparison touches
        # only the droplet bbox, never the full frame.
        dm_crop = (droplet_labels[sl] == parent if parent
                   else np.ones((r1-r0, c1-c0), bool))

        # ── generator's Stage-2 gate, unmodified ──────────────────────────
        npc_puncta = detect_npc_puncta(npc[sl], nuc_crop, dm_crop, config)
        passed, info = gate_droplet(npc_puncta, mem[sl], dm_crop, config)

        # ── ring/core raw-NPC contrast (NOT in the generator) ─────────────
        # The measurement that actually separates a real envelope from a flat
        # cap: 1.46 vs 1.00 on the synthetic check. Always recorded; only
        # enforced when gate_min_ring_contrast is set.
        sel = morphology.disk(config.npc_margin_px)
        zone = (morphology.binary_dilation(nuc_crop, sel)
                & ~morphology.binary_erosion(nuc_crop, sel))
        core = morphology.binary_erosion(nuc_crop, sel)
        if not core.any():
            core = nuc_crop
        npc_c = np.asarray(npc[sl], dtype=np.float32)
        ring_mean = float(npc_c[zone].mean()) if zone.any() else np.nan
        core_mean = float(npc_c[core].mean()) if core.any() else np.nan
        ring_contrast = (ring_mean / core_mean
                         if np.isfinite(ring_mean) and np.isfinite(core_mean)
                         and core_mean > 0 else np.nan)
        contrast_ok = True
        if config.gate_min_ring_contrast is not None:
            contrast_ok = (np.isfinite(ring_contrast)
                           and ring_contrast >= config.gate_min_ring_contrast)

        # ── droplet-relative area window (generator's _clean_and_gate) ────
        frac = (float(prop.area) / droplet_area) if droplet_area else np.nan
        frac_ok = True
        if config.gate_use_droplet_area_fraction and droplet_area:
            frac_ok = (config.nucleus_min_frac <= frac <= config.nucleus_max_frac)

        # ── optional shell-radius sanity check (NOT in the generator) ─────
        radius_ok = True
        r_eq_droplet = np.sqrt(droplet_area / np.pi) if droplet_area else np.nan
        shell_frac = (info["shell_r"] / r_eq_droplet
                      if np.isfinite(info.get("shell_r", np.nan))
                      and np.isfinite(r_eq_droplet) and r_eq_droplet > 0
                      else np.nan)
        if np.isfinite(shell_frac):
            if (config.gate_shell_r_frac_min is not None
                    and shell_frac < config.gate_shell_r_frac_min):
                radius_ok = False
            if (config.gate_shell_r_frac_max is not None
                    and shell_frac > config.gate_shell_r_frac_max):
                radius_ok = False

        m = {
            "parent_droplet_label":  parent,
            "droplet_area_px":       droplet_area,
            "nucleus_frac_of_droplet": frac,
            "n_puncta":       int(info["n_puncta"]),
            "shell_r_px":     float(info["shell_r"]),
            "shell_inliers":  int(info["shell_inliers"]),
            "shell_r_frac_of_droplet": float(shell_frac),
            "coloc_frac":     float(info["coloc_frac"]),
            "npc_ring_contrast": float(ring_contrast),
            "shell_ok":       bool(info["shell_ok"]),
            "membrane_ok":    bool(info["membrane_ok"]),
            "area_frac_ok":   bool(frac_ok),
            "shell_radius_ok": bool(radius_ok),
            "ring_contrast_ok": bool(contrast_ok),
        }
        m["stage2_verified"] = bool(passed and frac_ok and radius_ok and contrast_ok)
        metrics_by_label[int(prop.label)] = m
        if m["stage2_verified"]:
            keep_labels.append(int(prop.label))

    if not config.require_stage2_gate:
        return mask_bool, lab.astype(np.uint16), metrics_by_label, n_cand, n_cand

    if keep_labels:
        instance_labels = np.where(np.isin(lab, keep_labels), lab, 0).astype(np.uint16)
    else:
        instance_labels = np.zeros(lab.shape, np.uint16)
    return (instance_labels > 0, instance_labels, metrics_by_label,
            n_cand, len(keep_labels))


def segment_all_planes_for_timepoint(img_5d,t_idx,keep_z,model,config):
    """Compatibility entry point using the same eight-head baseline decoder."""
    valid=set(eligible_z_planes(img_5d.shape[1],config))
    if any(z not in valid for z in keep_z):
        raise ValueError('Requested target below Z floor or above ceiling')
    return {z:segment_single_plane_with_overlap(img_5d,t_idx,z,model,config) for z in keep_z}


def segment_single_plane_with_overlap(img_5d,t_idx,z_idx,model,config,patch_size=512,stride=384):
    validate_vulcan_model(model,config)
    heads=predict_vulcan_plane(img_5d,t_idx,z_idx,model,config)
    lab,drop=vulcan_instances(heads,np.asarray(img_5d[t_idx,z_idx,config.nuclear_channel_index],np.float32),
        config,img_5d[t_idx,z_idx,config.npc_channel_index],img_5d[t_idx,z_idx,config.membrane_channel_index],t_idx)
    display=collapse_label_map(drop>0,heads[...,3]>config.npc_threshold,lab>0,config)
    return heads[...,[0,1,3,4]],display,lab>0,lab


VULCAN_HEADS = ('background', 'droplet_interior', 'droplet_edge', 'npc',
                'nucleus_interior', 'nucleus_edge', 'nucleus_equatorial', 'z_offset')

def file_sha256(path):
    import hashlib
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            h.update(block)
    return h.hexdigest()


def eligible_z_planes(n_z, config):
    lo = max(0, int(config.focus_min_z))
    hi = n_z - 1 if config.focus_max_z is None else min(n_z - 1, config.focus_max_z)
    return list(range(lo, hi + 1))


def full_head_path(config):
    return config.seg_dir / 'vulcan_all_heads.tif'


def full_head_metadata_path(config):
    return config.seg_dir / 'vulcan_all_heads.json'


def validate_vulcan_model(model, config):
    if tuple(model.input_shape[1:]) != (config.patch_size, config.patch_size, 15):
        raise ValueError('Vulcan input must be (512, 512, 15): ' + str(model.input_shape))
    if int(model.output_shape[-1]) != 8:
        raise ValueError('Expected seven mask logits and one Z-offset output')
    masks = model.get_layer('masks_logits')
    zhead = model.get_layer('z_offset')
    if masks.filters != 7 or masks.activation.__name__ != 'linear':
        raise ValueError('Unexpected mask-head contract')
    if zhead.filters != 1 or zhead.activation.__name__ != 'softplus':
        raise ValueError('Unexpected Z-offset contract')
    if config.n_z_context != 5 or config.model_probability_output_indices != (0, 1, 3, 4):
        raise ValueError('Unsupported input context or semantic projection')


def predict_vulcan_plane(img_5d, t, z, model, config, channel_stats=None):
    """Streaming Hann-blended dihedral TTA, matching training H1 inference."""
    if z not in eligible_z_planes(img_5d.shape[1], config):
        raise ValueError('Target plane is outside the configured Z range')
    if channel_stats is None:
        channel_stats = normalization_stats_for_timepoint(img_5d, t, config)
    full = get_full_context_yxc(img_5d, t, z, config)
    if config.normalization_mode != 'global_t':
        raise ValueError('This validated model requires global_t normalization')
    full = _preprocess_patch(full, channel_stats)
    h, w = full.shape[:2]
    ps = config.patch_size
    if min(h, w) < ps:
        raise ValueError('Image must contain at least one full model patch')
    origins = [(y, x) for y in _generate_patch_starts(h, ps, config.patch_stride)
               for x in _generate_patch_starts(w, ps, config.patch_stride)]
    win = (np.hanning(ps)[:, None] * np.hanning(ps)[None, :]).astype(np.float32) + 1e-6
    acc = np.zeros((h, w, 8), np.float32)
    weight = np.zeros((h, w), np.float32)
    transforms = [(k, f) for k in range(4) for f in (False, True)] if config.tta else [(0, False)]
    for start in range(0, len(origins), config.batch_size):
        chunk = origins[start:start + config.batch_size]
        batch = np.stack([full[y:y+ps, x:x+ps] for y, x in chunk])
        mean = np.zeros(batch.shape[:3] + (8,), np.float32)
        for k, flip in transforms:
            xb = np.stack([np.rot90(np.flip(a, 1) if flip else a, k) for a in batch])
            p = np.asarray(model(xb, training=False).numpy(), np.float32)
            if p.shape != mean.shape or not np.isfinite(p).all():
                raise ValueError('Invalid or nonfinite model output')
            p[..., :7] = 1 / (1 + np.exp(-np.clip(p[..., :7], -80, 80)))
            for i, a in enumerate(p):
                a = np.rot90(a, -k)
                mean[i] += (np.flip(a, 1) if flip else a) / len(transforms)
        for (y, x), p in zip(chunk, mean):
            acc[y:y+ps, x:x+ps] += p * win[..., None]
            weight[y:y+ps, x:x+ps] += win
    return acc / np.maximum(weight[..., None], 1e-6)


def vulcan_instances(heads, nls, config, npc_raw=None, mem_raw=None, t_idx=None):
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed
    core, edge = heads[..., 4], heads[..., 5]
    seed = morphology.binary_erosion(core > config.watershed_core_thresh, morphology.disk(2))
    dist = ndi.distance_transform_edt(seed)
    comp = measure.label(seed)
    peaks = peak_local_max(dist, min_distance=config.watershed_min_marker_dist_px,
                           labels=comp, exclude_border=False)
    markers = np.zeros(seed.shape, np.int32)
    for i, (y, x) in enumerate(peaks, 1):
        markers[y, x] = i
    if not markers.any():
        markers = comp
    rng = ndi.maximum_filter(nls, config.sharpness_window_px) - ndi.minimum_filter(nls, config.sharpness_window_px)
    sharp = filters.sobel(nls) / (rng + 1e-3 * (nls.max() - nls.min() + 1e-9))
    lab = watershed(sharp + config.watershed_edge_weight * edge, markers,
                    mask=(core > config.nucleus_threshold) | (edge > config.nucleus_threshold))
    droplet = heads[..., 1] > config.droplet_threshold
    drops = measure.label(droplet, connectivity=2)
    # Optional legacy rejection is applied per instance; never merge watershed IDs.
    if config.require_stage2_gate:
        for r in measure.regionprops(lab):
            candidate = lab == r.label
            accepted, _, _, _, _ = apply_stage2_gate_to_plane(
                candidate, npc_raw, mem_raw, drops, config, t_idx=t_idx)
            if not accepted.any():
                lab[candidate] = 0
    if max(int(lab.max()), int(drops.max())) > np.iinfo(np.uint16).max:
        raise OverflowError('Instance count exceeds uint16; no truncated labels written')
    return lab.astype(np.uint16), drops.astype(np.uint16)



## 8. TIFF save helpers

In [ ]:
def _write_hyperstack_tiff(arr: np.ndarray, out_path: Path, axes: str,
                           prefer_imagej: bool = True) -> None:
    if tiff is None:
        raise ImportError("tifffile is required.")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(arr)
    # BigTIFF for arrays >= ~2 GB to avoid ImageJ truncation.
    use_imagej = prefer_imagej and arr.nbytes < (2 * 1024**3 - 32 * 1024**2)
    tiff.imwrite(out_path, arr,
                 imagej=use_imagej, bigtiff=not use_imagej,
                 metadata={"axes": axes})


def save_probability_hyperstack_tiff(probability_5d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(probability_5d, out_path, axes="TZCYX", prefer_imagej=False)


def save_class_hyperstack_tiff(class_mask_5d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff((class_mask_5d.astype(np.uint8) * 255), out_path,
                           axes="TZCYX", prefer_imagej=True)


def save_label_hyperstack_tiff(label_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(label_4d.astype(np.uint8), out_path,
                           axes="TZYX", prefer_imagej=True)


def save_instance_hyperstack_tiff(instance_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(instance_4d, out_path, axes="TZYX", prefer_imagej=True)


def create_probability_memmap(
    shape: Tuple[int, int, int, int, int],
    out_dir: Path,
    dtype: np.dtype = np.float16,
) -> Tuple[np.memmap, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    temp_path = out_dir / "probability_hyperstack_temp.dat"
    if temp_path.exists():
        temp_path.unlink()
    return np.memmap(temp_path, mode="w+", dtype=dtype, shape=shape), temp_path



def open_hyperstack_memmap(out_path: Path,
                           shape: Tuple[int, ...],
                           dtype,
                           axes: str) -> np.memmap:
    """Open a hyperstack TIFF for in-place plane writes, creating a
    zero-filled one of ``shape`` if it does not already exist.

    Backs the resumable segmentation path: planes are written straight into
    the final TIFF through the returned memmap, so an interrupted run keeps
    every finished plane on disk and a restart only fills the missing
    timepoints. No ``.dat`` scratch copy is used.

    An existing file whose shape/dtype does not match (an older run with
    different dims, or a non-memmappable ImageJ TIFF) is discarded and
    recreated.
    """
    if tiff is None:
        raise ImportError("tifffile is required.")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    dtype = np.dtype(dtype)
    shape = tuple(int(s) for s in shape)

    if out_path.exists():
        mm = tiff.memmap(str(out_path), mode="r+")
        if tuple(mm.shape) != shape or np.dtype(mm.dtype) != dtype:
            raise RuntimeError(f"Incompatible existing artifact: {out_path}; choose a new run")
        return mm

    return tiff.memmap(str(out_path), shape=shape, dtype=dtype,
                       bigtiff=True, photometric="minisblack", metadata={"axes": axes})


In [ ]:
# ============================================================================
# 8b. Segmentation checkpoint — single source of truth for stage state
# ============================================================================
# `seg_dir/_checkpoint.json` records what segmentation has produced and whether
# every artifact a downstream cell needs is still on disk. It supersedes the old
# `_resume/manifest.json` (the per-timepoint `_resume/t{NNN}.pkl` sidecars stay).
#
# The full-run vs resume vs already-complete decision is made by
# `assess_segmentation_state()` BEFORE the model is loaded and BEFORE the focus
# warm-up, so a finished (or half-finished) run costs nothing to detect.
#
#   status "in_progress"  some timepoints done, run not finalised
#   status "complete"     all timepoints done, seg_index + roi_table written,
#                         every artifact fingerprinted
#
# A "complete" checkpoint whose artifacts no longer verify -> mode "inconsistent"
# -> hard fail with recovery text (no implicit GPU work).

_SEG_DTYPES = {"class": "uint8", "label": "uint8",
               "nucleus": "uint16", "droplet": "uint16", "prob": "float16"}


@dataclass
class SegmentationState:
    mode: str                       # "complete" | "resume" | "fresh" | "inconsistent"
    completed_t: List[int]
    todo_t: List[int]
    checkpoint: Optional[dict]
    missing_artifacts: List[str]
    reason: str


def _seg_artifact_targets(config, save_probability_tiff: bool) -> Dict[str, tuple]:
    """artifact key -> ('file', Path) | ('npy_dir', Path). The set of files a
    completed segmentation must leave on disk for the downstream cells."""
    t = {
        "class_hyperstack":            ("file", config.segmentation_class_hyperstack_path),
        "label_hyperstack":            ("file", config.segmentation_label_hyperstack_path),
        "nucleus_instance_hyperstack": ("file", config.nucleus_instance_hyperstack_path),
        "droplet_instance_hyperstack": ("file", config.droplet_instance_hyperstack_path),
        "seg_index":                   ("file", config.segmentation_index_path),
        "roi_table":                   ("file", config.segmentation_roi_table_path),
        "npy_planes":                  ("npy_dir", config.seg_dir),
    }
    t["full_heads"] = ("file", full_head_path(config))
    t["full_heads_metadata"] = ("file", full_head_metadata_path(config))
    if save_probability_tiff:
        t["probability_hyperstack"] = ("file", config.segmentation_probability_hyperstack_path)
    return t


def _verify_seg_artifacts(cp: dict, needs=None) -> List[str]:
    """Return the artifact keys that are missing or changed since the checkpoint
    was written. `needs` limits the check to a subset of keys."""
    arts = cp.get("artifacts", {})
    keys = list(needs) if needs else list(arts.keys())
    missing: List[str] = []
    for k in keys:
        ent = arts.get(k)
        if not ent:
            missing.append(k); continue
        if "dir" in ent:                                   # npy plane directory
            d = Path(ent["dir"])
            n = len(list(d.glob("nuclear_mask_t*_z*.npy"))) if d.is_dir() else 0
            if n < int(ent.get("count", 0)):
                missing.append(k)
        else:                                              # single file
            p = Path(ent["path"])
            if not p.exists():
                missing.append(k)
            elif ent.get("bytes") is not None and p.stat().st_size != ent["bytes"]:
                missing.append(k)
    return missing


def _mask_stores_ok(config, T: int, Z: int, Y: int, X: int) -> tuple:
    """Are the 4 mask hyperstacks present and memmappable at the expected shape?
    A resume cannot write into a deleted store."""
    checks = [
        (full_head_path(config), (T, Z, 8, Y, X)),
        (config.segmentation_probability_hyperstack_path, (T, Z, 4, Y, X)),
        (config.segmentation_class_hyperstack_path, (T, Z, config.num_classes, Y, X)),
        (config.segmentation_label_hyperstack_path, (T, Z, Y, X)),
        (config.nucleus_instance_hyperstack_path,   (T, Z, Y, X)),
        (config.droplet_instance_hyperstack_path,   (T, Z, Y, X)),
    ]
    for p, want in checks:
        p = Path(p)
        if not p.exists():
            return False, f"{p.name} missing"
        try:
            mm = tiff.memmap(str(p), mode="r")
            shp = tuple(mm.shape); del mm
        except Exception as exc:
            return False, f"{p.name} not memmappable ({exc})"
        if shp != want:
            return False, f"{p.name} shape {shp} != {want}"
    return True, "ok"


def _recovery_hint(config) -> str:
    prob = Path(config.segmentation_probability_hyperstack_path)
    if prob.exists():
        return ("Recovery: a probability hyperstack is present — set "
                "cfg.probability_hyperstack_input to it and re-run §27 to rebuild the "
                "masks with no GPU; or run reset_segmentation(cfg, scope='all') and "
                "re-run §27 for a full rebuild.")
    return ("Recovery: run reset_segmentation(cfg, scope='all') in the §27 reset cell, "
            "then re-run §27 — no probability hyperstack remains, so this re-runs full "
            "GPU inference for every timepoint.")


def _inconsistent_reason(cp: dict, missing: List[str], config) -> str:
    lines = ["Segmentation checkpoint says status=complete, but these artifacts are "
             "missing or changed on disk:"]
    for k in missing:
        ent = cp.get("artifacts", {}).get(k, {})
        lines.append(f"    - {k}: {ent.get('path', ent.get('dir', '?'))}")
    lines.append(f"  checkpoint finalised: {cp.get('finalized_at', '?')}")
    lines.append(_recovery_hint(config))
    return "\n".join(lines)


def assess_segmentation_state(config, dims_tzyx: tuple,
                              save_probability_tiff: bool = True) -> SegmentationState:
    """Decide fresh / resume / complete / inconsistent from on-disk state only.
    No image load beyond the dims tuple, no TF, no GPU, no focus scoring."""
    T, Z, Y, X = (int(v) for v in dims_tzyx)
    all_t = list(range(T))
    cp_path = config.seg_dir / "_checkpoint.json"
    resume_dir = config.seg_dir / "_resume"
    legacy_manifest = resume_dir / "manifest.json"

    cp = None
    if cp_path.exists():
        try:
            cp = json.loads(cp_path.read_text())
        except Exception:
            cp = None

    if cp is None and legacy_manifest.exists():          # migrate old resume state
        try:
            _m = json.loads(legacy_manifest.read_text())
            cp = {"schema": 1, "stage": "segmentation", "status": "in_progress",
                  "dims": {"T": T, "Z": Z, "Y": Y, "X": X},
                  "completed_t": sorted(int(t) for t in _m.get("completed_t", [])),
                  "artifacts": {}, "note": "migrated from legacy _resume/manifest.json"}
        except Exception:
            cp = None

    if cp is None:
        return SegmentationState("fresh", [], all_t, None, [],
            f"No segmentation checkpoint ({cp_path.name}) — fresh run, "
            f"all {T} timepoints to segment.")

    if cp.get("config_signature") != json.loads(json.dumps(config.to_run_signature())):
        return SegmentationState("inconsistent", [], [], cp, [], "Checkpoint configuration differs; create a new run")
    want = {"T": T, "Z": Z, "Y": Y, "X": X}
    cp_dims = cp.get("dims", {})
    if any(cp_dims.get(k) != v for k, v in want.items()):
        return SegmentationState("fresh", [], all_t, cp, [],
            f"Checkpoint dims {cp_dims} != current {want} — stale checkpoint ignored, "
            f"re-running all {T} timepoints.")

    completed = sorted(int(t) for t in cp.get("completed_t", []) if 0 <= int(t) < T)
    status = cp.get("status")

    if status == "complete":
        missing = _verify_seg_artifacts(cp)
        if not missing:
            return SegmentationState("complete", all_t, [], cp, [],
                f"Segmentation checkpoint complete and all {len(cp.get('artifacts', {}))} "
                f"artifacts verified (finalised {cp.get('finalized_at', '?')}). "
                f"Loading saved tables — no model load, no focus scoring.")
        return SegmentationState("inconsistent", completed, [], cp, missing,
            _inconsistent_reason(cp, missing, config))

    # status == "in_progress" (or a partial checkpoint with completed_t)
    ok, msg = _mask_stores_ok(config, T, Z, Y, X)
    if completed and not ok:
        return SegmentationState("inconsistent", completed, [], cp, ["mask_hyperstacks"],
            f"Checkpoint says timepoints {completed} were segmented, but the mask "
            f"hyperstacks are gone or the wrong shape ({msg}). Cannot resume into a "
            f"deleted store.\n" + _recovery_hint(config))

    survived = [t for t in completed if (resume_dir / f"t{t:03d}.pkl").exists()]
    lost = sorted(set(completed) - set(survived))
    todo = [t for t in all_t if t not in survived]

    if not todo:
        return SegmentationState("resume", survived, [], cp, [],
            f"Checkpoint lists all {T} timepoints done but was never finalised "
            f"(status={status!r}). Re-running finalisation only.")

    reason = (f"Resuming segmentation: {len(survived)}/{T} timepoints already done "
              f"(t={survived}); {len(todo)} to segment (t={todo}).")
    if lost:
        reason += f"\n  sidecars missing for t={lost} — those will be re-segmented."
    return SegmentationState("resume", survived, todo, cp, [], reason)


def write_segmentation_checkpoint(config, *, status: str, dims_tzyx: tuple,
                                  dtypes: dict, save_probability_tiff: bool,
                                  completed_t) -> dict:
    """Write `seg_dir/_checkpoint.json` atomically, fingerprinting every artifact
    that exists right now (st_size for files, plane count for the npy dir)."""
    T, Z, Y, X = (int(v) for v in dims_tzyx)
    cp_path = config.seg_dir / "_checkpoint.json"
    prev = {}
    if cp_path.exists():
        try:
            prev = json.loads(cp_path.read_text())
        except Exception:
            prev = {}

    artifacts = {}
    for key, (kind, p) in _seg_artifact_targets(config, save_probability_tiff).items():
        p = Path(p)
        if kind == "npy_dir":
            artifacts[key] = {"dir": str(p),
                              "count": len(list(p.glob("nuclear_mask_t*_z*.npy")))}
        else:
            artifacts[key] = {"path": str(p),
                              "bytes": p.stat().st_size if p.exists() else None}

    now = datetime.now().isoformat(timespec="seconds")
    cp = {
        "schema": 1,
        "stage": "segmentation",
        "run_id": getattr(getattr(config, "paths", None), "run_id", None),
        "code_version": config.pipeline_revision,
        "config_signature": config.to_run_signature(),
        "model_sha256": config.model_sha256,
        "status": status,
        "dims": {"T": T, "Z": Z, "Y": Y, "X": X},
        "dtypes": dtypes,
        "save_probability_tiff": bool(save_probability_tiff),
        "completed_t": sorted(int(t) for t in completed_t),
        "artifacts": artifacts,
        "started_at": prev.get("started_at") or now,
        "updated_at": now,
    }
    if status == "complete":
        cp["finalized_at"] = now

    tmp = cp_path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(cp, indent=2, default=str))
    os.replace(tmp, cp_path)
    return cp


def require_segmentation(config, needs=()) -> dict:
    """Guard for downstream cells. Raise unless the segmentation checkpoint is
    'complete' and every artifact key in `needs` still verifies on disk."""
    cp_path = config.seg_dir / "_checkpoint.json"
    if not cp_path.exists():
        raise RuntimeError(
            f"{cp_path} not found — segmentation (§27) has not completed in this run. "
            f"Run §27 first.")
    cp = json.loads(cp_path.read_text())
    if cp.get("status") != "complete":
        raise RuntimeError(
            f"Segmentation checkpoint status={cp.get('status')!r}, not 'complete'. "
            f"Re-run §27 to finish it.")
    if cp.get("config_signature") != json.loads(json.dumps(config.to_run_signature())):
        raise RuntimeError("Configuration changed after segmentation; create a new run")
    missing = _verify_seg_artifacts(cp, needs=needs or None)
    if missing:
        raise RuntimeError(_inconsistent_reason(cp, missing, config))
    return cp


def reset_segmentation(config, scope: str) -> None:
    """Wipe segmentation state so the next §27 run rebuilds.

    scope="resume"  delete only the checkpoint + _resume/ (keep the TIFFs).
                    Forces a full recompute; use after editing a seg function.
    scope="all"     also delete the 5 hyperstack TIFFs, the per-plane .npy
                    masks, the seg index / ROI pickles, AND the stale downstream
                    object pickles. Nothing half-deleted.
    """
    if scope not in ("resume", "all"):
        raise ValueError("scope must be 'resume' or 'all'")
    removed: List[str] = []

    def _rm(p):
        p = Path(p)
        if p.is_dir():
            shutil.rmtree(p); removed.append(str(p) + "/")
        elif p.exists():
            p.unlink(); removed.append(str(p))

    _rm(config.seg_dir / "_checkpoint.json")
    _rm(config.seg_dir / "_checkpoint.json.tmp")
    _rm(config.seg_dir / "_resume")
    _rm(config.seg_dir / "segmentation_checkpoint.json")          # legacy

    if scope == "all":
        for p in (full_head_path(config), full_head_metadata_path(config), config.segmentation_class_hyperstack_path,
                  config.segmentation_label_hyperstack_path,
                  config.nucleus_instance_hyperstack_path,
                  config.droplet_instance_hyperstack_path,
                  config.segmentation_probability_hyperstack_path,
                  config.segmentation_index_path,
                  config.segmentation_roi_table_path):
            _rm(p)
        for name in ("class_mask_5d.dat", "class_label_4d.dat",
                     "nucleus_instance_4d.dat", "droplet_instance_4d.dat"):
            _rm(config.mask_tif_dir / name)
        _npy = list(config.seg_dir.glob("nuclear_mask_t*_z*.npy"))
        for p in _npy:
            p.unlink()
        if _npy:
            removed.append(f"{config.seg_dir}/nuclear_mask_t*_z*.npy  ({len(_npy)} files)")
        for name in ("plane_objects.pkl", "grouped_z_objects.pkl",
                     "best_z_nuclei.pkl", "best_z_nuclei_repaired.pkl",
                     "best_z_nuclei_with_exclusion.pkl"):
            _rm(config.obj_dir / name)
        _rm(config.obj_dir / "repaired_masks")

    print(f"reset_segmentation(scope={scope!r}) removed {len(removed)} path(s):")
    for r in removed:
        print("   ", r)
    if not removed:
        print("    (nothing to remove)")
    if scope == "all":
        print("Re-run §27 to rebuild. Stale downstream object pickles were also cleared.")

## 9. Region-props helper

In [ ]:
def regionprops_to_rows(
    labeled_mask: np.ndarray,
    t_idx: int,
    z_idx: int,
    class_id: int,
    class_name: str,
    min_area_px: int = 0,
    extra_by_label: Optional[Dict[int, Dict]] = None,
) -> List[Dict]:
    """
    Convert a labelled mask to a list of row dicts.

    Uses regionprops_table for a single vectorised C-level pass over all
    objects.  Perimeter is NOT computed here — it requires binary_erosion
    on every object which is catastrophically slow when the model produces
    large merged regions (e.g. droplet lattice at mid-stack Z planes).
    Circularity filtering happens earlier in clean_nucleus_mask; by the time
    we reach this function it is not needed again.

    extra_by_label (v16.2) merges per-object measurements keyed by label. Used
    to carry the Stage-2 gate metrics into the ROI table so a run can be audited
    — and thresholds re-tuned — without re-running segmentation.
    """
    if labeled_mask.max() == 0:
        return []

    tbl = measure.regionprops_table(
        labeled_mask,
        properties=("label", "area", "centroid", "bbox"),
    )

    labels   = tbl["label"]
    areas    = tbl["area"]
    cy_arr   = tbl["centroid-0"]
    cx_arr   = tbl["centroid-1"]
    rmin     = tbl["bbox-0"]
    cmin     = tbl["bbox-1"]
    rmax     = tbl["bbox-2"]
    cmax     = tbl["bbox-3"]

    rows = []
    for i in range(len(labels)):
        if areas[i] < min_area_px:
            continue
        row = {
            "t": t_idx, "z": z_idx,
            "class_id": class_id, "class_name": class_name,
            "label":          int(labels[i]),
            "centroid_x_px":  float(cx_arr[i]),
            "centroid_y_px":  float(cy_arr[i]),
            "area_px":        int(areas[i]),
            "bbox_min_row":   int(rmin[i]),
            "bbox_min_col":   int(cmin[i]),
            "bbox_max_row":   int(rmax[i]),
            "bbox_max_col":   int(cmax[i]),
        }
        if extra_by_label:
            row.update(extra_by_label.get(int(labels[i]), {}))
        rows.append(row)
    return rows

## 10. Full segmentation loop

In [ ]:
def run_segmentation_for_all_planes(img_5d, model, config, state=None,
                                    save_binary_masks=True, save_probability_tiff=True):
    """All eligible planes; fail-fast; timepoint resume; eight preserved heads."""
    if not save_binary_masks or not save_probability_tiff:
        raise ValueError('The reproducible baseline requires masks and probabilities')
    config.paths.assert_config_matches(config.to_run_signature())
    if file_sha256(config.model_path) != config.model_sha256:
        raise RuntimeError('Model changed since bootstrap; create a new run')
    T, Z, C, H, W = img_5d.shape
    if C != 3:
        raise ValueError('Expected microscopy axes TZCYX with three channels')
    zs = eligible_z_planes(Z, config)
    if not zs:
        raise ValueError('No eligible target planes')
    if config.use_focus_z_selection:
        raise ValueError('Baseline segments every eligible plane; disable focus window selection')
    done = set(state.completed_t if state is not None else [])
    if len(done) < T:
        validate_vulcan_model(model, config)
    stores = [
        open_hyperstack_memmap(config.segmentation_class_hyperstack_path,(T,Z,4,H,W),np.uint8,'TZCYX'),
        open_hyperstack_memmap(config.segmentation_label_hyperstack_path,(T,Z,H,W),np.uint8,'TZYX'),
        open_hyperstack_memmap(config.nucleus_instance_hyperstack_path,(T,Z,H,W),np.uint16,'TZYX'),
        open_hyperstack_memmap(config.droplet_instance_hyperstack_path,(T,Z,H,W),np.uint16,'TZYX'),
        open_hyperstack_memmap(config.segmentation_probability_hyperstack_path,(T,Z,4,H,W),np.float16,'TZCYX'),
        open_hyperstack_memmap(full_head_path(config),(T,Z,8,H,W),np.float16,'TZCYX')]
    classes, display_map, nuclei, droplets, canonical, all_heads = stores
    meta = dict(channels=list(VULCAN_HEADS), axes='TZCYX', z_offset_units='um',
                included_z=zs, model_sha256=config.model_sha256, config=config.to_run_signature())
    full_head_metadata_path(config).write_text(json.dumps(meta,indent=2),encoding='utf8')
    records, rows = [], []
    resume = config.seg_dir / '_resume'
    resume.mkdir(parents=True,exist_ok=True)
    try:
        for t in range(T):
            sidecar = resume / ('t%03d.pkl' % t)
            if t in done:
                saved = pd.read_pickle(sidecar)
                records.extend(saved['records']); rows.extend(saved['roi_rows'])
                continue
            tr, rr = [], []
            stats = normalization_stats_for_timepoint(img_5d,t,config)
            (resume / ('normalization_t%03d.json' % t)).write_text(json.dumps(stats),encoding='utf8')
            # Clear any unfinished timepoint so retry cannot retain stale planes.
            for mm in stores:
                mm[t] = 0
            for z in range(Z):
                path = config.seg_dir / ('nuclear_mask_t%03d_z%03d.npy' % (t,z))
                if z not in zs:
                    np.save(path,np.zeros((H,W),np.uint8))
                    tr.append(dict(t=t,z=z,included=False,reason='outside_z_range',mask_path=str(path)))
                    continue
                try:
                    heads = predict_vulcan_plane(img_5d,t,z,model,config,stats)
                    lab, drop = vulcan_instances(heads,np.asarray(img_5d[t,z,config.nuclear_channel_index],np.float32),
                        config,img_5d[t,z,config.npc_channel_index],img_5d[t,z,config.membrane_channel_index],t)
                except Exception as exc:
                    (config.seg_dir/'last_error.json').write_text(json.dumps(dict(t=t,z=z,error=repr(exc))),encoding='utf8')
                    raise RuntimeError('Segmentation failed at t=%d z=%d; timepoint not committed' % (t,z)) from exc
                all_heads[t,z] = np.moveaxis(heads,-1,0).astype(np.float16)
                canonical[t,z] = np.moveaxis(heads[..., [0,1,3,4]],-1,0).astype(np.float16)
                dm, nm, npc = drop>0, lab>0, heads[...,3]>config.npc_threshold
                classes[t,z] = np.stack([~dm,dm,npc,nm]).astype(np.uint8)
                display_map[t,z] = collapse_label_map(dm,npc,nm,config)
                nuclei[t,z], droplets[t,z] = lab,drop
                extra = {}
                for r in measure.regionprops(lab):
                    m = lab[r.slice] == r.label
                    extra[r.label] = dict(mean_p=float(heads[r.slice][...,4][m].mean()),
                        mean_z_offset=float(heads[r.slice][...,7][m].mean()),
                        mean_equatorial=float(heads[r.slice][...,6][m].mean()))
                rr.extend(regionprops_to_rows(lab,t,z,3,'nucleus',extra_by_label=extra))
                rr.extend(regionprops_to_rows(drop,t,z,1,'droplet'))
                np.save(path,nm.astype(np.uint8))
                tr.append(dict(t=t,z=z,included=True,reason='segmented',mask_path=str(path),
                    nucleus_pixels=int(nm.sum()),droplet_pixels=int(dm.sum())))
                print('t=%d z=%d: %d nuclei' % (t,z,len(extra)),flush=True)
            for mm in stores: mm.flush()
            tmp = sidecar.with_suffix('.tmp')
            pd.to_pickle(dict(records=tr,roi_rows=rr),tmp); os.replace(tmp,sidecar)
            records.extend(tr); rows.extend(rr); done.add(t)
            write_segmentation_checkpoint(config,status='in_progress',dims_tzyx=(T,Z,H,W),
                dtypes=_SEG_DTYPES,save_probability_tiff=True,completed_t=done)
            print('Timepoint %d committed' % t,flush=True)
        index = pd.DataFrame(records).sort_values(['t','z']).reset_index(drop=True)
        if len(index)!=T*Z or index.duplicated(['t','z']).any():
            raise RuntimeError('Incomplete or duplicated plane index')
        index.to_pickle(config.segmentation_index_path)
        pd.DataFrame(rows,columns=None if rows else
            ['t','z','label','class_name','class_id','area_px','centroid_x_px','centroid_y_px',
             'mean_p','mean_z_offset','mean_equatorial']).to_pickle(config.segmentation_roi_table_path)
        write_segmentation_checkpoint(config,status='complete',dims_tzyx=(T,Z,H,W),
            dtypes=_SEG_DTYPES,save_probability_tiff=True,completed_t=done)
        return index
    finally:
        for mm in stores: mm.flush()


## 10b. Legacy re-extraction guard

Four-channel re-extraction is disabled because it lacks edge and Z heads. This baseline requires fresh full-head inference.


In [ ]:
def run_reextraction_from_probability(config,save_binary_masks=True):
    raise RuntimeError('Legacy four-channel re-extraction is disabled for this baseline. '
                       'Set probability_hyperstack_input=None and run full inference.')


## 11. Object extraction from saved instance measurements

Preserves watershed IDs and per-instance head statistics.


In [ ]:
def extract_objects_from_saved_masks(seg_index_df, config):
    """Use measured instance IDs, never relabel a binary watershed mask."""
    require_segmentation(config)
    roi = pd.read_pickle(config.segmentation_roi_table_path)
    result = roi[roi.class_name.eq('nucleus')].copy()
    result['nucleus_area_um2'] = result.area_px * config.pixel_size_um**2
    result.to_pickle(config.obj_dir/'plane_objects.pkl')
    return result


## 12. Distance-based helpers

In [ ]:
def nearest_neighbor_matches(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    max_dist_px: float,
) -> List[Tuple[int, int, float]]:
    if source_df.empty or target_df.empty:
        return []
    src_xy = source_df[["centroid_x_px", "centroid_y_px"]].to_numpy()
    tgt_xy = target_df[["centroid_x_px", "centroid_y_px"]].to_numpy()
    tree = cKDTree(tgt_xy)
    dists, idxs = tree.query(src_xy, distance_upper_bound=max_dist_px)

    matches, used = [], set()
    for i in np.argsort(dists):
        dist, j = dists[i], idxs[i]
        if np.isinf(dist) or j in used:
            continue
        matches.append((int(i), int(j), float(dist)))
        used.add(int(j))
    return matches


## 13. Link adjacent Z planes by mask overlap


In [ ]:
def group_nuclei_across_z(objects_df, config):
    """Adjacent-plane overlap links; disconnected planes start new tracks."""
    d = objects_df.copy()
    d['nucleus_3d_id'] = -1
    mm = tiff.memmap(str(config.nucleus_instance_hyperstack_path),mode='r')
    nxt = 1
    for t, group in d.groupby('t',sort=True):
        previous, prev_area, previous_z = None, {}, None
        for z, plane in group.groupby('z',sort=True):
            lab = np.asarray(mm[int(t),int(z)])
            current = np.zeros(lab.shape,np.int32)
            for ix,r in plane.iterrows():
                mask = lab==int(r.label)
                nid = None
                if previous is not None and int(z)==previous_z+1:
                    ov=previous[mask]; ov=ov[ov>0]
                    if ov.size:
                        candidate=int(np.bincount(ov).argmax())
                        if (ov==candidate).sum() > .3*min(r.area_px,prev_area[candidate]):
                            nid=candidate
                if nid is None: nid=nxt; nxt+=1
                d.at[ix,'nucleus_3d_id']=nid
                current[mask]=nid
            previous=current; previous_z=int(z)
            prev_area=dict(zip(*np.unique(current[current>0],return_counts=True)))
    d.to_pickle(config.obj_dir/'grouped_z_objects.pkl')
    return d


## 14. Select best Z by predicted offset; save maximum-area comparison


In [ ]:
def select_best_z_per_nucleus(grouped_z_df, config):
    d=grouped_z_df.copy()
    if d.empty:
        d.to_pickle(config.obj_dir/'best_z_nuclei.pkl'); return d
    keys=['t','nucleus_3d_id']
    by_area=d.sort_values('area_px',ascending=False).drop_duplicates(keys)
    if config.best_z_mode=='argmin_z_offset':
        if not np.isfinite(d.mean_z_offset).all(): raise ValueError('Missing Z-offset measurements')
        best=d.sort_values(['mean_z_offset','area_px'],ascending=[True,False]).drop_duplicates(keys)
    elif config.best_z_mode=='max_area': best=by_area
    else: raise ValueError('Unknown best_z_mode')
    diagnostic=best[keys+['z','area_px']].merge(by_area[keys+['z','area_px']],on=keys,suffixes=('_selected','_max_area'))
    diagnostic.to_csv(config.qc_dir/'best_z_comparison.csv',index=False)
    best=best.sort_values(keys).reset_index(drop=True)
    best['n_planes']=best.set_index(keys).index.map(d.groupby(keys).z.nunique()).to_numpy()
    best.to_pickle(config.obj_dir/'best_z_nuclei.pkl')
    return best


## 15. Exclude likely multi-nucleus droplets

In [ ]:
def flag_close_nuclei(best_z_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if best_z_df.empty:
        return pd.DataFrame()
    out = []
    for _, df_t in best_z_df.groupby("t", sort=True):
        df_t = df_t.copy().reset_index(drop=True)
        coords = df_t[["centroid_x_px", "centroid_y_px"]].to_numpy()
        tree = cKDTree(coords)
        close_flags = np.zeros(len(df_t), dtype=bool)
        for i, xy in enumerate(coords):
            neighbors = [j for j in tree.query_ball_point(xy, r=config.multi_nucleus_exclusion_px) if j != i]
            if neighbors:
                close_flags[i] = True
                for j in neighbors:
                    close_flags[j] = True
        df_t["valid_single_nucleus"] = ~close_flags
        out.append(df_t)
    filtered_df = pd.concat(out, ignore_index=True)
    filtered_df.to_pickle(config.obj_dir / "best_z_nuclei_with_exclusion.pkl")
    return filtered_df


## 16. Tile assignment and true acquisition time

In [ ]:
def assign_tile_index_from_xy(
    x_px: float, y_px: float,
    image_width_px: int, image_height_px: int,
    config: PipelineConfig,
) -> Tuple[int, int, int]:
    col = min(int(x_px // (image_width_px  / config.tile_cols)), config.tile_cols - 1)
    row = min(int(y_px // (image_height_px / config.tile_rows)), config.tile_rows - 1)
    if config.serpentine_scan and row % 2 != 0:
        tile_index = row * config.tile_cols + (config.tile_cols - 1 - col)
    else:
        tile_index = row * config.tile_cols + col
    return row, col, tile_index


def add_tile_timing_metadata(
    nuclei_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    if nuclei_df.empty:
        return pd.DataFrame()
    image_height_px, image_width_px = img_5d.shape[-2], img_5d.shape[-1]
    out_rows = []
    tiles_per_frame = config.tile_rows * config.tile_cols * config.minutes_per_tile
    for row in nuclei_df.itertuples(index=False):
        tile_row, tile_col, tile_index = assign_tile_index_from_xy(
            row.centroid_x_px, row.centroid_y_px,
            image_width_px, image_height_px, config)
        d = row._asdict()
        d["tile_row"]        = tile_row
        d["tile_col"]        = tile_col
        d["tile_index"]      = tile_index
        d["tile_offset_min"] = tile_index * config.minutes_per_tile
        d["true_time_min"]   = row.t * tiles_per_frame + d["tile_offset_min"]
        out_rows.append(d)
    timed_df = pd.DataFrame(out_rows)
    timed_df.to_pickle(config.track_dir / "best_z_nuclei_timed.pkl")
    return timed_df


## 17. Track nuclei across time — hybrid DBSCAN tracker

1. Build local spatial neighbourhoods with DBSCAN from consecutive frames.
2. Score candidate links by distance + area consistency.
3. One-to-one Hungarian assignment inside each local neighbourhood.
4. Reject crowded or low-confidence links rather than forcing a track.


In [ ]:
def _safe_area_ratio(area_a: float, area_b: float) -> float:
    a, b = max(float(area_a), 1.0), max(float(area_b), 1.0)
    return max(a, b) / min(a, b)


def build_dbscan_candidate_clusters(
    prev_df: pd.DataFrame,
    curr_df: pd.DataFrame,
    config: PipelineConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if DBSCAN is None:
        raise ImportError("scikit-learn is required for DBSCAN-assisted tracking.")

    prev_local = prev_df.copy().reset_index(drop=True)
    curr_local = curr_df.copy().reset_index(drop=True)
    prev_local["frame_role"] = "prev"
    curr_local["frame_role"] = "curr"
    prev_local["frame_local_index"] = np.arange(len(prev_local))
    curr_local["frame_local_index"] = np.arange(len(curr_local))

    pooled = pd.concat([prev_local, curr_local], ignore_index=True)
    if pooled.empty:
        for col in ("dbscan_cluster_id", "dbscan_cluster_size", "dbscan_is_crowded"):
            pooled[col] = pd.Series(dtype=int if col != "dbscan_is_crowded" else bool)
        return prev_local, curr_local, pooled

    coords = pooled[["centroid_x_px", "centroid_y_px"]].to_numpy()
    labels = DBSCAN(eps=config.track_dbscan_eps_px,
                    min_samples=config.track_dbscan_min_samples).fit_predict(coords)
    pooled["dbscan_cluster_id"] = labels

    crowded_flags = []
    cluster_sizes = []
    for _, df_c in pooled.groupby("dbscan_cluster_id", sort=False):
        n_prev = int((df_c["frame_role"] == "prev").sum())
        n_curr = int((df_c["frame_role"] == "curr").sum())
        is_crowded = n_prev > 1 or n_curr > 1
        crowded_flags.extend([is_crowded] * len(df_c))
        cluster_sizes.extend([len(df_c)] * len(df_c))
    pooled["dbscan_cluster_size"] = cluster_sizes
    pooled["dbscan_is_crowded"]   = crowded_flags

    prev_out = pooled[pooled["frame_role"] == "prev"].copy().reset_index(drop=True)
    curr_out = pooled[pooled["frame_role"] == "curr"].copy().reset_index(drop=True)
    return prev_out, curr_out, pooled


def match_cluster_with_assignment(
    prev_cluster: pd.DataFrame,
    curr_cluster: pd.DataFrame,
    config: PipelineConfig,
) -> List[dict]:
    if prev_cluster.empty or curr_cluster.empty:
        return []

    n_prev, n_curr = len(prev_cluster), len(curr_cluster)
    cost_matrix = np.full((n_prev, n_curr), np.nan, dtype=float)
    max_dist_base = float(config.time_track_tolerance_px)

    for i, prev_row in prev_cluster.iterrows():
        for j, curr_row in curr_cluster.iterrows():
            dx  = float(curr_row["centroid_x_px"]) - float(prev_row["centroid_x_px"])
            dy  = float(curr_row["centroid_y_px"]) - float(prev_row["centroid_y_px"])
            dist_px = math.hypot(dx, dy)

            max_dist = max_dist_base
            if prev_row.get("dbscan_is_crowded", False) or curr_row.get("dbscan_is_crowded", False):
                max_dist *= config.track_crowded_distance_scale

            prev_area = max(float(prev_row["area_px"]), 1.0)
            curr_area = max(float(curr_row["area_px"]), 1.0)
            area_ratio = max(prev_area, curr_area) / min(prev_area, curr_area)

            if dist_px <= max_dist and area_ratio <= config.track_area_ratio_max:
                area_cost    = abs(math.log(curr_area / prev_area))
                dist_cost    = dist_px / max(max_dist, 1e-6)
                cost_matrix[i, j] = dist_cost + config.track_area_log_weight * area_cost

    finite_mask = np.isfinite(cost_matrix)
    valid_rows  = np.where(finite_mask.any(axis=1))[0]
    valid_cols  = np.where(finite_mask.any(axis=0))[0]
    if len(valid_rows) == 0 or len(valid_cols) == 0:
        return []

    large_penalty = 1e9
    reduced = np.where(np.isfinite(cost_matrix[np.ix_(valid_rows, valid_cols)]),
                       cost_matrix[np.ix_(valid_rows, valid_cols)], large_penalty)

    bad_rows = np.all(reduced >= large_penalty, axis=1)
    bad_cols = np.all(reduced >= large_penalty, axis=0)
    if bad_rows.any() or bad_cols.any():
        keep_r = np.where(~bad_rows)[0]
        keep_c = np.where(~bad_cols)[0]
        if len(keep_r) == 0 or len(keep_c) == 0:
            return []
        reduced    = reduced   [np.ix_(keep_r, keep_c)]
        valid_rows = valid_rows[keep_r]
        valid_cols = valid_cols[keep_c]

    row_ind, col_ind = linear_sum_assignment(reduced)

    matches = []
    for r, c in zip(row_ind, col_ind):
        orig_i = valid_rows[r]
        orig_j = valid_cols[c]
        link_cost = cost_matrix[orig_i, orig_j]
        if not np.isfinite(link_cost):
            continue
        pr = prev_cluster.iloc[orig_i]
        cr = curr_cluster.iloc[orig_j]
        dist_px = math.hypot(
            float(cr["centroid_x_px"]) - float(pr["centroid_x_px"]),
            float(cr["centroid_y_px"]) - float(pr["centroid_y_px"]),
        )
        matches.append({
            "prev_index":        int(pr["global_index"]),
            "curr_index":        int(cr["global_index"]),
            "link_cost":         float(link_cost),
            "link_distance_px":  float(dist_px),
            "dbscan_cluster_id":   int(pr.get("dbscan_cluster_id", -1)),
            "dbscan_cluster_size": int(pr.get("dbscan_cluster_size", 0)),
            "dbscan_is_crowded":   bool(pr.get("dbscan_is_crowded", False)),
        })
    return matches


def assign_track_ids_hybrid_dbscan(
    timed_df: pd.DataFrame,
    config: PipelineConfig,
    valid_only: bool = False,
) -> pd.DataFrame:
    if timed_df.empty:
        return pd.DataFrame()

    df = timed_df.copy().reset_index(drop=True)
    df["global_index"] = np.arange(len(df), dtype=int)
    df = df.sort_values(["t", "nucleus_3d_id"]).reset_index(drop=True)

    df["track_id"]                = -1
    df["track_link_distance_px"]  = np.nan
    df["track_link_cost"]         = np.nan
    df["track_dbscan_cluster_id"] = -1
    df["track_dbscan_cluster_size"] = 0
    df["track_dbscan_is_crowded"] = False
    df["track_link_method"]       = "unassigned"

    time_values = sorted(df["t"].unique())
    if not time_values:
        return df

    next_track_id = 1
    for idx in df.index[df["t"] == time_values[0]]:
        df.at[idx, "track_id"]          = next_track_id
        df.at[idx, "track_link_method"] = "seed"
        next_track_id += 1

    for t_prev, t_curr in zip(time_values[:-1], time_values[1:]):
        prev_df = df[df["t"] == t_prev].reset_index()
        curr_df = df[df["t"] == t_curr].reset_index()

        prev_clustered, curr_clustered, pooled = build_dbscan_candidate_clusters(
            prev_df, curr_df, config)

        all_matches: List[dict] = []
        for cluster_id in sorted(pooled["dbscan_cluster_id"].unique()):
            pc = prev_clustered[prev_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            cc = curr_clustered[curr_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            all_matches.extend(match_cluster_with_assignment(pc, cc, config))

        matched_curr_global: set = set()

        # FIX: loop body was previously unreachable due to a misindented `continue`.
        for match in sorted(all_matches, key=lambda x: x["link_cost"]):
            prev_global = int(match["prev_index"])
            curr_global = int(match["curr_index"])

            if curr_global in matched_curr_global:
                continue

            track_id = int(df.at[prev_global, "track_id"])
            if track_id == -1:
                track_id = next_track_id
                df.at[prev_global, "track_id"]          = track_id
                df.at[prev_global, "track_link_method"] = "backfilled_seed"
                next_track_id += 1

            matched_curr_global.add(curr_global)
            df.at[curr_global, "track_id"]                  = track_id
            df.at[curr_global, "track_link_distance_px"]    = float(match["link_distance_px"])
            df.at[curr_global, "track_link_cost"]           = float(match["link_cost"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(match["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(match["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(match["dbscan_is_crowded"])
            df.at[curr_global, "track_link_method"]         = "hybrid_dbscan"

        for _, row in curr_clustered.iterrows():
            curr_global = int(row["index"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(row["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(row["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(row["dbscan_is_crowded"])
            if df.at[curr_global, "track_id"] == -1:
                df.at[curr_global, "track_id"]          = next_track_id
                df.at[curr_global, "track_link_method"] = "new_track_unmatched"
                next_track_id += 1

    df.to_pickle(config.track_dir / "tracked_nuclei.pkl")
    return df


def summarize_tracking_debug(tracked_df: pd.DataFrame) -> pd.DataFrame:
    if tracked_df.empty:
        return pd.DataFrame()
    track_len = tracked_df.groupby("track_id").size()
    return pd.DataFrame([{
        "n_rows":                int(len(tracked_df)),
        "n_tracks":              int(tracked_df["track_id"].nunique()),
        "median_track_length":   float(track_len.median()),
        "mean_track_length":     float(track_len.mean()),
        "fraction_crowded_links":float(tracked_df["track_dbscan_is_crowded"].fillna(False).mean()),
        "fraction_new_tracks":   float((tracked_df["track_link_method"] == "new_track_unmatched").mean()),
    }])


## 18. Fiji-equivalent cumulative halos

In [ ]:
def build_cumulative_halo_masks(nucleus_mask: np.ndarray, config: PipelineConfig) -> Dict[str, np.ndarray]:
    dist_outside = ndi.distance_transform_edt(~nucleus_mask)
    masks = {"nucleus_mask": nucleus_mask.astype(bool)}

    for i in range(1, config.n_halos + 1):
        masks[f"halo_{i}_cum"] = dist_outside <= i * config.halo_step_px

    for i in range(1, config.n_halos + 1):
        prev = nucleus_mask if i == 1 else masks[f"halo_{i-1}_cum"]
        masks[f"ring_{i}"] = masks[f"halo_{i}_cum"] & ~prev

    masks["nucleus_eroded"] = (
        ndi.binary_erosion(nucleus_mask, structure=morphology.disk(config.erosion_px))
        if config.erosion_px > 0 else nucleus_mask.copy()
    )
    masks["cytoplasm_mask"] = masks[f"halo_{config.n_halos}_cum"] & ~nucleus_mask
    return masks


def _mask_stats(intensity_image: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
    area_px = int(mask.sum())
    if area_px == 0:
        return {"area_px": 0, "intden": 0.0, "mean_intensity": np.nan}
    vals = intensity_image[mask]
    return {"area_px": area_px, "intden": float(vals.sum()), "mean_intensity": float(vals.mean())}


def measure_fiji_equivalent_halos(
    intensity_image: np.ndarray,
    nucleus_mask: np.ndarray,
    config: PipelineConfig,
) -> Dict[str, float]:
    masks = build_cumulative_halo_masks(nucleus_mask, config)
    out: Dict[str, float] = {}

    for key in ("nucleus_mask", "nucleus_eroded", "cytoplasm_mask"):
        prefix = key.replace("_mask", "").replace("_cum", "")
        for k, v in _mask_stats(intensity_image, masks[key]).items():
            out[f"{prefix}_{k}"] = v

    for i in range(1, config.n_halos + 1):
        for k, v in _mask_stats(intensity_image, masks[f"halo_{i}_cum"]).items():
            out[f"halo_{i}_cum_{k}"] = v
        for k, v in _mask_stats(intensity_image, masks[f"ring_{i}"]).items():
            out[f"ring_{i}_{k}"] = v

    nm = out.get("nucleus_mean_intensity", np.nan)
    cm = out.get("cytoplasm_mean_intensity", np.nan)
    out["nc_ratio"]          = float(nm / cm)          if np.isfinite(nm) and np.isfinite(cm) and cm != 0         else np.nan
    out["nc_ratio_fraction"] = float(nm / (nm + cm))   if np.isfinite(nm) and np.isfinite(cm) and (nm + cm) != 0  else np.nan
    return out


def recover_nucleus_mask_from_plane(t_idx,z_idx,centroid_x_px,centroid_y_px,config):
    lab=tiff.memmap(str(config.nucleus_instance_hyperstack_path),mode='r')[t_idx,z_idx]
    regions=measure.regionprops(lab)
    if not regions: return np.zeros(lab.shape,bool)
    best=min(regions,key=lambda r:(r.centroid[1]-centroid_x_px)**2+(r.centroid[0]-centroid_y_px)**2)
    return lab==best.label


def _repaired_mask_path(config: PipelineConfig, t: int, nucleus_3d_id: int) -> Path:
    """Shared naming convention between repair_fragmented_nuclei (writer,
    Section 30b) and recover_nucleus_mask_for_row (reader, here)."""
    return config.obj_dir / "repaired_masks" / f"t{int(t):03d}_nuc{int(nucleus_3d_id):03d}.npy"


def recover_nucleus_mask_for_row(row,config):
    if config.repair_enabled:
        p=_repaired_mask_path(config,row.t,row.nucleus_3d_id)
        if p.exists(): return np.load(p).astype(bool)
    lab=tiff.memmap(str(config.nucleus_instance_hyperstack_path),mode='r')[int(row.t),int(row.z)]
    label=int(row.label)
    if not (lab==label).any(): raise ValueError('Instance missing for measured row')
    return lab==label


def _process_single_tracked_nucleus(row, img_5d: np.ndarray, config: PipelineConfig) -> dict:
    nucleus_mask  = recover_nucleus_mask_for_row(row, config)
    nuclear_plane = get_nuclear_plane(img_5d, int(row.t), int(row.z), config)
    halo_metrics  = measure_fiji_equivalent_halos(nuclear_plane, nucleus_mask, config)
    rec = row._asdict()
    rec.update(halo_metrics)
    rec["nucleus_area_um2"]   = rec.get("nucleus_area_px", 0) * config.pixel_size_um ** 2
    rec["cytoplasm_area_um2"] = rec.get("cytoplasm_area_px", 0) * config.pixel_size_um ** 2
    return rec


def run_halo_analysis_for_tracked_nuclei(
    tracked_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    if tracked_df.empty:
        halo_df = pd.DataFrame()
        halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
        return halo_df
    rows = Parallel(n_jobs=max(1, int(config.n_workers)), backend="threading", batch_size=1)(
        delayed(_process_single_tracked_nucleus)(row, img_5d, config)
        for row in tracked_df.itertuples(index=False)
    )
    halo_df = pd.DataFrame(rows)
    halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
    return halo_df


## 19. Radial sweep (droplet-bounded)

Replaces the v15 placeholder. Casts rays from each tracked nucleus centroid outward to the edge of its assigned droplet, sampling the membrane / NE channel. Runs on v15's already-tracked nuclei at their selected best-Z, using v15's own droplet instance masks — so `Nucleus_ID` (`track_id`) matches automatically with no reconciliation.

Two methodological choices are carried over unchanged from the standalone `Halo_Radial_Sweep` notebook (both previously flagged as future work):
- nearest-pixel sampling (`int(round(...))`), not sub-pixel interpolation;
- only *this* nucleus's interior is excluded — rays are not clipped at the territory boundary with a neighbouring nucleus sharing the same droplet.


In [ ]:
# =============================================================================
# 19. Radial sweep (droplet-bounded)  [v16]
# =============================================================================
# Reuses v15 artefacts:
#   - recover_nucleus_mask_from_plane(...) for the per-nucleus mask (Section 18)
#   - droplet_instance_hyperstack.tif (TZYX) written by segmentation (Section 27)
# Iterates the SAME tracked_df the halo analysis consumes, so every row already
# carries a persistent track_id == schema Nucleus_ID.

_DROPLET_STACK_CACHE: Dict[str, np.ndarray] = {}


def load_droplet_instance_stack(config: PipelineConfig) -> np.ndarray:
    """Load (and cache) the TZYX droplet instance hyperstack from segmentation."""
    key = str(config.droplet_instance_hyperstack_path)
    if key not in _DROPLET_STACK_CACHE:
        if not config.droplet_instance_hyperstack_path.exists():
            raise FileNotFoundError(
                f"Droplet instance hyperstack not found: "
                f"{config.droplet_instance_hyperstack_path}. Run segmentation "
                f"(Section 27) first, or point cfg at an existing run."
            )
        _DROPLET_STACK_CACHE[key] = tiff.imread(config.droplet_instance_hyperstack_path)
    return _DROPLET_STACK_CACHE[key]


def assign_nucleus_to_droplet(nucleus_mask: np.ndarray,
                              droplet_labeled_2d: np.ndarray) -> int:
    """Droplet label with greatest overlap with this nucleus (0 if no overlap)."""
    overlapping = droplet_labeled_2d[nucleus_mask]
    overlapping = overlapping[overlapping > 0]
    if overlapping.size == 0:
        return 0
    labels, counts = np.unique(overlapping, return_counts=True)
    return int(labels[np.argmax(counts)])


def sample_ray_to_droplet_edge(
    image_2d, droplet_mask, nucleus_mask, center_y, center_x, theta,
    step_size=1.0, exclude_nucleus=True, max_steps=4000,
):
    """Sample one radial ray from the centroid outward until it leaves the droplet.
    All distances in pixels. Nearest-pixel sampling (see Section 19 notes)."""
    ys, xs, vals, dists, inside_flags = [], [], [], [], []
    for step in range(max_steps):
        r = step * step_size
        y = center_y + r * np.sin(theta)
        x = center_x + r * np.cos(theta)
        yi = int(round(y)); xi = int(round(x))
        if yi < 0 or yi >= image_2d.shape[0] or xi < 0 or xi >= image_2d.shape[1]:
            break
        if not droplet_mask[yi, xi]:
            break
        is_nucleus = bool(nucleus_mask[yi, xi])
        if exclude_nucleus and is_nucleus:
            continue
        ys.append(yi); xs.append(xi); vals.append(image_2d[yi, xi])
        dists.append(r); inside_flags.append(is_nucleus)
    return {
        "y": np.asarray(ys, dtype=int),
        "x": np.asarray(xs, dtype=int),
        "values": np.asarray(vals, dtype=float),
        "distance_px": np.asarray(dists, dtype=float),
        "inside_nucleus": np.asarray(inside_flags, dtype=bool),
    }


def radial_profile_for_tracked_nucleus(
    radial_plane, nucleus_mask, droplet_mask, track_id, droplet_id, t, z,
    center_y, center_x, n_angles=180, step_size=1.0,
    exclude_nucleus=True, max_steps=4000,
) -> pd.DataFrame:
    """Long-form radial profile for one nucleus: one row per sampled ray point.
    distance_norm is the per-ray fraction of the distance to the droplet edge
    (schema Rho_Normalized)."""
    rows = []
    thetas = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    for theta_idx, theta in enumerate(thetas):
        ray = sample_ray_to_droplet_edge(
            radial_plane, droplet_mask, nucleus_mask, center_y, center_x, theta,
            step_size=step_size, exclude_nucleus=exclude_nucleus, max_steps=max_steps)
        if ray["distance_px"].size == 0:
            continue
        ray_len = float(ray["distance_px"][-1])
        norm = (np.zeros_like(ray["distance_px"]) if np.isclose(ray_len, 0)
                else ray["distance_px"] / ray_len)
        for i in range(ray["distance_px"].size):
            rows.append({
                "track_id": int(track_id), "t": int(t), "z": int(z),
                "droplet_id": int(droplet_id),
                "theta_index": theta_idx, "theta_rad": float(theta),
                "y": int(ray["y"][i]), "x": int(ray["x"][i]),
                "distance_px": float(ray["distance_px"][i]),
                "distance_norm": float(norm[i]),
                "intensity": float(ray["values"][i]),
                "ray_length_px": ray_len,
                "inside_nucleus": bool(ray["inside_nucleus"][i]),
            })
    return pd.DataFrame(rows)


def _process_single_tracked_nucleus_radial(row, img_5d, droplet_stack,
                                           config: PipelineConfig) -> pd.DataFrame:
    t, z = int(row.t), int(row.z)
    cx, cy = float(row.centroid_x_px), float(row.centroid_y_px)
    nucleus_mask = recover_nucleus_mask_for_row(row, config)
    droplet_labeled_2d = droplet_stack[t, z]
    droplet_id = assign_nucleus_to_droplet(nucleus_mask, droplet_labeled_2d)
    if droplet_id == 0:
        return pd.DataFrame()   # no assigned droplet -> no bounded sweep
    droplet_mask = droplet_labeled_2d == droplet_id
    radial_plane = img_5d[t, z, config.radial_channel_index]
    return radial_profile_for_tracked_nucleus(
        radial_plane, nucleus_mask, droplet_mask,
        track_id=int(row.track_id), droplet_id=droplet_id, t=t, z=z,
        center_y=cy, center_x=cx,
        n_angles=config.radial_n_angles,
        step_size=config.radial_step_size_px,
        exclude_nucleus=config.radial_exclude_nucleus,
        max_steps=config.radial_max_ray_steps)


def run_radial_sweep_for_tracked_nuclei(
    tracked_df: pd.DataFrame, img_5d: np.ndarray, config: PipelineConfig,
) -> pd.DataFrame:
    """Radial sweep over every tracked nucleus at its selected best-Z.
    Returns a long-form DataFrame (one row per ray sample point) and pickles it."""
    out_path = config.analysis_dir / "radial_sweep.pkl"
    if tracked_df.empty:
        radial_df = pd.DataFrame()
        radial_df.to_pickle(out_path)
        return radial_df

    droplet_stack = load_droplet_instance_stack(config)
    assert droplet_stack.shape[0] == img_5d.shape[0], "Droplet stack T mismatch vs image"
    assert droplet_stack.shape[1] == img_5d.shape[1], "Droplet stack Z mismatch vs image"
    assert droplet_stack.shape[-2:] == img_5d.shape[-2:], "Droplet stack Y/X mismatch vs image"

    dfs = Parallel(n_jobs=max(1, int(config.n_workers)), backend="threading", batch_size=1)(
        delayed(_process_single_tracked_nucleus_radial)(row, img_5d, droplet_stack, config)
        for row in tracked_df.itertuples(index=False)
    )
    dfs = [d for d in dfs if not d.empty]
    radial_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    radial_df.to_pickle(out_path)
    return radial_df


## 20. Portable export helper

In [ ]:
def _save_portable_table(df: pd.DataFrame, out_stem: Path) -> dict:
    """Save a table as CSV (human-readable) + pickle (lossless, fast).
    Parquet removed — requires pyarrow/fastparquet which are not available."""
    out_stem.parent.mkdir(parents=True, exist_ok=True)
    saved: dict = {}
    if df is None:
        return saved
    csv_path = out_stem.with_suffix(".csv")
    df.to_csv(csv_path, index=False)
    saved["csv"] = str(csv_path)
    pkl_path = out_stem.with_suffix(".pkl")
    df.to_pickle(pkl_path)
    saved["pickle"] = str(pkl_path)
    return saved


def export_portable_analysis_bundle(
    config: PipelineConfig,
    export_dir: Union[str, Path],
    seg_index_df: Optional[pd.DataFrame] = None,
    objects_df: Optional[pd.DataFrame] = None,
    grouped_z_df: Optional[pd.DataFrame] = None,
    best_z_df: Optional[pd.DataFrame] = None,
    filtered_df: Optional[pd.DataFrame] = None,
    timed_df: Optional[pd.DataFrame] = None,
    tracked_df: Optional[pd.DataFrame] = None,
    halo_df: Optional[pd.DataFrame] = None,
    copy_mask_npy_files: bool = False,
) -> Path:
    export_dir = Path(export_dir)
    dirs = {k: export_dir / k for k in ("tables", "masks", "config", "notes")}
    for p in [export_dir] + list(dirs.values()):
        p.mkdir(parents=True, exist_ok=True)

    manifest = {
        "created_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "input_image_name": config.input_image_name,
        "input_image_path_on_cluster": str(config.input_image_path),
        "pixel_size_um": float(config.pixel_size_um),
        "z_step_um": float(config.z_step_um),
        "nuclear_channel_index": int(config.nuclear_channel_index),
        "membrane_channel_index": int(config.membrane_channel_index),
        "copied_files": {}, "tables": {},
    }

    config_path = dirs["config"] / "pipeline_config.json"
    config_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))
    manifest["copied_files"]["pipeline_config"] = str(config_path)

    for key, src in {
        "segmentation_class_hyperstack":       config.segmentation_class_hyperstack_path,
        "segmentation_probability_hyperstack": config.segmentation_probability_hyperstack_path,
        "segmentation_label_hyperstack":       config.segmentation_label_hyperstack_path,
        "nucleus_instance_hyperstack":         config.nucleus_instance_hyperstack_path,
        "droplet_instance_hyperstack":         config.droplet_instance_hyperstack_path,
    }.items():
        src = Path(src)
        if src.exists():
            dst = dirs["masks"] / src.name
            shutil.copy2(src, dst)
            manifest["copied_files"][key] = str(dst)

    for name, df in {
        "segmentation_index":           seg_index_df,
        "plane_objects":                objects_df,
        "grouped_z_objects":            grouped_z_df,
        "best_z_nuclei":                best_z_df,
        "best_z_nuclei_with_exclusion": filtered_df,
        "best_z_nuclei_timed":          timed_df,
        "tracked_nuclei":               tracked_df,
        "halo_analysis":                halo_df,
    }.items():
        if df is not None:
            manifest["tables"][name] = _save_portable_table(df, dirs["tables"] / name)

    if copy_mask_npy_files and seg_index_df is not None and "mask_path" in seg_index_df.columns:
        plane_mask_dir = dirs["masks"] / "plane_npy_masks"
        plane_mask_dir.mkdir(parents=True, exist_ok=True)
        portable_idx = seg_index_df.copy()
        new_paths = []
        for row in portable_idx.itertuples(index=False):
            src = Path(row.mask_path)
            if src.exists():
                dst = plane_mask_dir / src.name
                shutil.copy2(src, dst)
                new_paths.append(str(dst))
            else:
                new_paths.append(None)
        portable_idx["portable_mask_path"] = new_paths
        manifest["tables"]["segmentation_index_portable"] = _save_portable_table(
            portable_idx, dirs["tables"] / "segmentation_index_portable")

    readme = (
        "Portable laptop analysis bundle\n\n"
        "Contents\n--------\n"
        "masks/  — Hyperstack TIFFs for segmentation QC, instance labels, and napari overlays.\n"
        "tables/ — Analysis tables (CSV + pickle).\n"
        "config/ — Pipeline configuration snapshot.\n\n"
        "Recommended workflow\n--------------------\n"
        "1. Open TIFF files in napari for segmentation debugging.\n"
        "2. Load halo_analysis.pkl or .csv for plotting and filtering.\n"
        "3. Use tracked_nuclei / best_z_nuclei tables for re-analysis without rerunning the model.\n"
    )
    (dirs["notes"] / "README.txt").write_text(readme)
    (export_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    return export_dir


## 21. QC plotting helpers

In [ ]:
def plot_nucleus_mask_overlay_qc(
    img_5d: np.ndarray,
    nucleus_mask_4d: np.ndarray,
    config: PipelineConfig,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Overlay raw nuclear channel with the binary nucleus mask."""
    T, Z, C, Y, X = img_5d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()

    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, t_idx in zip(axes, timepoints):
        z_idx = (Z // 2 if z_mode == "middle"
                 else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1))))
        raw = img_5d[t_idx, z_idx, config.nuclear_channel_index].astype(np.float32)
        if raw.max() > 0:
            raw /= raw.max()
        ax.imshow(raw, cmap="gray")
        ax.imshow(nucleus_mask_4d[t_idx, z_idx].astype(bool), alpha=0.35)
        ax.set_title(f"t={t_idx}, z={z_idx}")
        ax.axis("off")

    for ax in axes[len(timepoints):]:
        ax.axis("off")

    fig.suptitle("QC: Nuclear channel + nucleus mask overlay", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_nucleus_mask_qc_summary(
    nucleus_mask_4d: np.ndarray,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Mask snapshots with contours + total nuclear area over time."""
    from skimage.measure import find_contours
    T, Z, Y, X = nucleus_mask_4d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()

    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig = plt.figure(figsize=(4 * n_cols, 4 * (n_rows + 1)))
    gs  = fig.add_gridspec(n_rows + 1, n_cols)

    panel_axes = [fig.add_subplot(gs[r, c]) for r in range(n_rows) for c in range(n_cols)]

    for ax, t_idx in zip(panel_axes, timepoints):
        z_idx = (Z // 2 if z_mode == "middle"
                 else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1))))
        mask_plane = nucleus_mask_4d[t_idx, z_idx].astype(bool)
        labeled    = measure.label(mask_plane)
        ax.imshow(mask_plane, cmap="gray")
        for contour in find_contours(mask_plane.astype(float), level=0.5):
            ax.plot(contour[:, 1], contour[:, 0], linewidth=0.5)
        ax.set_title(f"t={t_idx}, z={z_idx}\nobjects={labeled.max()}")
        ax.axis("off")

    for ax in panel_axes[len(timepoints):]:
        ax.axis("off")

    area_ax = fig.add_subplot(gs[n_rows, :])
    area_ax.plot(np.arange(T), nucleus_mask_4d.reshape(T, -1).sum(axis=1), marker="o")
    area_ax.set_xlabel("Time index")
    area_ax.set_ylabel("Total nucleus mask area (px)")
    area_ax.set_title("QC: Total segmented nuclear area over time")
    plt.tight_layout()
    plt.show()


def plot_nucleus_count_over_time(nucleus_mask_4d: np.ndarray) -> None:
    """Count connected nucleus objects across all Z slices per time point."""
    T, Z = nucleus_mask_4d.shape[:2]
    counts = [
        sum(measure.label(nucleus_mask_4d[t, z].astype(bool)).max() for z in range(Z))
        for t in range(T)
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(T), counts, marker="o")
    ax.set_xlabel("Time index")
    ax.set_ylabel("Connected nucleus objects")
    ax.set_title("QC: Connected nucleus object count over time")
    plt.tight_layout()
    plt.show()


def plot_best_focus_nuclear_planes(
    img_5d: np.ndarray,
    config: PipelineConfig,
    figsize_per_panel: float = 4.0,
) -> pd.DataFrame:
    """Plot the single best-focus nuclear-channel Z plane for each time point."""
    n_t = img_5d.shape[0]
    n_cols = min(5, n_t)
    n_rows = math.ceil(n_t / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(figsize_per_panel * n_cols, figsize_per_panel * n_rows))
    axes = np.array(axes).reshape(-1)
    focus_rows = []

    for t in range(n_t):
        keep_z, scores, best_z = get_best_focus_z_indices(
            img_5d, t,
            nuc_channel_idx=config.effective_focus_channel_index,
            exclude_edge_z=config.focus_edge_z_exclusion,
            window_radius=0,
            metric=config.focus_metric,
            pixel_size_um=config.pixel_size_um,
            config=config,
        )
        ax = axes[t]
        if best_z is None:
            ax.set_title(f"t={t}: no valid focus plane")
            ax.axis("off")
            focus_rows.append({"t": t, "best_z": np.nan, "focus_score": np.nan,
                                "focus_metric": config.focus_metric,
                                "nuclear_channel_index": config.effective_focus_channel_index})
            continue
        nuc2d      = img_5d[t, best_z, config.effective_focus_channel_index]
        best_score = float(scores[best_z])
        ax.imshow(nuc2d, cmap="gray")
        ax.set_title(f"t={t}, best z={best_z}\nfocus={best_score:.4g}")
        ax.axis("off")
        focus_rows.append({"t": t, "best_z": int(best_z), "focus_score": best_score,
                           "focus_metric": config.focus_metric,
                           "nuclear_channel_index": config.effective_focus_channel_index})

    for ax in axes[n_t:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(focus_rows)


def plot_nucleus_probability_qc(class_prob_map: np.ndarray, raw_plane: np.ndarray,
                                title: str = "") -> None:
    """Show raw plane, nucleus probability map, and its histogram."""
    nucleus_prob = class_prob_map[..., 2]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw_plane, cmap="gray"); axes[0].set_title("Raw"); axes[0].axis("off")
    im = axes[1].imshow(nucleus_prob, cmap="inferno")
    axes[1].set_title("Nucleus probability"); plt.colorbar(im, ax=axes[1])
    axes[2].hist(nucleus_prob.ravel(), bins=100); axes[2].set_title("Probability distribution")
    fig.suptitle(title); plt.tight_layout(); plt.show()


def plot_nc_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot."); return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nc_ratio_fraction"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("N / (N + C)")
    ax.set_title("N/C ratio fraction by track")
    plt.tight_layout(); plt.show()


def plot_area_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot."); return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nucleus_area_um2"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear area (µm²)")
    ax.set_title("Largest nuclear cross-sectional area by track")
    plt.tight_layout(); plt.show()


def plot_largest_cross_sectional_area_vs_time(
    df: pd.DataFrame,
    config: PipelineConfig,
) -> None:
    """
    Scatter all individual nuclear areas plus population mean vs true acquisition time.

    Works with timed_df, tracked_df, or halo_df.  Requires 'true_time_min'.
    """
    if df.empty:
        print("No data to plot."); return
    plot_df = df.copy()
    if "nucleus_area_um2" not in plot_df.columns:
        if "area_px" not in plot_df.columns:
            raise ValueError("DataFrame must contain 'nucleus_area_um2' or 'area_px'.")
        plot_df["nucleus_area_um2"] = plot_df["area_px"] * config.pixel_size_um ** 2
    if "true_time_min" not in plot_df.columns:
        raise ValueError("DataFrame must contain 'true_time_min'.")

    plot_df = plot_df.sort_values("true_time_min")
    mean_df = (plot_df.groupby("true_time_min", as_index=False)["nucleus_area_um2"]
               .mean().rename(columns={"nucleus_area_um2": "mean_area"}))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_df["true_time_min"], plot_df["nucleus_area_um2"], alpha=0.35, s=20,
               label="Individual nuclei")
    ax.plot(mean_df["true_time_min"], mean_df["mean_area"], linewidth=2, label="Population mean")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear cross-sectional area (µm²)")
    ax.set_title("Nuclear area vs time")
    ax.legend(); plt.tight_layout(); plt.show()


## 22. Stage execution flags

Set a flag to `True` to recompute that stage; `False` loads the saved artefact from disk.

In [ ]:
# -- Stage execution flags -----------------------------------------------
# Segmentation (§27) has NO RUN_/RESUME_ flag: it reads its on-disk checkpoint
# (seg_dir/_checkpoint.json) and decides fresh / resume / complete automatically.
# To force a rebuild, call reset_segmentation(cfg, scope=...) in the §27 reset
# cell, then re-run §27.
SAVE_PROBABILITY_TIFF  = True    # write the ~35 GB probability hyperstack (enables no-GPU re-extraction)
RUN_SINGLE_PLANE_DEBUG = False   # §26 / cell-after-§27 single-plane inspection

RUN_OBJECT_EXTRACTION = True
RUN_GROUP_ACROSS_Z    = True
RUN_SELECT_BEST_Z     = True
RUN_FLAG_CLOSE_NUCLEI = True
RUN_ADD_TIMING        = True
RUN_TRACKING          = True
RUN_HALO_ANALYSIS     = True
RUN_RADIAL_SWEEP      = True
RUN_EXPORT_PORTABLE   = True

DEBUG_T_IDX = 9
DEBUG_Z_IDX = 16

## 23. Initialise runtime and confirm paths

In [ ]:
configure_tensorflow_for_gpu(cfg)

print(f"Model path:      {cfg.model_path}")
print(f"Input image:     {cfg.input_image_path}")
print(f"Output dir:      {cfg.output_dir}")

assert cfg.model_path.exists(),       f"Missing model: {cfg.model_path}"
assert cfg.input_image_path.exists(), f"Missing image: {cfg.input_image_path}"

save_config(cfg, cfg.output_dir / "debug_pipeline_config.json")
print("Configuration saved.")


## 24. Load image

In [ ]:
t0 = time.perf_counter()
img_5d = load_image_5d(cfg.input_image_path)
print(f"Shape: {img_5d.shape}  |  dtype: {img_5d.dtype}  |  load time: {time.perf_counter() - t0:.2f}s")


## 25. Optional focus-score QC

In [ ]:
y_starts = _generate_patch_starts(3889, cfg.patch_size, cfg.patch_stride)
x_starts = _generate_patch_starts(5732, cfg.patch_size, cfg.patch_stride)
print(f"Y starts: {len(y_starts)} patches, first={y_starts[0]}, last={y_starts[-1]}")
print(f"X starts: {len(x_starts)} patches, first={x_starts[0]}, last={x_starts[-1]}")
print(f"Total patches: {len(y_starts) * len(x_starts)}")

## 26. Single-plane segmentation debug

In [ ]:
# Single-plane segmentation debug (§26). Opt-in via RUN_SINGLE_PLANE_DEBUG
# (§22). Not part of the pipeline path — a full run (§27) loads its own model.
model = None

if RUN_SINGLE_PLANE_DEBUG:
    model = load_unet_model(cfg.model_path)
    print("Model loaded for single-plane debug.")

    debug_input_shape = get_full_context_yxc(
        img_5d, DEBUG_T_IDX, DEBUG_Z_IDX, config=cfg).shape
    print(f"Debug 2.5D input shape: {debug_input_shape}")

    _debug_batch_size = cfg.batch_size
    cfg.batch_size = 1  # single-plane debug: limit VRAM
    debug_prob_map, debug_class_map, debug_nucleus_mask, debug_instance_labels = (
        segment_single_plane_with_overlap(
            img_5d, DEBUG_T_IDX, DEBUG_Z_IDX, model, cfg,
            patch_size=cfg.patch_size, stride=cfg.patch_stride,
        )
    )
    print("Single-plane debug segmentation complete.")
    print(f"  Probability map shape: {debug_prob_map.shape}")
    print(f"  Unique class labels:   {np.unique(debug_class_map)}")
    print(f"  Nucleus pixels:        {int(debug_nucleus_mask.sum())}")
    for c in range(4):
        print(f"  Class {c} — min {debug_prob_map[...,c].min():.4f}  "
              f"max {debug_prob_map[...,c].max():.4f}  "
              f"mean {debug_prob_map[...,c].mean():.4f}")
    cfg.batch_size = _debug_batch_size
    del debug_prob_map, debug_class_map, debug_nucleus_mask
    del debug_instance_labels, debug_input_shape
    gc.collect()
    print("Debug arrays deleted from kernel memory.")
else:
    print("RUN_SINGLE_PLANE_DEBUG is False — skipping §26 single-plane debug.")

## 27. Run or load full segmentation

Restart the kernel after updating the notebook, then execute from the configuration downward.
The new revision and model fingerprint isolate this run from earlier segmentation outputs.


In [ ]:
# State is decided from the on-disk checkpoint BEFORE any model load or focus
# warm-up. reset_segmentation(cfg, scope=...) in the next cell forces a rebuild.
_T, _Z = img_5d.shape[0], img_5d.shape[1]
_Y, _X = img_5d.shape[-2], img_5d.shape[-1]

seg_state = assess_segmentation_state(cfg, (_T, _Z, _Y, _X),
                                      save_probability_tiff=SAVE_PROBABILITY_TIFF)
print(f"[segmentation state: {seg_state.mode}]")
print(seg_state.reason)

if cfg.probability_input_path is not None:
    # Layer 3: re-extract masks from a saved probability hyperstack (no model,
    # no inference). Point cfg.probability_hyperstack_input at any saved map.
    print(f"\nRe-extracting from probability hyperstack: {cfg.probability_input_path}")
    print(f"  thresholds  droplet={cfg.droplet_threshold}  "
          f"npc={cfg.npc_threshold}  nucleus={cfg.nucleus_threshold}")
    t0 = time.perf_counter()
    seg_index_df = run_reextraction_from_probability(config=cfg)
    print(f"Re-extraction finished in {time.perf_counter() - t0:.2f}s")

elif seg_state.mode == "complete":
    seg_index_df = pd.read_pickle(cfg.segmentation_index_path)
    print("Segmentation already complete — skipped model load and focus scoring.")

elif seg_state.mode == "inconsistent":
    raise RuntimeError(seg_state.reason)

elif seg_state.mode in ("fresh", "resume"):
    if seg_state.todo_t and ('model' not in globals() or model is None):
        model = load_unet_model(cfg.model_path)
    t0 = time.perf_counter()
    seg_index_df = run_segmentation_for_all_planes(
        img_5d=img_5d,
        model=(model if seg_state.todo_t else None),
        config=cfg, state=seg_state,
        save_probability_tiff=SAVE_PROBABILITY_TIFF)
    print(f"Segmentation finished in {time.perf_counter() - t0:.2f}s")

else:
    raise RuntimeError(f"unhandled seg_state.mode={seg_state.mode!r}")

print(seg_index_df.shape)
display(seg_index_df.head())

In [ ]:
# ============================================================
# Debug: Visualize single plane — raw channels + model predictions
# Opt-in: needs RUN_SINGLE_PLANE_DEBUG=True (§22) so `model` is loaded.
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

if not RUN_SINGLE_PLANE_DEBUG or ('model' not in globals()) or model is None:
    print("RUN_SINGLE_PLANE_DEBUG is False (or no model loaded) — "
          "skipping single-plane visualization.")
else:
    mem_raw  = img_5d[DEBUG_T_IDX, DEBUG_Z_IDX, cfg.membrane_channel_index].astype(np.float32)
    nls_raw  = img_5d[DEBUG_T_IDX, DEBUG_Z_IDX, cfg.nuclear_channel_index].astype(np.float32)
    npc_raw  = img_5d[DEBUG_T_IDX, DEBUG_Z_IDX, cfg.npc_channel_index].astype(np.float32)

    def norm(arr):
        lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
        return np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)

    prob_map, label_map, _, _ = segment_single_plane_with_overlap(
        img_5d, DEBUG_T_IDX, DEBUG_Z_IDX, model, cfg)

    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    fig.suptitle(f"t={DEBUG_T_IDX}  z={DEBUG_Z_IDX}  |  thr nuc={cfg.nucleus_threshold} npc={cfg.npc_threshold} drop={cfg.droplet_threshold}", fontsize=12)

    axes[0, 0].imshow(norm(mem_raw),  cmap="gray");    axes[0, 0].set_title("Ch0: Membrane (raw)")
    axes[0, 1].imshow(norm(nls_raw),  cmap="gray");    axes[0, 1].set_title("Ch1: NLS / Nucleus (raw)")
    axes[0, 2].imshow(norm(npc_raw),  cmap="gray");    axes[0, 2].set_title("Ch2: NPC (raw)")
    axes[0, 3].imshow(label_map, cmap="viridis", vmin=0, vmax=3)
    axes[0, 3].set_title("Predicted class label\n0=bg 1=droplet 2=NPC 3=nucleus")

    class_info = [
        (0, "Background prob",  "gray"),
        (1, "Droplet prob",     "Blues"),
        (2, "NPC prob",         "Purples"),
        (3, "Nucleus prob",     "Greens"),
    ]
    for c, title, cmap in class_info:
        im = axes[1, c].imshow(prob_map[..., c], cmap=cmap, vmin=0, vmax=1)
        axes[1, c].set_title(f"{title}\nmax={prob_map[...,c].max():.3f}  mean={prob_map[...,c].mean():.4f}")
        plt.colorbar(im, ax=axes[1, c], fraction=0.03)

    for ax in axes.ravel():
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"\nClass pixel counts in predicted label map:")
    for c, title, _ in class_info:
        px = int((label_map == c).sum())
        frac = px / label_map.size
        print(f"  Class {c} ({title.split()[0]:10s}): {px:>10,} px  ({frac:.3%})")

In [ ]:
# -- Reset segmentation state --------------------------------------------
# Uncomment ONE call, run it, then re-run §27.
#
#   scope="all"     wipe the checkpoint, _resume/, the 5 hyperstack TIFFs, the
#                   per-plane .npy masks, the seg index / ROI pickles, AND the
#                   stale downstream object pickles (plane / grouped / best_z /
#                   repaired, repaired_masks/). Nothing left half-deleted.
#   scope="resume"  wipe only the checkpoint + _resume/ (keep the TIFFs). Forces
#                   a full recompute; use after editing a segmentation function.
#
# reset_segmentation(cfg, scope="all")
# reset_segmentation(cfg, scope="resume")

print("Reset cell — edit to uncomment a reset_segmentation(...) call.")

## 28. Run or load object extraction

In [ ]:
# Plane indices are written transactionally by segmentation; no legacy reconstruction.


In [ ]:
require_segmentation(cfg, needs=("npy_planes",))

objects_path = cfg.obj_dir / "plane_objects.pkl"

if RUN_OBJECT_EXTRACTION:
    t0 = time.perf_counter()
    objects_df = extract_objects_from_saved_masks(seg_index_df, cfg)
    print(f"Object extraction finished in {time.perf_counter() - t0:.2f}s")
else:
    objects_df = pd.read_pickle(objects_path)
    print("Loaded saved object table.")

print(objects_df.shape)
display(objects_df.head())

## 29. Run or load Z grouping

In [ ]:
grouped_z_path = cfg.obj_dir / "grouped_z_objects.pkl"

if RUN_GROUP_ACROSS_Z:
    t0 = time.perf_counter()
    grouped_z_df = group_nuclei_across_z(objects_df, cfg)
    print(f"Z grouping finished in {time.perf_counter() - t0:.2f}s")
else:
    grouped_z_df = pd.read_pickle(grouped_z_path)
    print("Loaded grouped-Z table.")

print(grouped_z_df.shape)
display(grouped_z_df.head())


## 30. Run or load best-Z selection

In [ ]:
best_z_path = cfg.obj_dir / "best_z_nuclei.pkl"

if RUN_SELECT_BEST_Z:
    t0 = time.perf_counter()
    best_z_df = select_best_z_per_nucleus(grouped_z_df, cfg)
    print(f"Best-Z selection finished in {time.perf_counter() - t0:.2f}s")
else:
    best_z_df = pd.read_pickle(best_z_path)
    print("Loaded best-Z table.")

print(best_z_df.shape)
display(best_z_df.head())


In [ ]:
# QC: Z-grouping / best-Z-selection diagnostics (moved here from earlier in the
# notebook — they depend on grouped_z_df and best_z_df, which do not exist yet
# the first time Sections 27-30 run top to bottom).
edge = grouped_z_df.groupby(["t", "nucleus_3d_id"])["z"].agg(["min", "max"])
m = best_z_df.set_index(["t", "nucleus_3d_id"]).join(edge)
m["at_edge"] = (m.z == m["min"]) | (m.z == m["max"])
print("fraction selected at window edge, by t:")
print(m.groupby(level="t")["at_edge"].mean())
print("\nplanes per nucleus (how many Z actually grouped):")
print(grouped_z_df.groupby(["t", "nucleus_3d_id"]).size().value_counts())
print("\nselected z by t:")
print(best_z_df.groupby("t")["z"].value_counts().unstack(fill_value=0))

three = grouped_z_df.groupby(["t", "nucleus_3d_id"]).filter(lambda x: len(x) == 3)
prof = (three.sort_values("z")
             .groupby(["t", "nucleus_3d_id"])["area_px"]
             .agg(list).apply(lambda a: (max(a) - min(a)) / max(a)))
print("\nmedian fractional area spread across the 3 planes, by t:")
print(prof.groupby(level="t").median().round(3))

# is the max at the middle plane or an edge?
pos = (three.sort_values("z").groupby(["t", "nucleus_3d_id"])["area_px"]
            .agg(lambda a: int(np.argmax(list(a)))))
print("\nargmax position (0=low edge, 1=center, 2=high edge):")
print(pos.groupby(level="t").value_counts().unstack(fill_value=0))


## 30b. Optional legacy fragmentation repair

Disabled by default. Enabling it changes the run signature and measurements.


In [ ]:
from scipy import ndimage as ndi
from skimage import filters, measure, morphology, segmentation


def _resolve_instance(inst_plane, cx, cy, bbox=None):
    """
    objects_df.label is per-plane regionprops numbering; the instance
    hyperstack uses sparse global ids. They are different namespaces, so the
    label column cannot index the instance map — resolve by centroid instead.
    """
    H, W = inst_plane.shape
    yi, xi = int(round(cy)), int(round(cx))
    if 0 <= yi < H and 0 <= xi < W:
        v = int(inst_plane[yi, xi])
        if v > 0:
            return v
    if bbox is not None:
        r0, c0, r1, c1 = [int(b) for b in bbox]
        sub = inst_plane[max(r0, 0):min(r1, H), max(c0, 0):min(c1, W)]
        if sub.size:
            cts = np.bincount(sub.ravel()); cts[0] = 0
            if cts.max() > 0:
                return int(cts.argmax())
    return 0


def flatten_interior(img_norm, se_radius_px):
    """
    Opening- then closing-by-reconstruction. Removes bright structures smaller
    than the SE while leaving the position and shape of everything larger
    untouched — the property a Gaussian does not have.

    Without this the watershed does not move: the core's own edge is a stronger
    gradient than the envelope, so the basin stops at the core.
    """
    se = morphology.disk(max(int(se_radius_px), 1))
    opened = morphology.reconstruction(morphology.erosion(img_norm, se),
                                       img_norm, method="dilation")
    closed = morphology.reconstruction(morphology.dilation(opened, se),
                                       opened, method="erosion")
    return closed.astype(np.float32)


def seeded_watershed(seed, nls, bound, config, flatten_um=8.0,
                     smooth_um=0.8, wall_um=1.5):
    """Grow `seed` to the envelope, confined to `bound`."""
    if not seed.any() or not bound.any():
        return np.zeros(nls.shape, bool)
    lo, hi = np.percentile(nls[bound], [1, 99.5])
    if hi <= lo:
        return np.zeros(nls.shape, bool)
    norm = np.clip((nls - lo) / (hi - lo), 0, 1)
    sm = filters.gaussian(norm, sigma=max(smooth_um / config.pixel_size_um, 0.5),
                          preserve_range=True).astype(np.float32)
    sm = flatten_interior(sm, int(round(flatten_um / config.pixel_size_um)))

    er = max(int(round(wall_um / config.pixel_size_um)), 1)
    wall = bound & ~morphology.binary_erosion(bound, morphology.disk(er))
    if not wall.any():
        wall = bound & ~morphology.binary_erosion(bound, morphology.disk(1))
    markers = np.zeros(seed.shape, np.int32)
    markers[wall] = 1
    markers[seed & bound] = 2
    if not (markers == 2).any():
        return np.zeros(nls.shape, bool)
    ws = segmentation.watershed(filters.sobel(sm), markers, mask=bound)
    return ndi.binary_fill_holes(ws == 2)


def npc_ring_radius(npc_crop, bound, ly, lx, config, n_rays=72, k_std=1.5):
    """Median radius of peak NPC intensity over rays from (ly, lx)."""
    r_max = int(np.sqrt(max(bound.sum(), 1) / np.pi))
    rr = np.arange(2, max(r_max, 3))
    peaks = []
    for th in np.linspace(0, 2 * np.pi, n_rays, endpoint=False):
        ys = (ly + rr * np.sin(th)).astype(int)
        xs = (lx + rr * np.cos(th)).astype(int)
        ok = ((ys >= 0) & (ys < npc_crop.shape[0])
              & (xs >= 0) & (xs < npc_crop.shape[1]))
        if ok.sum() < 5:
            continue
        prof = npc_crop[ys[ok], xs[ok]]
        if prof.max() < prof.mean() + k_std * prof.std():
            continue
        peaks.append(rr[ok][int(np.argmax(prof))])
    if len(peaks) < n_rays // 8:
        return np.nan, len(peaks)
    return float(np.median(peaks)) * config.pixel_size_um, len(peaks)


def signed_contrast_ok(mask, nls, droplet, config):
    """Interior must beat the droplet median — kills dark-region grabs."""
    if not mask.any():
        return False, "empty"
    if droplet.any() and mask.sum() / droplet.sum() > config.repair_max_frac_droplet:
        return False, f"flooded ({mask.sum()/droplet.sum():.2f})"
    inside = float(nls[mask].mean())
    if droplet.any() and inside < float(np.median(nls[droplet])):
        return False, "interior darker than droplet"
    shell = (morphology.binary_dilation(
        mask, morphology.disk(max(int(1.3 / config.pixel_size_um), 1)))
        & ~mask & droplet)
    if shell.any():
        out = float(nls[shell].mean())
        if out > 0 and inside / out < config.repair_min_signed_margin:
            return False, f"low signed contrast ({inside/out:.2f})"
    return True, ""


def repair_plane(t, z, cx, cy, image_5d, nucleus_instance_4d,
                 droplet_instance_4d, config, seed_override=None):
    """Repair one nucleus on one plane. Returns (mask, metrics, offsets)."""
    half = int(round((config.repair_win_um / config.pixel_size_um) / 2))
    H, W = image_5d.shape[-2], image_5d.shape[-1]
    r0, c0 = max(int(cy) - half, 0), max(int(cx) - half, 0)
    r1, c1 = min(int(cy) + half, H), min(int(cx) + half, W)
    sl = (slice(r0, r1), slice(c0, c1))

    nls = np.asarray(image_5d[t, z, config.nuclear_channel_index], np.float32)[sl]
    npc = np.asarray(image_5d[t, z, config.npc_channel_index], np.float32)[sl]
    dpl = np.asarray(droplet_instance_4d[t, z])
    dl = _resolve_instance(dpl, cx, cy)
    droplet = (dpl == dl)[sl] if dl else np.ones(nls.shape, bool)

    if seed_override is not None:
        seed = seed_override & droplet
    else:
        inst = np.asarray(nucleus_instance_4d[t, z])
        nl = _resolve_instance(inst, cx, cy)
        seed = (inst == nl)[sl] if nl else np.zeros(nls.shape, bool)
    if not seed.any():
        return np.zeros(nls.shape, bool), {"reject": "no seed"}, (r0, c0)

    ly, lx = cy - r0, cx - c0
    r_um, n_rays = npc_ring_radius(npc, droplet, ly, lx, config)

    bound = droplet
    if np.isfinite(r_um) and n_rays >= config.repair_npc_min_rays:
        sy, sx = ndi.center_of_mass(seed)
        yy, xx = np.ogrid[:nls.shape[0], :nls.shape[1]]
        rp = (r_um * config.repair_npc_slack) / config.pixel_size_um
        cand = droplet & ((yy - sy) ** 2 + (xx - sx) ** 2 <= rp ** 2)
        if (seed & cand).any():
            bound = cand

    mask = seeded_watershed(seed, nls, bound, config,
                            flatten_um=config.repair_flatten_um)
    ok, why = signed_contrast_ok(mask, nls, droplet, config)
    px2 = config.pixel_size_um ** 2
    m = {"reject": "" if ok else why, "npc_r_um": r_um, "npc_n_rays": n_rays,
         "npc_area_um2": np.pi * r_um ** 2 if np.isfinite(r_um) else np.nan,
         "area_px": int(mask.sum()), "area_um2": float(mask.sum()) * px2}
    if ok and mask.any():
        rp_ = measure.regionprops(mask.astype(np.uint8))[0]
        m["solidity"] = float(rp_.solidity)
        m["centroid_y_px"] = r0 + rp_.centroid[0]
        m["centroid_x_px"] = c0 + rp_.centroid[1]
    else:
        mask = np.zeros(nls.shape, bool)
    return mask, m, (r0, c0)


def repair_fragmented_nuclei(best_z_df, grouped_z_df, image_5d,
                             nucleus_instance_4d, droplet_instance_4d,
                             config, verbose=True):
    """
    Detect fragmented nuclei, repair them, re-select z, return the corrected
    best-Z table.

    Overwrites `area_px`, `z`, and the centroids with the repaired values so
    every downstream stage consumes them unchanged. Originals are preserved as
    `*_original`, and provenance goes in `repair_status` / `repair_gain` /
    `ratio_vs_npc`.
    """
    px2 = config.pixel_size_um ** 2
    H, W = image_5d.shape[-2], image_5d.shape[-1]
    Z = image_5d.shape[1]
    em = max(int(round(config.repair_edge_margin_um / config.pixel_size_um)), 1)

    d = best_z_df.copy().reset_index(drop=True)
    d["area_px_original"] = d.area_px
    d["z_original"] = d.z
    d["nucleus_area_um2"] = d.area_px * px2

    # ---- edge status: droplet OR nucleus touching the frame ----
    def touches(arr, lab):
        if not lab:
            return True
        ys, xs = np.where(arr == lab)
        return bool(ys.size == 0 or ys.min() <= em or xs.min() <= em
                    or ys.max() >= H - 1 - em or xs.max() >= W - 1 - em)

    edge, sol = [], []
    for r in d.itertuples():
        inst = np.asarray(nucleus_instance_4d[int(r.t), int(r.z)])
        dpl = np.asarray(droplet_instance_4d[int(r.t), int(r.z)])
        nl = _resolve_instance(inst, r.centroid_x_px, r.centroid_y_px)
        dl = _resolve_instance(dpl, r.centroid_x_px, r.centroid_y_px)
        edge.append(touches(inst, nl) or touches(dpl, dl))
        if nl:
            rp = measure.regionprops((inst == nl).astype(np.uint8))[0]
            sol.append(float(rp.solidity))
        else:
            sol.append(np.nan)
    d["exclude_edge"] = edge
    d["solidity_original"] = sol

    pool = d[~d.exclude_edge]
    ref = pool.groupby("t").nucleus_area_um2.quantile(0.75).rename("_ref")
    d = d.join(ref, on="t")
    d["area_deficit"] = 1.0 - d.nucleus_area_um2 / d._ref
    d["is_fragmented"] = ((d.solidity_original < config.repair_min_solidity)
                          | (d.area_deficit > config.repair_area_deficit))
    d = d.drop(columns=["_ref"])

    for c in ("repair_gain", "ratio_vs_npc", "npc_r_um", "npc_area_um2",
              "npc_n_rays", "z_shift", "n_planes_repaired"):
        d[c] = np.nan
    d["repair_status"] = "not fragmented"

    if verbose:
        print(f"{len(d)} nuclei | {int(d.exclude_edge.sum())} at frame edge "
              f"| {int(d.is_fragmented.sum())} fragmented "
              f"({100*d.is_fragmented.mean():.1f}%)")

    todo = d.index[d.is_fragmented & ~d.exclude_edge]
    n_ok = n_fail = 0
    for i in todo:
        r = d.loc[i]
        t, z0 = int(r.t), int(r.z)
        cx, cy = float(r.centroid_x_px), float(r.centroid_y_px)
        mask0, m0, off0 = repair_plane(t, z0, cx, cy, image_5d,
                                       nucleus_instance_4d, droplet_instance_4d,
                                       config)
        if not mask0.any():
            d.at[i, "repair_status"] = f"rejected: {m0['reject']}"
            n_fail += 1
            continue

        best, best_z, best_mask, best_off, seen = m0, z0, mask0, off0, 1
        for direction in (-1, +1):
            seed = mask0
            for step in range(1, config.repair_z_pad + 1):
                z = z0 + direction * step
                if z not in eligible_z_planes(Z, config):
                    break
                sd = morphology.binary_erosion(seed, morphology.disk(3))
                if not sd.any():
                    break
                mk, mm, off = repair_plane(t, z, cx, cy, image_5d,
                                           nucleus_instance_4d,
                                           droplet_instance_4d, config,
                                           seed_override=sd)
                if not mk.any() or mm["area_px"] < 0.15 * best["area_px"]:
                    break
                seen += 1
                if mm["area_px"] > best["area_px"]:
                    best, best_z, best_mask, best_off = mm, z, mk, off
                seed = mk

        d.at[i, "area_px"] = best["area_px"]
        d.at[i, "nucleus_area_um2"] = best["area_um2"]
        d.at[i, "z"] = best_z
        d.at[i, "z_shift"] = best_z - z0
        d.at[i, "n_planes_repaired"] = seen
        if "centroid_x_px" in best:
            d.at[i, "centroid_x_px"] = best["centroid_x_px"]
            d.at[i, "centroid_y_px"] = best["centroid_y_px"]
        d.at[i, "repair_gain"] = best["area_px"] / max(r.area_px, 1)
        d.at[i, "npc_r_um"] = best["npc_r_um"]
        d.at[i, "npc_area_um2"] = best["npc_area_um2"]
        d.at[i, "npc_n_rays"] = best["npc_n_rays"]
        if np.isfinite(best["npc_area_um2"]) and best["npc_area_um2"] > 0:
            d.at[i, "ratio_vs_npc"] = best["area_um2"] / best["npc_area_um2"]
        d.at[i, "repair_status"] = "repaired"

        # ── persist the winning mask (full-frame) so halo/radial-sweep find it ──
        # recover_nucleus_mask_from_plane would otherwise reload the pre-repair
        # per-plane .npy and silently undo the repair (see Section 18/19).
        r0, c0 = best_off
        canvas = np.zeros((H, W), dtype=bool)
        bh, bw = best_mask.shape
        canvas[r0:r0 + bh, c0:c0 + bw] = best_mask
        out_path = _repaired_mask_path(config, t, r.nucleus_3d_id)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, canvas)

        n_ok += 1

    if verbose:
        rep = d[d.repair_status == "repaired"]
        print(f"  repaired {n_ok} | rejected {n_fail}")
        if n_fail:
            print(d.loc[todo, "repair_status"][
                d.loc[todo, "repair_status"].str.startswith("rejected")]
                .value_counts().head(5).to_string())
        if len(rep):
            print(f"  median gain {rep.repair_gain.median():.2f}x | "
                  f"z moved {int((rep.z_shift != 0).sum())}/{len(rep)} | "
                  f"median planes {rep.n_planes_repaired.median():.0f}")
            print(f"  ratio vs NPC ring {rep.ratio_vs_npc.median():.3f} "
                  f"(target 0.88-0.95)")
            if rep.ratio_vs_npc.median() > 1.0:
                print("  WARNING: above 1.0 — the mask is crossing the "
                      "envelope. Lower cfg.repair_npc_slack.")
            elif rep.ratio_vs_npc.median() < 0.85:
                print("  NOTE: below 0.85 — still under-recovering. Consider "
                      "raising cfg.repair_flatten_um.")
    return d

### 30c. Run the repair

Sits between best-Z selection (§30) and close-nuclei exclusion (§32), and
overwrites `best_z_df`, so §31 onward run unchanged.

In [ ]:
repaired_path = cfg.obj_dir / "best_z_nuclei_repaired.pkl"
RUN_FRAGMENTATION_REPAIR = cfg.repair_enabled

if cfg.repair_enabled and RUN_FRAGMENTATION_REPAIR:
    require_segmentation(cfg, needs=("nucleus_instance_hyperstack",
                                    "droplet_instance_hyperstack"))
    t0 = time.perf_counter()
    nucleus_instance_4d = tiff.memmap(cfg.nucleus_instance_hyperstack_path, mode="r")
    droplet_instance_4d = tiff.memmap(cfg.droplet_instance_hyperstack_path, mode="r")
    best_z_df = repair_fragmented_nuclei(best_z_df, grouped_z_df, img_5d,
                                         nucleus_instance_4d,
                                         droplet_instance_4d, cfg)
    best_z_df.to_pickle(repaired_path)
    print(f"Fragmentation repair finished in {time.perf_counter() - t0:.2f}s")
elif cfg.repair_enabled:
    best_z_df = pd.read_pickle(repaired_path)
    print("Loaded repaired best-Z table.")
else:
    print("cfg.repair_enabled is False — using unrepaired best-Z table.")

print(best_z_df.shape)
display(best_z_df.head())

### 30d. Repair QC

Read `ratio_vs_npc` first — it is the only reference independent of the NLS
channel. The scatter should straddle 0.9, not sit at 1.2 (crossing the
envelope) or 0.6 (still fragmented).

In [ ]:
def plot_repair_qc(d, config, time_col=None, figsize=(16, 4.6)):
    dd = d[~d.exclude_edge].copy()
    if time_col is None:
        time_col = "true_time_min" if "true_time_min" in dd.columns else "t"
    frag = dd.is_fragmented
    if "nucleus_area_um2_original" not in dd:
        dd["nucleus_area_um2_original"] = dd.area_px_original * config.pixel_size_um ** 2

    fig, ax = plt.subplots(1, 3, figsize=figsize)
    for a, col, ttl in ((ax[0], "nucleus_area_um2_original", "BEFORE"),
                        (ax[1], "nucleus_area_um2", "AFTER — repaired")):
        a.scatter(dd.loc[~frag, time_col], dd.loc[~frag, col], s=11, alpha=0.45,
                  c="tab:blue", label="never fragmented")
        a.scatter(dd.loc[frag, time_col], dd.loc[frag, col], s=15, alpha=0.75,
                  c="tab:red", label="fragmented")
        a.set_title(ttl); a.set_xlabel(time_col); a.grid(alpha=0.25)
        a.legend(fontsize=8)
    ax[0].set_ylabel("nuclear area (µm²)")
    ax[1].set_ylim(ax[0].get_ylim())

    rep = dd[dd.repair_status == "repaired"]
    a = ax[2]
    if len(rep):
        a.scatter(rep.npc_area_um2, rep.nucleus_area_um2, s=14, alpha=0.6)
        lim = [0, float(rep.npc_area_um2.quantile(0.99))]
        a.plot(lim, lim, "k--", lw=1, label="equal")
        a.plot(lim, [0.9 * x for x in lim], "g--", lw=1, label="0.90 (expected)")
        a.set_xlabel("NPC ring area (µm²)")
        a.set_ylabel("repaired NLS area (µm²)")
        a.set_title(f"vs the independent reference "
                    f"(median {rep.ratio_vs_npc.median():.2f})")
        a.legend(fontsize=8); a.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

    print(f"{'':<28}{'before':>10}{'after':>10}")
    print(f"{'median area, fragmented':<28}"
          f"{dd[frag].nucleus_area_um2_original.median():>10.0f}"
          f"{dd[frag].nucleus_area_um2.median():>10.0f}")
    print(f"{'median area, all':<28}"
          f"{dd.nucleus_area_um2_original.median():>10.0f}"
          f"{dd.nucleus_area_um2.median():>10.0f}")
    print(f"{'never-fragmented median':<28}"
          f"{dd[~frag].nucleus_area_um2_original.median():>10.0f}")
    return dd


if cfg.repair_enabled:
    _ = plot_repair_qc(best_z_df, cfg)

## 31. Optional area-based cleanup

In [ ]:
def filter_nuclei_by_area(
    df: pd.DataFrame,
    config: PipelineConfig,
    min_area_um2: float = 100.0,
    max_area_um2: float = 600.0,
) -> pd.DataFrame:
    """Remove objects that are too small or too large to be real nuclei."""
    df = df.copy()
    if "nucleus_area_um2" not in df.columns:
        df["nucleus_area_um2"] = df["area_px"] * config.pixel_size_um ** 2
    filtered = df[(df["nucleus_area_um2"] >= min_area_um2) & (df["nucleus_area_um2"] <= max_area_um2)]
    print(f"Filtered: {len(df)} → {len(filtered)} nuclei")
    return filtered


# Optional debug usage:
# v17: was (100, 300) while the function default is (100, 600) --
# the debug call silently used a different ceiling than the docstring.
best_z_area_filtered_df = filter_nuclei_by_area(best_z_df, cfg, 100, 600)
display(best_z_area_filtered_df.head())


## 32. Run or load close-nuclei exclusion

In [ ]:
filtered_path = cfg.obj_dir / "best_z_nuclei_with_exclusion.pkl"

if RUN_FLAG_CLOSE_NUCLEI:
    t0 = time.perf_counter()
    filtered_df = flag_close_nuclei(best_z_df, cfg)
    print(f"Close-nuclei flagging finished in {time.perf_counter() - t0:.2f}s")
else:
    filtered_df = pd.read_pickle(filtered_path)
    print("Loaded exclusion table.")

print(filtered_df.shape)
display(filtered_df.head())


## 33. Run or load timing metadata

In [ ]:
timed_path = cfg.track_dir / "best_z_nuclei_timed.pkl"

if RUN_ADD_TIMING:
    t0 = time.perf_counter()
    timed_df = add_tile_timing_metadata(filtered_df, img_5d, cfg)
    print(f"Timing metadata finished in {time.perf_counter() - t0:.2f}s")
else:
    timed_df = pd.read_pickle(timed_path)
    print("Loaded timed nuclei table.")

print(timed_df.shape)
display(timed_df.head())


## 34. Run or load tracking

In [ ]:
tracked_path = cfg.track_dir / "tracked_nuclei.pkl"

if RUN_TRACKING:
    t0 = time.perf_counter()
    tracked_df = assign_track_ids_hybrid_dbscan(timed_df, cfg, valid_only=False)
    print(f"Hybrid DBSCAN tracking finished in {time.perf_counter() - t0:.2f}s")
else:
    tracked_df = pd.read_pickle(tracked_path)
    print("Loaded tracked nuclei table.")

print(tracked_df.shape)
display(tracked_df.head())
display(summarize_tracking_debug(tracked_df))


## 35. Run or load halo analysis

In [ ]:
halo_path = cfg.analysis_dir / "halo_analysis.pkl"

if RUN_HALO_ANALYSIS:
    t0 = time.perf_counter()
    halo_df = run_halo_analysis_for_tracked_nuclei(tracked_df, img_5d, cfg)
    print(f"Halo analysis finished in {time.perf_counter() - t0:.2f}s")
else:
    halo_df = pd.read_pickle(halo_path)
    print("Loaded halo analysis table.")

print(halo_df.shape)
display(halo_df.head())


## 35b. Run or load radial sweep (droplet-bounded)

Runs on the same `tracked_df` as the halo analysis (best-Z, one row per nucleus per timepoint). Requires the droplet instance hyperstack from segmentation. Output is long-form (`radial_df`), pickled to `analysis_dir/radial_sweep.pkl`.

In [ ]:
radial_sweep_path = cfg.analysis_dir / "radial_sweep.pkl"

if RUN_RADIAL_SWEEP:
    t0 = time.perf_counter()
    radial_df = run_radial_sweep_for_tracked_nuclei(tracked_df, img_5d, cfg)
    print(f"Radial sweep finished in {time.perf_counter() - t0:.2f}s")
else:
    radial_df = pd.read_pickle(radial_sweep_path)
    print("Loaded radial sweep table.")

print(radial_df.shape)
display(radial_df.head())


In [ ]:
radial_sweep_path = cfg.analysis_dir / "radial_sweep.pkl"

if RUN_RADIAL_SWEEP:
    t0 = time.perf_counter()
    radial_df = run_radial_sweep_for_tracked_nuclei(tracked_df, img_5d, cfg)
    print(f"Radial sweep finished in {time.perf_counter() - t0:.2f}s")
else:
    radial_df = pd.read_pickle(radial_sweep_path)
    print("Loaded radial sweep table.")

print(radial_df.shape)
display(radial_df.head())


In [ ]:
radial_df = pd.read_pickle(cfg.analysis_dir / "radial_sweep.pkl")
print(radial_df.shape)
radial_df.head()

In [ ]:
import numpy as np, pandas as pd

radial_df = pd.read_pickle(cfg.analysis_dir / "radial_sweep.pkl")
print(radial_df.shape)
print(radial_df.columns.tolist())
print(radial_df["t"].value_counts().sort_index())

px = cfg.pixel_size_um
out = cfg.exports_dir   # created by the resolver patch

# (1) Mean radial intensity profile: median intensity vs absolute radius, per t
prof = (radial_df
        .assign(r_um=(radial_df.distance_px * px).round(1))
        .groupby(["t", "r_um"])["intensity"]
        .agg(median="median", mean="mean", n="size")
        .reset_index())
prof.to_csv(out / "radial_profile_binned.csv", index=False)

# (2) Per-nucleus envelope peak: radius of max membrane intensity per ray,
#     then median across rays for each nucleus. This is the mask-independent
#     nuclear radius estimate.
peak = (radial_df.loc[radial_df.groupby(["t","track_id","theta_index"])["intensity"].idxmax()]
        .groupby(["t","track_id"])
        .agg(peak_r_um=("distance_px", lambda s: float(np.median(s)) * px),
             peak_intensity=("intensity", "median"),
             n_rays=("theta_index", "nunique"))
        .reset_index())
peak["implied_area_um2"] = np.pi * peak.peak_r_um**2
peak.to_csv(out / "radial_envelope_peaks.csv", index=False)

print(peak.groupby("t")[["peak_r_um","implied_area_um2"]].median().round(1))
print([f"{p.name}: {p.stat().st_size/1e6:.1f}MB" for p in out.glob("radial_*.csv")])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

for t, d in prof.groupby("t"):
    ax[0].plot(d.r_um, d["median"], label=f"t={t}", lw=1.4)
ax[0].axvline(6.2, ls="--", c="k", lw=1, label="current mask R")
ax[0].set_xlabel("distance from centroid (µm)")
ax[0].set_ylabel("median membrane intensity")
ax[0].set_title("Radial profile by timepoint")
ax[0].legend(fontsize=7, ncol=2)

late = peak[peak.t >= 7]
ax[1].hist(late.peak_r_um, bins=40)
ax[1].axvline(6.2, ls="--", c="k", lw=1, label="current mask R")
ax[1].axvline(11.3, ls="--", c="r", lw=1, label="expected R (400 µm²)")
ax[1].set_xlabel("envelope peak radius (µm)")
ax[1].set_ylabel("nuclei")
ax[1].set_title("Peak radius, t≥7")
ax[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
import numpy as np, pandas as pd, tifffile as tiff
from skimage import measure

drop = tiff.memmap(cfg.droplet_instance_hyperstack_path, mode="r")
plane = np.asarray(drop[9, 15])          # late t, in-focus z
d = pd.DataFrame(measure.regionprops_table(plane, properties=("area","equivalent_diameter")))
d = d[d.area > 1000]
med_px = d.equivalent_diameter.median()
print("droplet diameter (px):", round(med_px, 1))
print("droplet diameter (µm) @0.108:", round(med_px * cfg.pixel_size_um, 1))
print("implied px size if droplets are 65 µm:", round(65 / med_px, 4))


In [ ]:
import numpy as np, pandas as pd, tifffile as tiff
from skimage import measure

drop = tiff.memmap(cfg.droplet_instance_hyperstack_path, mode="r")
b = pd.read_pickle(cfg.obj_dir / "best_z_nuclei.pkl")

rows = []
for (t, z), grp in b[b.t >= 7].groupby(["t", "z"]):
    plane = np.asarray(drop[t, z])
    props = {r.label: r.area for r in measure.regionprops(plane)}
    for r in grp.itertuples():
        did = plane[int(r.centroid_y_px), int(r.centroid_x_px)]
        if did > 0 and props.get(did, 0) > 0:
            rows.append({"t": t, "nuc_px": r.area_px, "drop_px": props[did],
                         "ratio": r.area_px / props[did]})

d = pd.DataFrame(rows)
print("n nuclei matched:", len(d))
print("median droplet area (px):", round(float(d.drop_px.median())))
print("median droplet diameter (px):", round(float(2*np.sqrt(d.drop_px.median()/np.pi)), 1))
print("median nucleus/droplet area ratio:", round(float(d.ratio.median()), 4))

D_UM = 65.0   # <-- set to your actual RAN008 droplet diameter
diam_px = float(2*np.sqrt(d.drop_px.median()/np.pi))
print(f"\nif droplets are {D_UM} µm diameter -> nucleus area:",
      round(float(d.ratio.median()) * np.pi * (D_UM/2)**2, 1), "µm²")
print("implied pixel size:", round(D_UM / diam_px, 4), "µm/px")

In [ ]:
class_mask_5d       = tiff.imread(cfg.segmentation_class_hyperstack_path)
label_mask_4d       = tiff.imread(cfg.segmentation_label_hyperstack_path)
nucleus_instance_4d = tiff.imread(cfg.nucleus_instance_hyperstack_path)

print(f"class_mask_5d shape:       {class_mask_5d.shape}")
print(f"label_mask_4d shape:       {label_mask_4d.shape}")
print(f"nucleus_instance_4d shape: {nucleus_instance_4d.shape}")

nucleus_mask_4d = class_mask_5d[:, :, cfg.nucleus_class_index] > 0
print(f"nucleus_mask_4d shape:     {nucleus_mask_4d.shape}")


In [ ]:
plot_nucleus_mask_overlay_qc(img_5d, nucleus_mask_4d, cfg, z_mode="max_mask")
plot_nucleus_mask_qc_summary(nucleus_mask_4d, z_mode="max_mask")
plot_nucleus_count_over_time(nucleus_mask_4d)


## 36. Figures — analysis plot set

Everything below runs off the tables the pipeline already writes; nothing here
re-reads the image stack. Each figure is saved to `<run>/exports/figures/` as
PNG + PDF, so a run is reproducible from its own directory.

| figure | function | source table |
|---|---|---|
| N/C ratio, area vs time | `plot_nc_vs_time`, `plot_area_vs_time` | `halo_df` |
| rose — envelope radius by angle | `plot_rose_individual`, `plot_rose_pooled` | `radial_df` |
| membrane intensity by angle × distance | `plot_membrane_angle_distance` | `radial_df` |
| membrane asymmetry vs time | `plot_membrane_asymmetry_vs_time` | `radial_df` + `tracked_df` |
| cross-sectional area, 6-min boxes | `plot_area_boxplot_binned` | `halo_df` / `timed_df` |

**Three things changed relative to the scratch versions further down the
notebook**, each flagged in the section it affects:

1. `distance_norm` in `radial_sweep.pkl` is *not* the nucleus-surface-to-wall
   coordinate it was being plotted as (§36a).
2. An unrestricted per-ray `argmax` for the envelope peak finds the droplet
   interface, not the nuclear envelope (§36b).
3. Polar axes are now drawn in the image frame, so a rose can be laid against
   a raw crop without mirroring (§36b).

The scratch cells in §37+ are superseded by this section and can be deleted
once you're happy with the output.

In [ ]:
# ── Section 36 parameters ─────────────────────────────────────────────────
# Peak search is capped at this fraction of the nucleus->wall gap. See 36b for
# why: the membrane channel also images the droplet interface, which outshines
# the NE. Raise toward 1.0 only after checking frac_at_search_edge.
ENVELOPE_SEARCH_RHO_MAX = 0.6

# Perinuclear shell used for the asymmetry metrics, in microns outward from the
# nuclear mask edge.
ASYMMETRY_BAND_UM = (0.0, 3.0)

# Width of the acquisition-time bins for the box plots, in minutes.
# 2x3 tiles at 1 min/tile => 6 min per frame, so 6.0 == one frame per box.
AREA_BIN_MIN = 6.0

In [ ]:
# ── Bind the tables this section plots, loading from the current run if the
# kernel doesn't already hold them. Lets Section 36 run against a finished
# run without re-executing the pipeline above.
for _name, _path in [
    ("halo_df",    cfg.analysis_dir / "halo_analysis.pkl"),
    ("timed_df",   cfg.track_dir    / "best_z_nuclei_timed.pkl"),
    ("tracked_df", cfg.track_dir    / "tracked_nuclei.pkl"),
    ("radial_df",  cfg.analysis_dir / "radial_sweep.pkl"),
]:
    if _name not in dir():
        globals()[_name] = pd.read_pickle(_path)
        print(f"Loaded {_name} from disk.")

print({k: globals()[k].shape for k in ("halo_df", "timed_df", "tracked_df", "radial_df")})

In [ ]:
from typing import Sequence  # not in the Section 1 import block

SAVE_FIGURES = True
FIG_FORMATS = ("png", "pdf")
FIG_DPI = 300


def figures_dir(config) -> Path:
    d = config.exports_dir / "figures"
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_figure(fig, name: str, config, formats=FIG_FORMATS, dpi: int = FIG_DPI) -> List[Path]:
    """Write a figure to <run>/exports/figures/<name>.<ext> for each format."""
    if not SAVE_FIGURES:
        return []
    out = []
    d = figures_dir(config)
    for ext in formats:
        p = d / f"{name}.{ext}"
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        out.append(p)
    return out


def resolve_area_um2(df: pd.DataFrame, config, out_col: str = "nucleus_area_um2") -> pd.DataFrame:
    """Guarantee an area column in um^2, recomputed from pixels when possible.

    Recomputing from *_area_px means the figure always reflects the live
    pixel_size_um rather than whatever calibration was in force when the
    table was pickled.
    """
    df = df.copy()
    px2 = config.pixel_size_um ** 2
    if "nucleus_area_px" in df.columns:
        df[out_col] = df["nucleus_area_px"] * px2
    elif "area_px" in df.columns:
        df[out_col] = df["area_px"] * px2
    elif out_col in df.columns:
        pass
    else:
        raise ValueError("need one of nucleus_area_px / area_px / " + out_col)
    return df


def attach_true_time(df: pd.DataFrame, tracked_df: pd.DataFrame) -> pd.DataFrame:
    """Merge true_time_min onto a table keyed by (t, track_id).

    radial_df carries only (t, track_id, ...) -- acquisition time lives on
    tracked_df / timed_df.
    """
    if "true_time_min" in df.columns:
        return df
    if tracked_df is None or "true_time_min" not in tracked_df.columns:
        out = df.copy()
        out["true_time_min"] = out["t"].astype(float)
        return out
    lut = (tracked_df[["t", "track_id", "true_time_min"]]
           .drop_duplicates(subset=["t", "track_id"]))
    return df.merge(lut, on=["t", "track_id"], how="left")


def _time_bins(time_min: pd.Series, bin_min: float) -> Tuple[pd.Series, np.ndarray]:
    """Left edge of each observation's bin, plus the sorted unique edges."""
    lo = np.floor(np.asarray(time_min, dtype=float) / bin_min) * bin_min
    return pd.Series(lo, index=time_min.index), np.unique(lo[np.isfinite(lo)])


def _close_polar(theta: np.ndarray, r: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Repeat the first point at theta+2pi so a polar curve closes."""
    theta = np.asarray(theta, dtype=float)
    r = np.asarray(r, dtype=float)
    if theta.size == 0:
        return theta, r
    return np.append(theta, theta[0] + 2 * np.pi), np.append(r, r[0])


def _orient_polar(ax) -> None:
    """Put a polar axis in the same frame as the displayed image.

    The sweep casts rays as y = cy + r*sin(theta), x = cx + r*cos(theta). Image
    y increases DOWNWARD, so theta increases clockwise on screen and theta=0
    points right (+x). Matplotlib's polar default is zero at east but
    counter-clockwise, which mirrors the nucleus about the horizontal axis --
    fine for a shape summary, wrong the moment you compare a rose against a
    raw crop or against phi_deg from the asymmetry table.
    """
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(-1)


def _t_colors(all_t: Sequence) -> Dict:
    cmap = plt.get_cmap("viridis")
    n = max(len(all_t) - 1, 1)
    return {t: cmap(i / n) for i, t in enumerate(sorted(all_t))}

### 36a. Radius coordinate — `rho_wall`

`radial_profile_for_tracked_nucleus` writes `distance_norm = distance_px /
ray_length_px`, which is a fraction of the **centroid**-to-wall distance. Since
`radial_exclude_nucleus=True` drops interior samples, the smallest value on a
ray is `R_nucleus / R_droplet` — around 0.17 on this dataset, not 0. Plotting it
on an axis labelled *nucleus edge → droplet edge* therefore mislabels the origin,
and worse, the same `distance_norm` value means a different physical distance
from the envelope for a small nucleus than for a large one, so binning it pools
non-comparable points as nuclei grow.

`add_wall_normalised_radius` recomputes the schema's `Rho_Normalized` per ray —
0 at the nuclear surface, 1 at the wall — and also adds `r_um` and
`r_from_edge_um` for work in absolute microns. It does not modify
`radial_sweep.pkl`; it returns an augmented copy.

In [ ]:
def add_wall_normalised_radius(radial_df: pd.DataFrame, config,
                               out_col: str = "rho_wall") -> pd.DataFrame:
    """Add a radius coordinate that runs 0 at the nuclear surface -> 1 at the wall.

    `distance_norm` as written by the sweep is distance_px / ray_length_px, i.e.
    a fraction of the CENTROID-to-wall distance. Because the sweep skips samples
    inside the nucleus, the smallest value on a ray is R_nucleus / R_droplet
    (~0.2-0.5), not 0 -- so a plot with `distance_norm` on the x axis labelled
    "nucleus edge -> droplet edge" is mis-labelled, and the effective bin width
    differs between a big nucleus and a small one.

    This recomputes the schema's Rho_Normalized per ray:

        rho_wall = (r - r_edge) / (r_wall - r_edge)

    r_edge is the last sample inside the nucleus when the sweep kept interior
    points, otherwise the first retained sample (which sits just outside the
    mask boundary). Also adds r_um and r_from_edge_um for absolute-distance work.
    """
    if radial_df is None or radial_df.empty:
        return radial_df

    df = radial_df.copy()
    keys = ["t", "track_id", "theta_index"]
    g = df.groupby(keys)["distance_px"]

    r_first = g.transform("min")
    if "inside_nucleus" in df.columns and bool(df["inside_nucleus"].any()):
        inside_d = df["distance_px"].where(df["inside_nucleus"])
        r_edge = inside_d.groupby([df[k] for k in keys]).transform("max")
        r_edge = r_edge.fillna(r_first)
    else:
        r_edge = r_first

    r_wall = (df["ray_length_px"] if "ray_length_px" in df.columns
              else g.transform("max"))

    gap = (r_wall - r_edge).replace(0, np.nan)
    df[out_col] = ((df["distance_px"] - r_edge) / gap).clip(0.0, 1.0)
    df["r_um"] = df["distance_px"] * config.pixel_size_um
    df["r_edge_um"] = r_edge * config.pixel_size_um
    df["r_from_edge_um"] = df["r_um"] - df["r_edge_um"]
    df["theta_deg"] = np.degrees(df["theta_rad"]) % 360.0
    return df

### 36b. Rose plots — envelope radius by angle

For each ray, the radius at which the NE channel peaks — a mask-independent
estimate of nuclear shape, and the one that shows whether growth is isotropic.

**The peak search is capped** at `rho_wall <= search_rho_max` (default 0.6).
The radial channel is Ch0, which images the droplet oil–water interface as well
as the envelope, and on this rig the interface is usually the brightest thing on
the ray. An uncapped `argmax` therefore returns the droplet boundary for any
nucleus whose NE signal is weaker than its own droplet edge, and the rose plot
becomes a picture of the droplet — which is what the earlier scratch cell was
showing when its peak-radius histogram sat near the droplet radius instead of
the expected nuclear radius.

Watch `frac_at_search_edge` in the printout. If it's high, the cap rather than
the data is picking the peak and you should look at a radial profile directly
before trusting the rose.

Polar axes use `_orient_polar`: zero at east, clockwise. The sweep casts rays
with `y = cy + r·sin θ` and image y increases downward, so θ runs clockwise on
screen — matplotlib's counter-clockwise default silently mirrors the nucleus.

In [ ]:
def compute_envelope_radius_by_angle(
    radial_df: pd.DataFrame,
    config,
    search_rho_max: float = 0.6,
    verbose: bool = True,
) -> pd.DataFrame:
    """Per (t, track_id, ray): the radius at which the NE channel peaks.

    The search is restricted to rho_wall <= search_rho_max. The radial channel
    is Ch0, which images the droplet oil-water interface as well as the nuclear
    envelope, and the interface is usually the brightest thing on the ray -- an
    unrestricted argmax returns the droplet wall for any nucleus whose NE signal
    is weaker than its own droplet boundary, which turns the rose plot into a
    picture of the droplet. Pass search_rho_max=1.0 for the unrestricted
    behaviour, and watch `frac_at_search_edge` in the printout: a high value
    means the cap is doing the deciding, not the data.
    """
    cols = ["t", "track_id", "theta_index", "theta_rad", "peak_r_um",
            "peak_intensity", "rho_at_peak", "at_search_edge"]
    if radial_df is None or radial_df.empty:
        return pd.DataFrame(columns=cols)

    df = radial_df
    if "rho_wall" not in df.columns:
        df = add_wall_normalised_radius(df, config)

    search = df[df["rho_wall"] <= float(search_rho_max)]
    if search.empty:
        return pd.DataFrame(columns=cols)

    idx = search.groupby(["t", "track_id", "theta_index"])["intensity"].idxmax()
    peaks = search.loc[idx, ["t", "track_id", "theta_index", "theta_rad",
                             "distance_px", "intensity", "rho_wall"]].copy()
    peaks = peaks.rename(columns={"intensity": "peak_intensity",
                                  "rho_wall": "rho_at_peak"})
    peaks["peak_r_um"] = peaks["distance_px"] * config.pixel_size_um
    peaks["at_search_edge"] = peaks["rho_at_peak"] >= 0.98 * float(search_rho_max)
    peaks = peaks.drop(columns="distance_px").reset_index(drop=True)[cols]

    if verbose:
        frac = float(peaks["at_search_edge"].mean())
        print(f"envelope peaks: {len(peaks)} rays, "
              f"search_rho_max={search_rho_max}, "
              f"frac_at_search_edge={frac:.3f}")
        print(peaks.groupby("t")["peak_r_um"].median().round(2).to_string())
    return peaks


def plot_rose_individual(peak_df: pd.DataFrame, tracked_df: pd.DataFrame, config,
                         n: int = 5, selection: str = "longest",
                         share_r: bool = True, save_as: Optional[str] = "rose_individual"):
    """Polar envelope-shape plot for n example nuclei, one curve per timepoint."""
    if peak_df is None or peak_df.empty:
        print("No data to plot."); return []

    if selection == "longest":
        counts = (tracked_df.groupby("track_id")["t"].nunique()
                  .sort_values(ascending=False))
        track_ids = [tid for tid in counts.index if tid in set(peak_df["track_id"])][:n]
    else:
        track_ids = sorted(peak_df["track_id"].unique())[:n]
    if not track_ids:
        print("No tracks in common between peak_df and tracked_df."); return []

    n_cols = min(5, len(track_ids))
    n_rows = math.ceil(len(track_ids) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.6 * n_cols, 3.9 * n_rows),
                             subplot_kw={"projection": "polar"}, squeeze=False)
    axes = axes.ravel()
    colors = _t_colors(peak_df["t"].unique())
    r_max = float(peak_df[peak_df.track_id.isin(track_ids)]["peak_r_um"].quantile(0.99))

    for ax, tid in zip(axes, track_ids):
        d = peak_df[peak_df.track_id == tid]
        for t in sorted(d["t"].unique()):
            dt = d[d.t == t].sort_values("theta_index")
            th, r = _close_polar(dt.theta_rad.to_numpy(), dt.peak_r_um.to_numpy())
            ax.plot(th, r, color=colors[t], lw=1.2, label=f"t={t}")
        ax.set_title(f"track {tid}", fontsize=10)
        _orient_polar(ax)
        ax.tick_params(labelsize=7)
        if share_r:
            ax.set_ylim(0, r_max)

    for ax in axes[len(track_ids):]:
        ax.axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right", fontsize=7,
               title="frame", title_fontsize=7)
    fig.suptitle("Envelope radius by angle (\u00b5m) \u2014 individual nuclei", fontsize=13)
    fig.tight_layout(rect=(0, 0, 0.90, 1))
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return track_ids


def plot_rose_pooled(peak_df: pd.DataFrame, config, show_iqr: bool = True,
                     min_rays: int = 5, save_as: Optional[str] = "rose_pooled"):
    """Population-pooled rose: radius = median peak_r_um over nuclei per (t, theta)."""
    if peak_df is None or peak_df.empty:
        print("No data to plot."); return

    pooled = (peak_df.groupby(["t", "theta_index"])
              .agg(theta_rad=("theta_rad", "first"),
                   med=("peak_r_um", "median"),
                   q1=("peak_r_um", lambda s: s.quantile(0.25)),
                   q3=("peak_r_um", lambda s: s.quantile(0.75)),
                   n=("peak_r_um", "size"))
              .reset_index())
    pooled = pooled[pooled["n"] >= min_rays]
    if pooled.empty:
        print(f"No (t, theta) bin reached min_rays={min_rays}."); return

    fig, ax = plt.subplots(figsize=(6.4, 6.4), subplot_kw={"projection": "polar"})
    colors = _t_colors(pooled["t"].unique())
    for t in sorted(pooled["t"].unique()):
        dt = pooled[pooled.t == t].sort_values("theta_index")
        th, r = _close_polar(dt.theta_rad.to_numpy(), dt["med"].to_numpy())
        ax.plot(th, r, color=colors[t], lw=1.6, label=f"t={t}")
        if show_iqr:
            _, lo = _close_polar(dt.theta_rad.to_numpy(), dt["q1"].to_numpy())
            th2, hi = _close_polar(dt.theta_rad.to_numpy(), dt["q3"].to_numpy())
            ax.fill_between(th2, lo, hi, color=colors[t], alpha=0.12, lw=0)
    _orient_polar(ax)
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.05, 1.05), title="frame")
    ax.set_title("Envelope radius by angle (\u00b5m)\nmedian \u00b1 IQR, pooled over nuclei",
                 fontsize=12, pad=18)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return pooled

### 36c. Membrane intensity by angle and distance

The θ × r intensity map, one panel per frame: the raw evidence behind both the
rose and the asymmetry index. A complete envelope is a continuous bright band at
r ≈ 0 spanning all 360°; a partial one is a band with gaps; directional delivery
is a band that is brighter over a contiguous arc.

`radius_col="rho_wall"` spans the full gap and shows the droplet wall as a
second bright band at r = 1. Because the wall usually outshines the envelope, it
compresses the NE contrast — pass `radius_col="r_from_edge_um"` with
`r_max_um≈8` to crop to the perinuclear zone, which is the more readable view
for membrane delivery.

Empty (θ, r) cells are drawn in grey rather than interpolated, and the function
warns when more than 5 % of the grid is empty — that means `rbins` is finer than
the ray sampling supports.

In [ ]:
def build_angle_distance_grid(radial_df: pd.DataFrame, config, rbins: int = 60,
                              radius_col: str = "rho_wall",
                              r_max_um: Optional[float] = None,
                              agg: str = "mean") -> Dict:
    """theta x radius intensity grids, one per timepoint.

    radius_col='rho_wall'        -> 0 at the nuclear surface, 1 at the droplet wall
    radius_col='r_from_edge_um'  -> absolute microns outward from the nuclear surface
    """
    if radial_df is None or radial_df.empty:
        return {}
    df = radial_df
    if radius_col not in df.columns:
        df = add_wall_normalised_radius(df, config)

    if radius_col == "rho_wall":
        edges = np.linspace(0.0, 1.0, rbins + 1)
    else:
        hi = float(r_max_um if r_max_um is not None
                   else np.nanpercentile(df[radius_col], 99))
        edges = np.linspace(0.0, hi, rbins + 1)

    d = df.copy()
    d["rbin"] = pd.cut(d[radius_col], edges, labels=False, include_lowest=True)
    d = d[d["rbin"].notna()]

    n_ang = int(config.radial_n_angles)
    grids = {}
    for t, sub in d.groupby("t"):
        grids[int(t)] = (sub.pivot_table(index="theta_index", columns="rbin",
                                         values="intensity", aggfunc=agg)
                         .reindex(index=range(n_ang), columns=range(rbins))
                         .to_numpy())
    return {"grids": grids, "edges": edges, "radius_col": radius_col}


def plot_membrane_angle_distance(radial_df: pd.DataFrame, config,
                                 track_id: Optional[int] = None,
                                 rbins: int = 60,
                                 radius_col: str = "rho_wall",
                                 r_max_um: Optional[float] = None,
                                 clip_pct: Tuple[float, float] = (2, 98),
                                 cmap: str = "inferno",
                                 max_cols: int = 5,
                                 save_as: Optional[str] = None):
    """Membrane intensity as a function of angle and distance, one panel per frame.

    track_id=None pools every nucleus in absolute lab angle. Pooling is only
    meaningful if asymmetry shares a direction across nuclei -- check panel (c)
    of the asymmetry figure first; if the population direction is random, the
    pooled map washes out and the single-nucleus view is the informative one.
    """
    if radial_df is None or radial_df.empty:
        print("No data to plot."); return

    df = radial_df
    if "rho_wall" not in df.columns:
        df = add_wall_normalised_radius(df, config)

    if track_id is None:
        span = df.groupby("track_id")["t"].nunique().sort_values(ascending=False)
        label = "pooled over all nuclei"
        sub = df
    else:
        sub = df[df.track_id == int(track_id)]
        label = f"track {int(track_id)}"
        if sub.empty:
            print(f"track {track_id} not present in radial_df."); return

    built = build_angle_distance_grid(sub, config, rbins=rbins,
                                      radius_col=radius_col, r_max_um=r_max_um)
    grids, edges = built["grids"], built["edges"]
    if not grids:
        print("No data to plot."); return

    stacked = np.concatenate([g.ravel() for g in grids.values()])
    allv = stacked[np.isfinite(stacked)]
    if allv.size == 0:
        print("Grid is entirely empty."); return
    vmin, vmax = np.percentile(allv, clip_pct)

    empty_frac = 1.0 - allv.size / stacked.size
    if empty_frac > 0.05:
        print(f"WARNING: {empty_frac:.1%} of (theta, r) cells have no sample. "
              f"Lower rbins (currently {rbins}) or raise config.radial_step_size_px "
              f"resolution -- empty cells are drawn in grey, not interpolated.")

    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad("0.88")

    ts = sorted(grids)
    ncol = min(max_cols, len(ts))
    nrow = math.ceil(len(ts) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow),
                             squeeze=False, constrained_layout=True)
    xlabel = ("r (nuclear surface \u2192 droplet wall)" if radius_col == "rho_wall"
              else "distance from nuclear surface (\u00b5m)")
    for ax, t in zip(axes.ravel(), ts):
        im = ax.imshow(np.ma.masked_invalid(grids[t]), aspect="auto", origin="lower",
                       extent=[edges[0], edges[-1], 0, 360],
                       vmin=vmin, vmax=vmax, cmap=cmap_obj)
        ax.set_title(f"t={t}", fontsize=9)
        ax.set_xlabel(xlabel, fontsize=7)
        ax.set_ylabel("\u03b8 (deg)", fontsize=7)
        ax.set_yticks([0, 90, 180, 270, 360])
        ax.tick_params(labelsize=7)
    for ax in axes.ravel()[len(ts):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes, shrink=0.75, label="membrane intensity (a.u.)")
    fig.suptitle(f"Membrane intensity by angle and distance \u2014 {label}", fontsize=12)
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return built

### 36d. Membrane asymmetry vs time

`I(θ)` is the mean membrane intensity in a shell just outside the nuclear
surface (default 0–3 µm from the mask edge). Fitting the first Fourier harmonic

$$I(\theta) \approx A_0 + A_1\cos(\theta - \phi)$$

gives the headline metric **`asym_index` = A₁/A₀** — 0 for a uniform shell,
rising as the signal becomes one-sided — and **`phi_deg`**, the direction of the
bright side. `cv` and `lobe_ratio` are reported alongside because a single
harmonic is blind to two-lobed asymmetry; if `cv` climbs while `asym_index` does
not, the shell is patchy rather than polarised.

**Panel (c) is the control to read first.** It asks whether every nucleus is
bright on the *same* side of the image, via the circular resultant `R` of the
population's `phi` values against a Rayleigh threshold. Stochastic
dynein-mediated delivery should be strong per nucleus and randomly oriented
across them, giving R near zero. R above the threshold means the bright side has
a preferred lab-frame direction — which is the signature of an illumination
gradient, a tilted coverslip, or a systematic z-offset, not of biology. Rule
that out before interpreting panel (a).

A₁/A₀ > 1 is possible and means the harmonic fit implies negative intensity
somewhere — read it as "essentially all the signal is on one side".

In [ ]:
def _first_harmonic(theta: np.ndarray, values: np.ndarray) -> Tuple[float, float, float]:
    """Fit v(theta) ~ a0 + a1*cos(theta - phi); return (a0, a1, phi).

    phi is the angle of maximum signal. Uniform theta sampling assumed.
    """
    theta = np.asarray(theta, dtype=float)
    v = np.asarray(values, dtype=float)
    ok = np.isfinite(v) & np.isfinite(theta)
    if ok.sum() < 8:
        return np.nan, np.nan, np.nan
    theta, v = theta[ok], v[ok]
    a0 = float(v.mean())
    c1 = np.mean(v * np.exp(-1j * theta))
    return a0, float(2 * np.abs(c1)), float(-np.angle(c1))


def compute_membrane_asymmetry(radial_df: pd.DataFrame, config,
                               tracked_df: Optional[pd.DataFrame] = None,
                               band: Tuple[float, float] = (0.0, 3.0),
                               band_units: str = "um",
                               min_rays: int = 30) -> pd.DataFrame:
    """Per (t, track_id) asymmetry of the perinuclear membrane signal.

    The signal is I(theta) = mean intensity in a shell just outside the nuclear
    surface -- by default 0-3 microns from the mask edge, which is where
    delivered membrane accumulates. Set band_units='rho' to use a fraction of
    the nucleus-to-wall gap instead (scale-free but droplet-size dependent).

    Columns
    -------
    a0              mean perinuclear intensity around the nucleus
    a1              amplitude of the first Fourier harmonic
    asym_index      a1 / a0, the headline number: 0 = uniform shell,
                    ~0.3+ = clearly one-sided
    phi_deg         direction of the bright side, degrees (image frame, CCW from +x)
    cv              std/mean across angle -- catches multi-lobed asymmetry that
                    a single harmonic misses
    lobe_ratio      brightest decile of angles / dimmest decile
    n_rays          rays contributing
    """
    empty = pd.DataFrame(columns=["t", "track_id", "a0", "a1", "asym_index",
                                  "phi_rad", "phi_deg", "cv", "lobe_ratio",
                                  "n_rays", "true_time_min"])
    if radial_df is None or radial_df.empty:
        return empty

    df = radial_df
    if "rho_wall" not in df.columns:
        df = add_wall_normalised_radius(df, config)

    lo, hi = float(band[0]), float(band[1])
    col = "r_from_edge_um" if band_units == "um" else "rho_wall"
    shell = df[(df[col] >= lo) & (df[col] <= hi)]
    if shell.empty:
        print(f"No samples in band {band} ({band_units}).")
        return empty

    per_ray = (shell.groupby(["t", "track_id", "theta_index"])
               .agg(theta_rad=("theta_rad", "first"), I=("intensity", "mean"))
               .reset_index())

    rows = []
    for (t, tid), d in per_ray.groupby(["t", "track_id"]):
        if len(d) < min_rays:
            continue
        d = d.sort_values("theta_index")
        th, I = d.theta_rad.to_numpy(), d.I.to_numpy()
        a0, a1, phi = _first_harmonic(th, I)
        k = max(1, len(I) // 10)
        srt = np.sort(I)
        dim = float(srt[:k].mean())
        rows.append({
            "t": int(t), "track_id": int(tid),
            "a0": a0, "a1": a1,
            "asym_index": (a1 / a0) if (np.isfinite(a0) and a0 > 0) else np.nan,
            "phi_rad": phi, "phi_deg": float(np.degrees(phi) % 360.0),
            "cv": float(np.std(I) / a0) if a0 else np.nan,
            "lobe_ratio": float(srt[-k:].mean() / dim) if dim > 0 else np.nan,
            "n_rays": int(len(I)),
        })

    asym_df = pd.DataFrame(rows)
    if asym_df.empty:
        return empty
    return attach_true_time(asym_df, tracked_df).sort_values(["t", "track_id"])


def _circular_stats(phi_rad: np.ndarray) -> Tuple[float, float, float]:
    """Mean direction, resultant length R, and Rayleigh p for a set of angles."""
    phi = np.asarray(phi_rad, dtype=float)
    phi = phi[np.isfinite(phi)]
    n = phi.size
    if n == 0:
        return np.nan, np.nan, np.nan
    C = np.mean(np.exp(1j * phi))
    R = float(np.abs(C))
    p = float(np.exp(-n * R ** 2))          # Rayleigh test, large-n approximation
    return float(np.angle(C)), R, p


def plot_membrane_asymmetry_vs_time(asym_df: pd.DataFrame, config,
                                    metric: str = "asym_index",
                                    bin_min: float = 6.0,
                                    show_tracks: bool = True,
                                    save_as: Optional[str] = "membrane_asymmetry_vs_time"):
    """Three panels: asymmetry vs time, its distribution per time bin, and whether
    the bright side points the same way across the population.

    Panel (c) is the control that matters. If the population resultant R sits
    above the Rayleigh line, every nucleus is bright on the same side of the
    image -- which is what an illumination gradient or a coverslip tilt looks
    like, not what stochastic dynein-driven delivery looks like. Biological
    asymmetry should be strong per nucleus and randomly oriented across them.
    """
    if asym_df is None or asym_df.empty:
        print("No data to plot."); return

    d = asym_df.dropna(subset=[metric, "true_time_min"]).copy()
    if d.empty:
        print("No finite values to plot."); return
    d["bin_lo"], edges = _time_bins(d["true_time_min"], bin_min)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

    # (a) per-track trajectories + population median/IQR
    ax = axes[0]
    if show_tracks:
        for tid, dt in d.groupby("track_id"):
            if len(dt) < 2:
                continue
            dt = dt.sort_values("true_time_min")
            ax.plot(dt.true_time_min, dt[metric], color="0.7", lw=0.6, alpha=0.6, zorder=1)
    g = d.groupby("true_time_min")[metric]
    med, q1, q3 = g.median(), g.quantile(0.25), g.quantile(0.75)
    ax.fill_between(med.index, q1, q3, alpha=0.25, color="#4C72B0", lw=0, zorder=2)
    ax.plot(med.index, med, lw=2.2, color="#4C72B0", zorder=3, label="median \u00b1 IQR")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel({"asym_index": "asymmetry index  $A_1/A_0$",
                   "cv": "CV of perinuclear intensity",
                   "lobe_ratio": "bright decile / dim decile"}.get(metric, metric))
    ax.set_title("(a) Membrane asymmetry vs time")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

    # (b) distribution per time bin
    ax = axes[1]
    groups = [d.loc[d.bin_lo == lo, metric].to_numpy() for lo in edges]
    centers = edges + bin_min / 2
    ax.boxplot(groups, positions=centers, widths=bin_min * 0.7,
               showfliers=False, patch_artist=True,
               boxprops=dict(facecolor="#DD8452", alpha=0.55, lw=0.8),
               medianprops=dict(color="k", lw=1.4),
               whiskerprops=dict(lw=0.8), capprops=dict(lw=0.8))
    ax.set_xlim(edges[0] - bin_min * 0.7, edges[-1] + bin_min * 1.7)
    ax.set_xticks(centers)
    ax.set_xticklabels([f"{lo:g}\u2013{lo + bin_min:g}" for lo in edges],
                       rotation=45, ha="right", fontsize=8)
    ax.set_xlabel(f"Time bin (min, width {bin_min:g})")
    ax.set_ylabel("asymmetry index")
    ax.set_title(f"(b) Distribution per {bin_min:g}-min bin")
    ax.spines[["top", "right"]].set_visible(False)

    # (c) directional consistency across the population
    ax = axes[2]
    stats = []
    for lo in edges:
        sub = d[d.bin_lo == lo]
        mu, R, p = _circular_stats(sub["phi_rad"].to_numpy())
        stats.append({"bin_lo": lo, "n": len(sub), "mean_dir_deg": np.degrees(mu) % 360,
                      "R": R, "rayleigh_p": p})
    stats = pd.DataFrame(stats)
    ax.plot(stats.bin_lo + bin_min / 2, stats.R, marker="o", lw=1.8, color="#55A868",
            label="population resultant $R$")
    n_ref = stats["n"].replace(0, np.nan)
    ax.plot(stats.bin_lo + bin_min / 2, np.sqrt(-np.log(0.05) / n_ref),
            ls="--", color="k", lw=1, label="Rayleigh p=0.05")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time bin (min)")
    ax.set_ylabel("directional concentration $R$")
    ax.set_title("(c) Do all nuclei point the same way?")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Perinuclear membrane asymmetry", fontsize=13)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return stats


def plot_asymmetry_direction_rose(asym_df: pd.DataFrame, config, nbins: int = 24,
                                  save_as: Optional[str] = "asymmetry_direction_rose"):
    """Polar histogram of the bright-side direction, one panel per frame."""
    if asym_df is None or asym_df.empty:
        print("No data to plot."); return
    ts = sorted(asym_df["t"].unique())
    ncol = min(5, len(ts))
    nrow = math.ceil(len(ts) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.2 * nrow),
                             subplot_kw={"projection": "polar"}, squeeze=False)
    edges = np.linspace(0, 2 * np.pi, nbins + 1)
    for ax, t in zip(axes.ravel(), ts):
        phi = asym_df.loc[asym_df.t == t, "phi_rad"].dropna().to_numpy() % (2 * np.pi)
        counts, _ = np.histogram(phi, bins=edges)
        ax.bar(edges[:-1], counts, width=np.diff(edges), align="edge",
               color="#8172B3", alpha=0.8, edgecolor="w", lw=0.4)
        mu, R, p = _circular_stats(asym_df.loc[asym_df.t == t, "phi_rad"].to_numpy())
        if np.isfinite(R) and counts.max() > 0:
            ax.plot([mu, mu], [0, counts.max()], color="k", lw=1.6)
        ax.set_title(f"t={t}  n={len(phi)}\nR={R:.2f}, p={p:.3g}", fontsize=8)
        _orient_polar(ax)
        ax.tick_params(labelsize=6)
        ax.set_yticklabels([])
    for ax in axes.ravel()[len(ts):]:
        ax.axis("off")
    fig.suptitle("Direction of the membrane-bright side (image frame)", fontsize=12)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()

### 36e. Cross-sectional area by time interval

Box plot of nuclear cross-sectional area in fixed acquisition-time bins.

On bin width: `true_time_min = t · (tile_rows · tile_cols · minutes_per_tile) +
tile_offset_min`. With the current 2 × 3 tiling at 1 min/tile that is **6 minutes
per frame**, so a 6-minute bin is exactly one frame with its six tile offsets
folded in. Binning narrower than 6 min splits a frame by tile — that is a
stage-position axis, not a time axis, and any structure it shows is tile order.
Wider bins (12, 18) pool frames and are legitimate.

Pass `halo_df` for repaired, halo-measured areas, or `timed_df` for every
detection before tracking. Area is always recomputed from the pixel count with
the live `cfg.pixel_size_um`, so a stale pickle from a different calibration
can't leak into the figure.

In [ ]:
def plot_area_boxplot_binned(df: pd.DataFrame, config, bin_min: float = 6.0,
                             time_col: str = "true_time_min",
                             area_col: Optional[str] = None,
                             show_points: bool = True,
                             show_median_trend: bool = True,
                             show_n: bool = True,
                             ylim: Optional[Tuple[float, float]] = None,
                             title: Optional[str] = None,
                             save_as: Optional[str] = "area_boxplot_6min"):
    """Nuclear cross-sectional area, boxed into fixed-width acquisition-time bins.

    Note on bin width: true_time_min = t * (tile_rows * tile_cols *
    minutes_per_tile) + tile_offset_min. With the current 2x3 tiling at 1 min
    per tile that is 6 minutes per frame, so a 6-minute bin is exactly one
    frame and the tile offsets fall inside it. Narrower bins split a frame by
    tile, which is a stage-position axis, not a biology axis.
    """
    if df is None or df.empty:
        print("No data to plot."); return

    d = df.copy()
    if area_col is None:
        d = resolve_area_um2(d, config)
        area_col = "nucleus_area_um2"
    if time_col not in d.columns:
        raise ValueError(f"DataFrame must contain '{time_col}'.")
    d = d.dropna(subset=[area_col, time_col])
    if d.empty:
        print("No finite values to plot."); return

    d["bin_lo"], edges = _time_bins(d[time_col], bin_min)
    groups = [d.loc[d.bin_lo == lo, area_col].to_numpy() for lo in edges]
    centers = edges + bin_min / 2

    fig, ax = plt.subplots(figsize=(max(7.5, 0.85 * len(edges) + 3), 5))

    if show_points:
        rng = np.random.default_rng(0)
        for lo, vals in zip(edges, groups):
            if vals.size == 0:
                continue
            x = lo + bin_min / 2 + rng.uniform(-bin_min * 0.22, bin_min * 0.22, vals.size)
            ax.scatter(x, vals, s=7, alpha=0.25, color="0.35", lw=0, zorder=1)

    ax.boxplot(groups, positions=centers, widths=bin_min * 0.7,
               showfliers=False, patch_artist=True, zorder=2,
               boxprops=dict(facecolor="#4C72B0", alpha=0.45, lw=0.9),
               medianprops=dict(color="k", lw=1.6),
               whiskerprops=dict(lw=0.9), capprops=dict(lw=0.9))

    if show_median_trend:
        meds = [np.median(v) if v.size else np.nan for v in groups]
        ax.plot(centers, meds, color="#C44E52", lw=1.8, marker="o", ms=4,
                zorder=3, label="median")
        ax.legend(fontsize=9, frameon=False, loc="upper left", bbox_to_anchor=(0.0, 0.95))

    if show_n:
        # blended transform: x in data units, y in axes fraction, so the labels
        # sit inside the axes and cannot collide with the title
        trans = ax.get_xaxis_transform()
        for c, v in zip(centers, groups):
            ax.text(c, 0.99, f"n={v.size}", transform=trans, ha="center", va="top",
                    fontsize=7, color="0.35")

    ax.set_xlim(edges[0] - bin_min * 0.7, edges[-1] + bin_min * 1.7)
    ax.set_xticks(centers)
    ax.set_xticklabels([f"{lo:g}\u2013{lo + bin_min:g}" for lo in edges],
                       rotation=45, ha="right")
    ax.set_xlabel(f"Acquisition time bin (min, width {bin_min:g})")
    ax.set_ylabel("Nuclear cross-sectional area (\u00b5m\u00b2)")
    ax.set_title(title or f"Nuclear cross-sectional area per {bin_min:g}-min interval")
    if ylim:
        ax.set_ylim(*ylim)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()

    summary = (d.groupby("bin_lo")[area_col]
               .agg(n="size", median="median",
                    q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75),
                    mean="mean", std="std")
               .reset_index().rename(columns={"bin_lo": "bin_start_min"}))
    summary["bin_end_min"] = summary["bin_start_min"] + bin_min
    return summary.round(2)

### 36f. Run the full figure set

In [ ]:
# ── Full figure set ───────────────────────────────────────────────────────
# Every figure is written to <run>/exports/figures/ as PNG + PDF.
SAVE_FIGURES = True

# 0. carried over from v17 — N/C ratio and area vs time
plot_nc_vs_time(halo_df)
plot_area_vs_time(halo_df)
plot_largest_cross_sectional_area_vs_time(timed_df, cfg)

# 1. one-time prep: proper nucleus-surface -> wall radius on the sweep
radial_n_df = add_wall_normalised_radius(radial_df, cfg)

# 2. rose plots — envelope shape by angle
envelope_peak_df = compute_envelope_radius_by_angle(
    radial_n_df, cfg, search_rho_max=ENVELOPE_SEARCH_RHO_MAX)
_ = plot_rose_individual(envelope_peak_df, tracked_df, cfg, n=5, selection="longest")
pooled_rose_df = plot_rose_pooled(envelope_peak_df, cfg)

# 3. membrane intensity by angle and distance
_longest = tracked_df.groupby("track_id")["t"].nunique().idxmax()
plot_membrane_angle_distance(radial_n_df, cfg, track_id=_longest,
                             save_as=f"angle_distance_track{_longest}")
# rbins kept coarse here on purpose: rays are sampled every
# radial_step_size_px (1 px = 0.1625 um), so bins finer than ~1.5 px come back
# part-empty. 8 um / 32 bins = 0.25 um = 1.5 px.
plot_membrane_angle_distance(radial_n_df, cfg, track_id=None, rbins=32,
                             radius_col="r_from_edge_um", r_max_um=8.0,
                             save_as="angle_distance_pooled")

# 4. membrane asymmetry vs time
asym_df = compute_membrane_asymmetry(radial_n_df, cfg, tracked_df=tracked_df,
                                     band=ASYMMETRY_BAND_UM, band_units="um")
asym_stats = plot_membrane_asymmetry_vs_time(asym_df, cfg, bin_min=AREA_BIN_MIN)
plot_asymmetry_direction_rose(asym_df, cfg)
asym_df.to_csv(cfg.exports_dir / "membrane_asymmetry.csv", index=False)

# 5. cross-sectional area boxed into fixed time intervals
area_summary = plot_area_boxplot_binned(halo_df, cfg, bin_min=AREA_BIN_MIN)
display(area_summary)
area_summary.to_csv(cfg.exports_dir / "area_by_time_bin.csv", index=False)

print("figures ->", figures_dir(cfg))

---
## 37. Shell-resolved membrane asymmetry

The faceted rose — radial shells down the rows, timepoints across the columns —
plus the normalised-score supplement and the direction-agnostic aggregate.

Four things about the original version change what it says, so each is
implemented as an explicit, switchable choice rather than a silent default.

**1. Summed intensity is biased by the droplet wall.** Rays stop at the wall,
so a shell defined in absolute distance from the nuclear surface is clipped by
different amounts at different θ as soon as the nucleus sits off-centre in its
droplet. A sum then counts more pixels on the far side and reports asymmetry
pointing away from the near wall — *for a perfectly uniform field*. With a
uniform intensity field and a nucleus 6 µm off-centre, the outermost shell
returns R = 0.42; at 9 µm it returns 0.60. §37a bins on **mean per sample**,
which is invariant to how many pixels a cell happens to contain, and §37b adds
`R_geom` — the same resultant computed on the sample counts with intensity
discarded — as a per-facet floor.

**2. Radius encoding exaggerates magnitude.** On a polar bar the eye reads
area, and area goes as r². A linear radius shows a 2× difference as a 4× wedge.
§37c defaults to `radius_scale="sqrt"`, so wedge **area** is proportional to the
value. `"linear"` restores the literal encoding.

**3. The resultant needs a baseline that is not its own minimum.** With a
background pedestal *b* and signal *s*, R = |Σs·e^{iθ}| / (N·b + Σs), so a
bright cytoplasm drives R toward zero however one-sided the membrane is.
Subtracting each profile's own minimum overcorrects in a way that is worse:
R then becomes a pure *shape* statistic, returning **0.525 for a cosine of any
amplitude**, so a nucleus twice as one-sided scores identically. §37b subtracts
a single dataset-wide floor, giving R = (A₁/A₀)/2 — bounded on [0, 1], zero for
an even shell, linear in how one-sided it is.

**4. The resultant needs a null.** Even a uniform shell returns R ≈ 1/√n_eff,
about 0.17 for 36 bins, so every facet looks mildly asymmetric. §37b reports
`R_corr` (debiased), a Rayleigh p on the weighted resultant using Kish's
effective sample size, and `R_crit`, drawn on each facet as a dashed ring: an
arrow inside that ring is noise.

In [ ]:
from typing import Sequence                      # not in the Section 1 import block
from matplotlib.colors import LinearSegmentedColormap

# ── Section 37 parameters ─────────────────────────────────────────────────
# Shell edges, microns outward from the nuclear mask surface. These are the
# edges from the original figure; change them freely, but see 37b on why the
# outermost shell needs the geometry control.
DEFAULT_SHELL_EDGES_UM = (0.0, 2.34, 5.36, 9.55, 23.5)

# Angular bins. 36 bins = 10 degrees. Fewer bins raises the per-nucleus
# detection floor (R_crit goes as 1/sqrt(n_eff)); more bins makes each bin
# noisier without adding information once bins are finer than the ray spacing.
N_THETA_BINS = 36

# Fraction of theta below which a shell is flagged as wall-clipped.
MIN_SHELL_COVERAGE = 0.80

### 37a. Binning into (shell, θ) cells

Returns a **complete** θ grid per unit — cells with no sample are kept with
`n_samples = 0` rather than dropped, because their absence is the geometry
signal that §37b tests for.

The per-cell statistic is `I_mean`, not `I_sum`. `I_sum` is still carried so
the original encoding can be reproduced for comparison.

In [ ]:
def bin_shell_angle(radial_df: pd.DataFrame, config,
                    shell_edges_um: Sequence[float] = DEFAULT_SHELL_EDGES_UM,
                    n_theta_bins: int = N_THETA_BINS,
                    radius_col: str = "r_from_edge_um") -> pd.DataFrame:
    """Bin the sweep into (shell, theta) cells per nucleus per timepoint.

    Returns one row per (t, track_id, shell_idx, theta_bin) on a COMPLETE theta
    grid -- cells with no sample are kept with n_samples = 0 and I_mean = NaN,
    because their absence is itself the geometry signal that
    compute_shell_resultants tests for.

    I_mean, not I_sum, is the per-cell statistic. Rays terminate at the droplet
    wall, so a shell defined in absolute distance is clipped by different
    amounts at different theta whenever the nucleus is off-centre. A sum then
    counts more pixels on the far side and reports asymmetry pointing away from
    the near wall, for a perfectly uniform field. A mean is invariant to how
    many pixels a cell happens to contain.
    """
    if radial_df is None or radial_df.empty:
        return pd.DataFrame()

    df = radial_df
    if radius_col not in df.columns:
        raise ValueError(f"{radius_col} missing -- run add_wall_normalised_radius first.")

    edges = np.asarray(shell_edges_um, dtype=float)
    d = df[["t", "track_id", "theta_rad", radius_col, "intensity"]].copy()
    d["shell_idx"] = pd.cut(d[radius_col], edges, labels=False, include_lowest=True)
    d = d[d["shell_idx"].notna()]
    if d.empty:
        return pd.DataFrame()
    d["shell_idx"] = d["shell_idx"].astype(int)

    dtheta = 2 * np.pi / n_theta_bins
    d["theta_bin"] = ((d["theta_rad"] % (2 * np.pi)) // dtheta).astype(int)
    d["theta_bin"] = d["theta_bin"].clip(0, n_theta_bins - 1)

    g = (d.groupby(["t", "track_id", "shell_idx", "theta_bin"])["intensity"]
         .agg(n_samples="size", I_sum="sum", I_mean="mean")
         .reset_index())

    # complete the theta grid so empty cells are explicit
    units = g[["t", "track_id", "shell_idx"]].drop_duplicates()
    grid = units.merge(pd.DataFrame({"theta_bin": np.arange(n_theta_bins)}), how="cross")
    out = grid.merge(g, on=["t", "track_id", "shell_idx", "theta_bin"], how="left")
    out["n_samples"] = out["n_samples"].fillna(0).astype(int)
    out["I_sum"] = out["I_sum"].fillna(0.0)

    out["theta_rad"] = (out["theta_bin"] + 0.5) * dtheta
    out["shell_lo_um"] = edges[out["shell_idx"].to_numpy()]
    out["shell_hi_um"] = edges[out["shell_idx"].to_numpy() + 1]
    out["shell_label"] = [f"({lo:g}, {hi:g}]" for lo, hi
                          in zip(out.shell_lo_um, out.shell_hi_um)]
    out.attrs["n_theta_bins"] = n_theta_bins
    out.attrs["shell_edges_um"] = tuple(edges)
    return out.sort_values(["t", "track_id", "shell_idx", "theta_bin"]).reset_index(drop=True)

### 37b. Resultant vectors, baselines, and the two nulls

Four resultants per unit, each answering a different question:

| column | weights | what it measures |
|---|---|---|
| `R` | `I_mean − background` | **the score.** (A₁/A₀)/2 for a cosine; 0 = even, 1 = all one side |
| `R_raw` | `I_mean` | what an uncorrected rose shows; deflated by the pedestal |
| `R_shape` | `I_mean −` own 10th pct | concentration only — saturates at 0.525, never quote as magnitude |
| `R_geom` | `n_samples` | asymmetry from droplet-wall clipping alone |

`R` must clear both `R_crit` (the noise floor) and `R_geom` (the geometry
floor) to mean anything.

On `background`: it has to be estimated from outside the profile being scored —
`estimate_background` uses a low quantile of every sweep sample in the dataset,
which is the camera offset plus the isotropic cytoplasmic floor. Pass your own
value if you have a dark-frame measurement.

`n_eff` is Kish's effective sample size, `(Σw)²/Σw²`: the number of
equally-weighted bins carrying the same information. It is what the Rayleigh
statistic needs when observations are intensity weights rather than counts.

In [ ]:
def _resultant(theta: np.ndarray, w: np.ndarray) -> Tuple[float, float, float]:
    """Weighted circular resultant. Returns (R, phi, n_eff).

    n_eff is Kish's effective sample size (sum w)^2 / sum(w^2): the number of
    equally-weighted bins that would carry the same information. It is what the
    Rayleigh statistic needs when the observations are intensity weights rather
    than counts.
    """
    w = np.asarray(w, dtype=float)
    ok = np.isfinite(w) & np.isfinite(theta) & (w > 0)
    if ok.sum() < 3:
        return np.nan, np.nan, np.nan
    th, w = theta[ok], w[ok]
    sw = w.sum()
    if sw <= 0:
        return np.nan, np.nan, np.nan
    C = (w * np.exp(1j * th)).sum() / sw
    n_eff = float(sw ** 2 / (w ** 2).sum())
    return float(np.abs(C)), float(np.angle(C)), n_eff


def _debias_R(R: float, n_eff: float) -> float:
    """Bias-corrected resultant length.

    E[R^2] ~= rho^2 + (1 - rho^2)/n_eff, so even a perfectly uniform shell
    returns R ~ 1/sqrt(n_eff) -- about 0.17 for 36 bins. Reporting raw R makes
    every facet look mildly asymmetric. This inverts the relation and clips at
    zero.
    """
    if not (np.isfinite(R) and np.isfinite(n_eff)) or n_eff <= 1:
        return np.nan
    rho2 = (n_eff * R ** 2 - 1.0) / (n_eff - 1.0)
    return float(np.sqrt(max(rho2, 0.0)))


def _rayleigh_p(R: float, n_eff: float) -> float:
    """Rayleigh test on the weighted resultant (Zar's two-term approximation)."""
    if not (np.isfinite(R) and np.isfinite(n_eff)) or n_eff <= 1:
        return np.nan
    z = n_eff * R ** 2
    p = np.exp(-z) * (1 + (2 * z - z ** 2) / (4 * n_eff)
                      - (24 * z - 132 * z ** 2 + 76 * z ** 3 - 9 * z ** 4)
                      / (288 * n_eff ** 2))
    return float(np.clip(p, 0.0, 1.0))


def R_critical(n_eff: float, alpha: float = 0.05) -> float:
    """Resultant length a uniform shell would exceed with probability alpha."""
    if not np.isfinite(n_eff) or n_eff <= 1:
        return np.nan
    return float(np.sqrt(-np.log(alpha) / n_eff))


def estimate_background(radial_df: pd.DataFrame, q: float = 0.05) -> float:
    """Isotropic intensity floor for the whole dataset: the q-quantile of every
    sweep sample.

    This is the zero of the asymmetry scale, and it has to come from OUTSIDE the
    profile being scored. Subtracting a profile's own minimum instead makes R a
    shape statistic that saturates -- a cosine of ANY amplitude returns
    R = 0.525, so a nucleus twice as one-sided scores the same. Subtracting a
    common floor gives R = (A1/A0)/2, which is what a normalised asymmetry
    score should do.
    """
    if radial_df is None or radial_df.empty:
        return 0.0
    return float(np.quantile(radial_df["intensity"].to_numpy(), q))


def compute_shell_resultants(binned: pd.DataFrame,
                             background: Optional[float] = None,
                             background_q: float = 0.05,
                             min_coverage: float = 0.80,
                             alpha: float = 0.05) -> pd.DataFrame:
    """Per (t, track_id, shell): resultant vector, its null, and a geometry control.

    Four numbers per unit, each answering a different question:

    R        THE SCORE. Resultant of (I_mean - background), background being one
             dataset-wide floor (see estimate_background). For a cosine profile
             this is exactly (A1/A0)/2, so it is bounded on [0, 1], zero for an
             even shell, and linear in how one-sided the shell is.
    R_raw    the same without subtracting anything. Reported because it is what
             an uncorrected rose shows, and it is deflated by the pedestal: a
             bright cytoplasm pushes it toward zero however one-sided the
             membrane is. Compare the two to see how much background was
             masking.
    R_shape  resultant after removing each shell's OWN 10th percentile. This
             discards amplitude entirely and measures only how concentrated the
             bright side is -- a narrow lobe scores above a broad one. Useful,
             but never quote it as an asymmetry magnitude.
    R_geom   resultant of n_samples, intensity discarded. The asymmetry a
             perfectly uniform field would show given only how the droplet wall
             clipped the rays. R has to clear R_geom to mean anything.

    Also: R_corr (R debiased for finite bin count), rayleigh_p, R_crit,
    coverage, and A1_over_A0 = 2R for a cosine, quoted directly for continuity
    with Section 36d.
    """
    cols = ["t", "track_id", "shell_idx", "shell_label", "shell_lo_um", "shell_hi_um",
            "I_bar", "I_bg", "R", "R_corr", "R_raw", "R_shape",
            "phi_rad", "phi_deg", "n_eff", "rayleigh_p", "R_crit", "significant",
            "R_geom", "phi_geom_deg", "R_excess", "coverage", "n_occupied",
            "A1_over_A0", "n_samples_total"]
    if binned is None or binned.empty:
        return pd.DataFrame(columns=cols)

    if background is None:
        occ_vals = binned.loc[binned.n_samples > 0, "I_mean"].to_numpy()
        background = float(np.quantile(occ_vals, background_q)) if occ_vals.size else 0.0
        print(f"background floor estimated from binned data: {background:.1f} "
              f"(q={background_q}). Pass background=... to override.")

    rows = []
    for (t, tid, sh), d in binned.groupby(["t", "track_id", "shell_idx"], sort=True):
        d = d.sort_values("theta_bin")
        th = d["theta_rad"].to_numpy()
        I = d["I_mean"].to_numpy(dtype=float)
        n = d["n_samples"].to_numpy(dtype=float)

        occ = n > 0
        coverage = float(occ.mean())
        if occ.sum() < 3:
            continue

        Ifin = np.where(occ, I, np.nan)
        I_bar = float(np.nanmean(Ifin))

        w_raw = np.where(occ, Ifin, 0.0)
        w_sig = np.clip(w_raw - background, 0.0, None) * occ
        w_shape = np.clip(w_raw - np.nanquantile(Ifin, 0.10), 0.0, None) * occ

        R_raw, _, _ = _resultant(th, w_raw)
        R, phi, n_eff = _resultant(th, w_sig)
        R_shape, _, _ = _resultant(th, w_shape)
        R_geom, phi_g, _ = _resultant(th, n)

        # first harmonic of the background-subtracted profile
        sig = np.where(occ, Ifin - background, np.nan)
        base = np.nanmean(sig)
        if occ.sum() >= 8 and np.isfinite(base) and base > 0:
            c1 = np.nanmean(sig * np.exp(-1j * th))
            A1_A0 = float(2 * np.abs(c1) / base)
        else:
            A1_A0 = np.nan

        R_corr = _debias_R(R, n_eff)
        p = _rayleigh_p(R, n_eff)
        Rc = R_critical(n_eff, alpha)
        rows.append({
            "t": int(t), "track_id": int(tid), "shell_idx": int(sh),
            "shell_label": d["shell_label"].iloc[0],
            "shell_lo_um": float(d["shell_lo_um"].iloc[0]),
            "shell_hi_um": float(d["shell_hi_um"].iloc[0]),
            "I_bar": I_bar, "I_bg": float(background),
            "R": R, "R_corr": R_corr, "R_raw": R_raw, "R_shape": R_shape,
            "phi_rad": phi,
            "phi_deg": float(np.degrees(phi) % 360.0) if np.isfinite(phi) else np.nan,
            "n_eff": n_eff, "rayleigh_p": p, "R_crit": Rc,
            "significant": bool(np.isfinite(p) and p < alpha),
            "R_geom": R_geom,
            "phi_geom_deg": float(np.degrees(phi_g) % 360.0) if np.isfinite(phi_g) else np.nan,
            "R_excess": (R_corr - R_geom) if np.isfinite(R_corr) and np.isfinite(R_geom) else np.nan,
            "coverage": coverage, "n_occupied": int(occ.sum()),
            "A1_over_A0": A1_A0,
            "n_samples_total": float(n.sum()),
        })

    out = pd.DataFrame(rows, columns=cols)
    if not out.empty:
        out["low_coverage"] = out["coverage"] < min_coverage
        n_low = int(out["low_coverage"].sum())
        if n_low:
            print(f"NOTE: {n_low}/{len(out)} shell-units below {min_coverage:.0%} theta "
                  f"coverage -- the droplet wall cuts these shells off. They are kept "
                  f"but flagged; read R_geom for them.")
    return out

### 37c. The faceted rose

Rows = shells, columns = timepoints. Reading the facet:

- **coloured wedges** — mean intensity above the background floor, area-scaled
- **grey wedges** — angles with no sample at all (wall clipping), marked rather
  than left blank
- **red arrow** — the resultant, length = R on the 0–1 radius; solid red when
  p < 0.05, grey otherwise
- **dashed ring** — `R_crit`; an arrow inside it is noise
- **dotted grey arrow** — `R_geom`. Where it rivals the red arrow, the facet is
  showing droplet placement
- **facet label** — `R = score`, `g = R_geom`, `*` = significant

`share_r="row"` puts a shell's timepoints on one radial scale so the columns are
comparable; `"all"` compares shells too; `"none"` autoscales each facet and
should only be used to read shape.

In [ ]:
def _rose_axis_values(d: pd.DataFrame, res_row, value: str) -> np.ndarray:
    occ = d["n_samples"].to_numpy() > 0
    I = d["I_mean"].to_numpy(dtype=float)
    if value == "mean":
        v = np.where(occ, I, 0.0)
    elif value == "mean_bgsub":
        v = np.where(occ, I - float(res_row["I_bg"]), 0.0)
    elif value == "sum":
        v = d["I_sum"].to_numpy(dtype=float)
    else:
        raise ValueError(f"unknown value {value!r}")
    return np.clip(v, 0.0, None)


def plot_shell_rose_facets(binned: pd.DataFrame, resultants: pd.DataFrame, config,
                           track_id: Optional[int] = None,
                           timepoints: Optional[Sequence[int]] = None,
                           value: str = "mean_bgsub",
                           radius_scale: str = "sqrt",
                           share_r: str = "row",
                           show_geom_arrow: bool = True,
                           alpha: float = 0.05,
                           save_as: str = "shell_rose_facets"):
    """Rose facets: rows = radial shells, columns = timepoints.

    Encoding choices, all of which change what the reader concludes:

    radius_scale="sqrt" (default) makes wedge AREA proportional to the value.
        On a polar bar the eye reads area, and area goes as r^2, so a linear
        radius shows a 2x difference as a 4x wedge. Every Nightingale rose has
        this problem; sqrt is the standard correction. Pass "linear" to get the
        literal radius encoding back.

    value="mean_bgsub" (default) plots mean intensity per sample with the
        per-shell baseline removed, so the origin is background rather than
        zero counts and the bars show the part of the signal that can be
        asymmetric. "mean" keeps the pedestal and draws it as a grey ring so
        you can see how much of each bar is background. "sum" reproduces the
        original encoding and is biased wherever coverage < 1.

    share_r="row" puts every timepoint in a shell on one radial scale, so the
        columns are comparable; "all" compares shells too; "none" autoscales
        each facet and should only be used for shape.

    The red arrow is the background-subtracted resultant, length R x r_limit.
    The dashed ring is R_crit, the length a uniform shell clears 5 % of the
    time -- an arrow inside that ring is noise. The open grey arrow is R_geom,
    the resultant of the sample counts alone: the asymmetry produced by the
    droplet wall clipping the rays. Where the grey arrow rivals the red one,
    the facet is showing geometry.
    """
    if binned is None or binned.empty or resultants is None or resultants.empty:
        print("No data to plot."); return

    b, r = binned, resultants
    if track_id is not None:
        b = b[b.track_id == int(track_id)]
        r = r[r.track_id == int(track_id)]
        subtitle = f"track {int(track_id)}"
        if b.empty:
            print(f"track {track_id} not present."); return
    else:
        subtitle = "per-nucleus facets require track_id; pooling shown"

    ts = sorted(b["t"].unique()) if timepoints is None else list(timepoints)
    shells = sorted(b["shell_idx"].unique())
    if not ts or not shells:
        print("No data to plot."); return

    key = ["t", "track_id", "shell_idx"]
    res_lut = r.set_index(key)

    # precompute values so scales can be shared
    vals: Dict[Tuple[int, int], np.ndarray] = {}
    for sh in shells:
        for t in ts:
            try:
                row = res_lut.loc[(t, int(track_id), sh)]
            except KeyError:
                continue
            d = b[(b.t == t) & (b.shell_idx == sh)].sort_values("theta_bin")
            if d.empty:
                continue
            vals[(sh, t)] = _rose_axis_values(d, row, value)

    if not vals:
        print("No data to plot."); return

    def _lim(keys):
        pool = np.concatenate([vals[k] for k in keys if k in vals])
        pool = pool[np.isfinite(pool)]
        return float(np.percentile(pool, 99.5)) if pool.size else 1.0

    if share_r == "all":
        lim_all = _lim(list(vals))
        limits = {sh: lim_all for sh in shells}
    elif share_r == "row":
        limits = {sh: _lim([(sh, t) for t in ts]) for sh in shells}
    else:
        limits = None

    xf = (lambda v, L: np.sqrt(np.clip(v, 0, None) / L) if radius_scale == "sqrt"
          else np.clip(v, 0, None) / L)

    dtheta = 2 * np.pi / b.attrs.get("n_theta_bins", N_THETA_BINS)
    cmap = LinearSegmentedColormap.from_list(
        "shells", ["#7FD1B9", "#2A9D8F", "#3D5A99", "#1B1B2F"])
    shell_colors = {sh: cmap(i / max(len(shells) - 1, 1)) for i, sh in enumerate(shells)}

    fig, axes = plt.subplots(len(shells), len(ts),
                             figsize=(2.05 * len(ts), 2.25 * len(shells)),
                             subplot_kw={"projection": "polar"}, squeeze=False)

    for i, sh in enumerate(shells):
        for j, t in enumerate(ts):
            ax = axes[i][j]
            ax.set_theta_zero_location("E")
            ax.set_theta_direction(-1)          # image frame: y grows downward
            ax.set_yticklabels([])
            ax.set_xticks(np.radians([0, 90, 180, 270]))
            ax.set_xticklabels([])   # orientation stated once in the subtitle
            ax.tick_params(labelsize=5.5, pad=-2)
            ax.grid(alpha=0.25, lw=0.4)
            ax.set_ylim(0, 1.0)

            if (sh, t) not in vals:
                ax.set_axis_off(); continue
            row = res_lut.loc[(t, int(track_id), sh)]
            d = b[(b.t == t) & (b.shell_idx == sh)].sort_values("theta_bin")
            v = vals[(sh, t)]
            L = limits[sh] if limits else max(float(np.max(v)), 1e-12)

            occ = d["n_samples"].to_numpy() > 0
            rr = xf(v, L)
            colors = [shell_colors[sh] if o else "none" for o in occ]
            ax.bar(d["theta_rad"].to_numpy(), rr, width=dtheta * 0.92,
                   bottom=0.0, color=colors, edgecolor="none", align="center")
            # unsampled angles marked, not silently blank
            if (~occ).any():
                ax.bar(d["theta_rad"].to_numpy()[~occ], np.ones((~occ).sum()),
                       width=dtheta * 0.92, bottom=0.0, color="0.90", zorder=0)

            if value == "mean" and np.isfinite(row["I_bg"]):
                ax.plot(np.linspace(0, 2 * np.pi, 200),
                        np.full(200, xf(row["I_bg"], L)),
                        color="0.45", lw=0.7, ls=":", zorder=4)

            Rv, phi = float(row["R"]), float(row["phi_rad"])
            Rc = float(row["R_crit"])
            if np.isfinite(Rc):
                ax.plot(np.linspace(0, 2 * np.pi, 200), np.full(200, Rc),
                        color="k", lw=0.6, ls="--", alpha=0.5, zorder=4)
            if np.isfinite(Rv) and np.isfinite(phi):
                sig = bool(row["significant"])
                ax.annotate("", xy=(phi, Rv), xytext=(0, 0),
                            arrowprops=dict(arrowstyle="-|>", lw=1.4,
                                            color="#D7263D" if sig else "0.55",
                                            alpha=1.0 if sig else 0.7), zorder=6)
            if show_geom_arrow and np.isfinite(row["R_geom"]) and row["R_geom"] > 0.02:
                ax.annotate("", xy=(np.radians(row["phi_geom_deg"]), row["R_geom"]),
                            xytext=(0, 0),
                            arrowprops=dict(arrowstyle="-|>", lw=1.1, color="0.35",
                                            ls=":", alpha=0.9), zorder=5)

            star = "*" if row["significant"] else ""
            ax.annotate(f"R={row['R']:.2f}{star}  g={row['R_geom']:.2f}",
                        xy=(0.5, -0.13), xycoords="axes fraction",
                        ha="center", fontsize=6.5,
                        color="0.15" if row["significant"] else "0.5")
            if i == 0:
                ax.annotate(f"t = {t}", xy=(0.5, 1.18), xycoords="axes fraction",
                            ha="center", fontsize=9)
            if j == 0:
                ax.annotate(f"{row['shell_label']} \u00b5m", xy=(-0.38, 0.5),
                            xycoords="axes fraction", rotation=90,
                            va="center", ha="center", fontsize=7.5)

    vlab = {"mean_bgsub": "mean intensity above shell baseline",
            "mean": "mean intensity per sample",
            "sum": "summed intensity (biased where coverage < 1)"}[value]
    rlab = ("radius \u221d \u221avalue, so wedge AREA \u221d value"
            if radius_scale == "sqrt" else "radius \u221d value (area \u221d value\u00b2)")
    fig.suptitle("Membrane intensity by angle and distance", fontsize=13, x=0.02, ha="left", y=0.995)
    fig.text(0.02, 0.963, f"10\u00b0 bins \u00b7 {vlab} \u00b7 {rlab} \u00b7 {subtitle}",
             fontsize=8, ha="left", color="0.25")
    fig.text(0.02, 0.941, "0\u00b0 = image +x (right); angle increases clockwise, "
             "matching the displayed image", fontsize=7.5, ha="left", color="0.45")
    fig.text(0.02, 0.012,
             "red arrow = background-subtracted resultant (length = R); dashed ring = R at p=0.05; "
             "dotted grey arrow = R_geom, the resultant of sample counts alone (droplet-wall clipping). "
             "Facet labels: R = asymmetry score, g = R_geom, * = p < %.2f. "
             "Grey wedges = angles with no sample." % alpha,
             fontsize=6.5, color="0.3")
    fig.tight_layout(rect=(0.02, 0.03, 1, 0.925))
    save_figure(fig, save_as, config)
    plt.show()
    return fig

### 37d. Normalised asymmetry score — table and graph

`build_asymmetry_score_table` collapses the per-nucleus vectors by (shell, t):
median score with IQR, the geometry floor, `R_excess` (the part geometry does
not explain), the fraction of nuclei individually clearing Rayleigh, and
`pop_R` — the coherence of φ **across** nuclei.

`pop_R` is the artefact check carried over from §36d. Stochastic delivery
should be strong per nucleus and randomly oriented across them. A `pop_R` above
its p = 0.05 line means the bright side has a preferred lab-frame direction,
which is what an illumination gradient or a tilted coverslip looks like.

In [ ]:
def build_asymmetry_score_table(resultants: pd.DataFrame,
                                by: Sequence[str] = ("shell_label", "t")) -> pd.DataFrame:
    """Aggregate the per-nucleus vectors into a reportable score table.

    Columns
    -------
    n                 shell-units contributing
    R_corr_median     normalised asymmetry score, 0 = even, 1 = all on one side
    R_corr_q1/q3      IQR across nuclei
    R_geom_median     the same statistic on sample counts alone -- the floor
                      set by droplet geometry
    R_excess_median   R_corr - R_geom, the part not explained by geometry
    frac_significant  fraction of nuclei clearing the per-nucleus Rayleigh test
    pop_R             coherence of phi ACROSS nuclei (lab-frame directionality)
    pop_p             Rayleigh p on pop_R
    coverage_min      worst theta coverage in the group
    """
    if resultants is None or resultants.empty:
        return pd.DataFrame()

    def agg(d):
        phi = d["phi_rad"].dropna().to_numpy()
        n = phi.size
        if n:
            C = np.mean(np.exp(1j * phi))
            pop_R = float(np.abs(C))
            pop_dir = float(np.degrees(np.angle(C)) % 360)
            pop_p = float(np.exp(-n * pop_R ** 2))
        else:
            pop_R = pop_dir = pop_p = np.nan
        return pd.Series({
            "n": len(d),
            "R_median": d["R"].median(),
            "R_q1": d["R"].quantile(0.25),
            "R_q3": d["R"].quantile(0.75),
            "R_corr_median": d["R_corr"].median(),
            "R_raw_median": d["R_raw"].median(),
            "R_geom_median": d["R_geom"].median(),
            "R_excess_median": d["R_excess"].median(),
            "R_shape_median": d["R_shape"].median(),
            "A1_over_A0_median": d["A1_over_A0"].median(),
            "frac_significant": d["significant"].mean(),
            "pop_R": pop_R, "pop_dir_deg": pop_dir, "pop_p": pop_p,
            "coverage_min": d["coverage"].min(),
        })

    out = (resultants.groupby(list(by), sort=True)
           .apply(agg, include_groups=False).reset_index())
    return out.round(4)


def plot_asymmetry_score_summary(resultants: pd.DataFrame, config,
                                 score: str = "R",
                                 save_as: str = "asymmetry_score_summary"):
    """Three panels: score vs time per shell, the geometry floor, and lab-frame
    coherence across nuclei."""
    if resultants is None or resultants.empty:
        print("No data to plot."); return

    tab = build_asymmetry_score_table(resultants)
    shells = sorted(resultants["shell_idx"].unique())
    labels = (resultants.drop_duplicates("shell_idx")
              .set_index("shell_idx")["shell_label"].to_dict())
    cmap = LinearSegmentedColormap.from_list(
        "shells", ["#7FD1B9", "#2A9D8F", "#3D5A99", "#1B1B2F"])
    colors = {sh: cmap(i / max(len(shells) - 1, 1)) for i, sh in enumerate(shells)}

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

    ax = axes[0]
    for sh in shells:
        d = (resultants[resultants.shell_idx == sh]
             .groupby("t")[score].agg(["median", lambda s: s.quantile(.25),
                                       lambda s: s.quantile(.75)]))
        d.columns = ["med", "q1", "q3"]
        ax.fill_between(d.index, d.q1, d.q3, color=colors[sh], alpha=0.15, lw=0)
        ax.plot(d.index, d.med, color=colors[sh], lw=2, marker="o", ms=4,
                label=f"{labels[sh]} \u00b5m")
    med_neff = resultants["n_eff"].median()
    if np.isfinite(med_neff):
        ax.axhline(R_critical(med_neff), color="k", lw=0.9, ls="--", alpha=0.6,
                   label="single-nucleus noise floor (p=0.05)")
    ax.set_xlabel("timepoint"); ax.set_ylabel("normalised asymmetry score $R$")
    ax.set_title("(a) Asymmetry score by shell")
    ax.legend(fontsize=7, title="shell", title_fontsize=7)
    ax.set_ylim(0, 1); ax.spines[["top", "right"]].set_visible(False)

    ax = axes[1]
    for sh in shells:
        d = resultants[resultants.shell_idx == sh].groupby("t")[["R", "R_geom"]].median()
        ax.plot(d.index, d.R, color=colors[sh], lw=2, marker="o", ms=4)
        ax.plot(d.index, d.R_geom, color=colors[sh], lw=1.2, ls=":", marker="^", ms=3)
    ax.plot([], [], color="0.3", lw=2, label="$R$ (signal)")
    ax.plot([], [], color="0.3", lw=1.2, ls=":", label="$R_{geom}$ (sampling)")
    ax.set_xlabel("timepoint"); ax.set_ylabel("$R$")
    ax.set_title("(b) Signal vs the geometry floor")
    ax.legend(fontsize=8); ax.set_ylim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)

    ax = axes[2]
    for sh in shells:
        d = tab[tab.shell_label == labels[sh]].sort_values("t")
        ax.plot(d.t, d.pop_R, color=colors[sh], lw=2, marker="o", ms=4)
        ax.plot(d.t, np.sqrt(-np.log(0.05) / d["n"].replace(0, np.nan)),
                color=colors[sh], lw=0.8, ls="--", alpha=0.6)
    ax.set_xlabel("timepoint"); ax.set_ylabel("population resultant of $\\phi$")
    ax.set_title("(c) Do nuclei agree on a lab direction?")
    ax.set_ylim(0, 1)
    ax.plot([], [], color="0.3", lw=0.8, ls="--", label="p = 0.05")
    ax.legend(fontsize=8); ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Normalised asymmetry score", fontsize=13)
    fig.tight_layout()
    save_figure(fig, save_as, config)
    plt.show()
    return tab

### 37e. Aggregate across the dataset, direction discarded

Three views, none of which assume a common direction:

**(a) and (b)** pool the score over every nucleus × every frame per shell —
magnitude only, direction thrown away — against the geometry floor and the
uniform-shell noise line.

**(c) the alignment-averaged profile.** Each nucleus is rotated so its *own*
asymmetry axis sits at 0°, then all are averaged. A population whose bright
sides point in random directions still produces a lobe here, provided each
nucleus is individually one-sided — that is exactly the direction-agnostic
statement, and it also shows the *shape* of the asymmetry: a broad cosine means
gradual enrichment, a narrow spike means a delivery hotspot.

**The null is not optional here.** Choosing the rotation from the same data you
then average is circular: pure noise, aligned to its own noisiest angle,
produces a lobe. `aligned_mean_profile` repeats the entire procedure on
θ-permuted profiles — same values, spatial structure destroyed — `n_perm` times,
building a distribution of *mean* profiles, and returns its 95th percentile.
Comparing an averaged observation against unaveraged null draws understates the
null by √n_units. The test is read at lag 0 only, since both curves peak there
by construction.

In [ ]:
def aligned_mean_profile(binned: pd.DataFrame, resultants: pd.DataFrame,
                         shell_idx: int, n_perm: int = 200,
                         min_coverage: float = 1.0, t_min: Optional[int] = None,
                         seed: int = 0) -> Optional[Dict]:
    """Rotate every nucleus so its own asymmetry axis is at 0, then average.

    Direction-agnostic by construction: no lab frame is assumed, so a
    population whose bright sides point in random directions still produces a
    lobe here as long as each nucleus is individually one-sided.

    The catch, and why the null matters: the rotation is chosen from the same
    data that is then averaged. Pure noise, aligned to its own noisiest angle,
    produces a lobe. The null repeats the ENTIRE procedure -- permute each
    unit's theta bins (same values, spatial structure destroyed), realign each
    to its own permuted axis, average across units -- and does that n_perm
    times to build a distribution of MEAN profiles. The envelope returned is
    the 95th percentile across those replicate means, so it is on the same
    footing as the observed mean. Comparing an averaged observation against
    unaveraged null draws understates the null by a factor of sqrt(n_units).
    """
    if binned is None or binned.empty or resultants is None or resultants.empty:
        return None
    rng = np.random.default_rng(seed)
    n_bins = binned.attrs.get("n_theta_bins", N_THETA_BINS)
    dtheta = 2 * np.pi / n_bins
    th = (np.arange(n_bins) + 0.5) * dtheta

    res = resultants[(resultants.shell_idx == shell_idx)
                     & (resultants.coverage >= min_coverage)]
    if t_min is not None:
        res = res[res.t >= t_min]
    if res.empty:
        return None
    lut = res.set_index(["t", "track_id"])

    units, prof = [], []
    for (t, tid), d in binned[binned.shell_idx == shell_idx].groupby(["t", "track_id"]):
        if (t, tid) not in lut.index:
            continue
        row = lut.loc[(t, tid)]
        d = d.sort_values("theta_bin")
        if not (d["n_samples"].to_numpy() > 0).all():
            continue
        I = d["I_mean"].to_numpy(dtype=float)
        bg = float(row["I_bg"])
        s = I - bg
        base = s.mean()
        if not np.isfinite(base) or base <= 0 or not np.isfinite(row["phi_rad"]):
            continue
        p = s / base                                  # normalised, unitless
        units.append(p)
        prof.append(np.roll(p, -int(np.round(float(row["phi_rad"]) / dtheta))))

    if len(prof) < 3:
        return None
    prof = np.vstack(prof)

    # null: n_perm replicate MEAN profiles, each built the same way as the observation
    null_means = np.empty((n_perm, n_bins))
    for b in range(n_perm):
        acc = np.empty((len(units), n_bins))
        for i, p in enumerate(units):
            q = rng.permutation(p)
            _, phi_q, _ = _resultant(th, np.clip(q, 0, None))
            sh_q = 0 if not np.isfinite(phi_q) else int(np.round(phi_q / dtheta))
            acc[i] = np.roll(q, -sh_q)
        null_means[b] = acc.mean(axis=0)

    lag_deg = (np.degrees(th) + 180) % 360 - 180
    order = np.argsort(lag_deg)
    obs = prof.mean(axis=0)
    null_hi = np.percentile(null_means, 95, axis=0)

    # Score at lag 0 only. Both the observation and the null are aligned so
    # their maximum sits at lag 0 by construction, so that is the one bin the
    # test is about; taking a max over all 36 lags instead re-introduces a
    # multiple-comparisons problem and lets a uniform field clear its own null.
    lag0 = int(np.argmin(np.abs(lag_deg)))
    return {
        "lag_deg": lag_deg[order],
        "mean": obs[order],
        "sem": (prof.std(axis=0, ddof=1) / np.sqrt(len(prof)))[order],
        "null_hi": null_hi[order],
        "null_mean": null_means.mean(axis=0)[order],
        "n_units": len(prof),
        "obs_at_axis": float(obs[lag0]),
        "null_at_axis": float(null_hi[lag0]),
        "peak_excess": float(obs[lag0] - null_hi[lag0]),
    }


def plot_aggregate_asymmetry(binned: pd.DataFrame, resultants: pd.DataFrame, config,
                             score: str = "R", n_perm: int = 200,
                             save_as: str = "aggregate_asymmetry"):
    """Whole-dataset asymmetry with direction discarded.

    (a) distribution of the per-nucleus score by shell, against the geometry
        floor and the uniform-shell null -- magnitude only, no direction.
    (b) ECDF of the same, so the whole distribution is visible rather than a
        median.
    (c) the alignment-averaged profile: what the asymmetry LOOKS like once
        every nucleus is rotated onto its own axis, with the permutation null.
    """
    if resultants is None or resultants.empty:
        print("No data to plot."); return

    shells = sorted(resultants["shell_idx"].unique())
    labels = (resultants.drop_duplicates("shell_idx")
              .set_index("shell_idx")["shell_label"].to_dict())
    cmap = LinearSegmentedColormap.from_list(
        "shells", ["#7FD1B9", "#2A9D8F", "#3D5A99", "#1B1B2F"])
    colors = {sh: cmap(i / max(len(shells) - 1, 1)) for i, sh in enumerate(shells)}

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))

    # (a) violins of R_corr, with the geometry floor overlaid
    ax = axes[0]
    data = [resultants.loc[resultants.shell_idx == sh, score].dropna().to_numpy()
            for sh in shells]
    pos = np.arange(len(shells))
    keep = [i for i, d in enumerate(data) if d.size > 1]
    if keep:
        vp = ax.violinplot([data[i] for i in keep], positions=pos[keep],
                           showmedians=True, widths=0.75)
        for i, body in zip(keep, vp["bodies"]):
            body.set_facecolor(colors[shells[i]]); body.set_alpha(0.55)
        for k in ("cmins", "cmaxes", "cbars", "cmedians"):
            if k in vp:
                vp[k].set_color("0.25"); vp[k].set_linewidth(1.0)
    for i, sh in enumerate(shells):
        g = resultants.loc[resultants.shell_idx == sh, "R_geom"].median()
        ax.hlines(g, i - 0.38, i + 0.38, color="#D7263D", lw=1.8, ls=":", zorder=5)
    ax.plot([], [], color="#D7263D", lw=1.8, ls=":", label="median $R_{geom}$")
    med_neff = resultants["n_eff"].median()
    if np.isfinite(med_neff):
        ax.axhline(R_critical(med_neff), color="k", lw=0.9, ls="--", alpha=0.6,
                   label="uniform-shell $R$ at p=0.05")
    ax.set_xticks(pos)
    ax.set_xticklabels([f"{labels[sh]}" for sh in shells], fontsize=8)
    ax.set_xlabel("shell (\u00b5m from nuclear surface)")
    ax.set_ylabel("asymmetry score $R$")
    ax.set_title("(a) Score distribution, all nuclei \u00d7 all frames")
    ax.set_ylim(0, 1); ax.legend(fontsize=7.5)
    ax.spines[["top", "right"]].set_visible(False)

    # (b) ECDF
    ax = axes[1]
    for sh in shells:
        d = np.sort(resultants.loc[resultants.shell_idx == sh, score].dropna().to_numpy())
        if d.size == 0:
            continue
        ax.step(d, np.arange(1, d.size + 1) / d.size, where="post",
                color=colors[sh], lw=1.8, label=f"{labels[sh]}  (n={d.size})")
        g = np.sort(resultants.loc[resultants.shell_idx == sh, "R_geom"].dropna().to_numpy())
        if g.size:
            ax.step(g, np.arange(1, g.size + 1) / g.size, where="post",
                    color=colors[sh], lw=1.0, ls=":", alpha=0.8)
    ax.set_xlabel("asymmetry score $R$"); ax.set_ylabel("cumulative fraction")
    ax.set_title("(b) ECDF (dotted = $R_{geom}$)")
    ax.set_xlim(0, 1); ax.legend(fontsize=7)
    ax.spines[["top", "right"]].set_visible(False)

    # (c) alignment-averaged profile
    ax = axes[2]
    any_prof = False
    for sh in shells:
        pr = aligned_mean_profile(binned, resultants, sh, n_perm=n_perm)
        if pr is None:
            continue
        any_prof = True
        ax.plot(pr["lag_deg"], pr["mean"], color=colors[sh], lw=2,
                label=f"{labels[sh]}  (n={pr['n_units']})")
        ax.fill_between(pr["lag_deg"], pr["mean"] - 1.96 * pr["sem"],
                        pr["mean"] + 1.96 * pr["sem"], color=colors[sh], alpha=0.18, lw=0)
        if pr["null_hi"] is not None:
            ax.plot(pr["lag_deg"], pr["null_hi"], color=colors[sh], lw=0.9,
                    ls="--", alpha=0.75)
    if any_prof:
        ax.axhline(1.0, color="0.4", lw=0.8)
        ax.set_xlim(-180, 180); ax.set_xticks([-180, -90, 0, 90, 180])
        ax.plot([], [], color="0.3", lw=0.9, ls="--", label="permutation null (95th pct)")
        ax.legend(fontsize=7)
    else:
        ax.text(0.5, 0.5, "no fully-covered shell units", ha="center", va="center",
                transform=ax.transAxes, fontsize=9, color="0.4")
    ax.set_xlabel("angle from each nucleus's own asymmetry axis (\u00b0)")
    ax.set_ylabel("intensity / shell mean")
    ax.set_title("(c) Alignment-averaged profile")
    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Aggregate membrane asymmetry \u2014 direction discarded", fontsize=13)
    fig.tight_layout()
    save_figure(fig, save_as, config)
    plt.show()

### 37f. Run the shell-resolved analysis

In [ ]:
# ── Shell-resolved asymmetry: full run ────────────────────────────────────
radial_n_df = add_wall_normalised_radius(radial_df, cfg)

# One background floor for the whole dataset — see 37b for why it must not
# come from the profile being scored.
BG = estimate_background(radial_n_df, q=0.05)
print(f"background floor: {BG:.1f} (5th percentile of all sweep samples)")

binned_df = bin_shell_angle(radial_n_df, cfg,
                            shell_edges_um=DEFAULT_SHELL_EDGES_UM,
                            n_theta_bins=N_THETA_BINS)
shell_res_df = compute_shell_resultants(binned_df, background=BG,
                                        min_coverage=MIN_SHELL_COVERAGE)
print(shell_res_df.shape)

# 1. the rose itself, for the longest-lived nucleus
_longest = int(tracked_df.groupby("track_id")["t"].nunique().idxmax())
plot_shell_rose_facets(binned_df, shell_res_df, cfg, track_id=_longest,
                       value="mean_bgsub", radius_scale="sqrt", share_r="row",
                       save_as=f"shell_rose_track{_longest}")

# 2. normalised asymmetry score — table + graph
shell_score_table = plot_asymmetry_score_summary(shell_res_df, cfg)
display(shell_score_table)

# 3. aggregate across the whole dataset, direction discarded
plot_aggregate_asymmetry(binned_df, shell_res_df, cfg, n_perm=200)

# exports
shell_res_df.to_csv(cfg.exports_dir / "shell_asymmetry_per_nucleus.csv", index=False)
shell_score_table.to_csv(cfg.exports_dir / "shell_asymmetry_score_table.csv", index=False)
binned_df.to_pickle(cfg.analysis_dir / "shell_angle_binned.pkl")
print("figures ->", figures_dir(cfg))

In [ ]:
# ── Sanity read of the geometry control ───────────────────────────────────
# Any shell where R does not clear R_geom is reporting droplet placement,
# not membrane. Read this before interpreting the rose.
chk = (shell_res_df.groupby("shell_label")
       .agg(n=("R", "size"),
            R=("R", "median"),
            R_geom=("R_geom", "median"),
            excess=("R_excess", "median"),
            coverage=("coverage", "min"),
            frac_sig=("significant", "mean"))
       .round(3))
display(chk)

flagged = chk[chk.R_geom >= 0.5 * chk.R]
if len(flagged):
    print("\nShells where sampling geometry accounts for half or more of the "
          "measured asymmetry — do not quote these as membrane asymmetry:")
    display(flagged)
else:
    print("\nNo shell has R_geom within 2x of R.")

In [ ]:
PIPELINE_VERSION = "v18.1"  # matches this notebook; bump manually per release
DB_EXPORT_DIR = cfg.output_dir / "database_export"
DB_EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def _fmt_slice_id(z: int) -> str:
    """Match the S#### convention used in the schema workbook."""
    return f"S{int(z):04d}"


def _fmt_centroid_xyz(x: float, y: float, z: float) -> str:
    return f"({x:.1f}, {y:.1f}, {z:.1f})"


In [ ]:
def build_nuclei_table(halo_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """One row per Nucleus_ID (track_id) per Time_Frame — analysis-ready table."""
    if halo_df.empty:
        return pd.DataFrame(columns=[
            "Nucleus_ID", "Droplet_ID", "FOV_ID", "Experiment_ID",
            "Time_Frame", "Time_Real_Minutes", "Stage_Classification",
            "Selected_Slice_ID", "Centroid_XYZ_px",
            "Cross_Sectional_Area_um2", "Volume_3D_um3", "NC_Ratio",
            "Segmentation_Pipeline_Version", "Model_Version",
            "QC_Flag", "QC_Notes", "Source_File_Path",
        ])

    experiment_id = Path(cfg.input_image_name).stem
    rows = []
    for row in halo_df.itertuples(index=False):
        qc_pass = bool(getattr(row, "valid_single_nucleus", True))
        rows.append({
            "Nucleus_ID":                    int(row.track_id),
            "Droplet_ID":                    "",  # not tracked as a distinct ID by this pipeline
            "FOV_ID":                        "FOV01",  # single stitched image per run
            "Experiment_ID":                 experiment_id,
            "Time_Frame":                    int(row.t),
            "Time_Real_Minutes":             float(getattr(row, "true_time_min", np.nan)),
            "Stage_Classification":          "",  # not computed by this pipeline
            "Selected_Slice_ID":             _fmt_slice_id(row.z),
            "Centroid_XYZ_px":               _fmt_centroid_xyz(row.centroid_x_px, row.centroid_y_px, row.z),
            "Cross_Sectional_Area_um2":      float(getattr(row, "nucleus_area_um2", np.nan)),
            "Volume_3D_um3":                 np.nan,  # 2D best-Z pipeline only
            "NC_Ratio":                      float(getattr(row, "nc_ratio", np.nan)),
            "Segmentation_Pipeline_Version": PIPELINE_VERSION,
            "Model_Version":                 cfg.model_name,
            "QC_Flag":                       "PASS" if qc_pass else "FAIL",
            "QC_Notes":                      "" if qc_pass else "excluded: multiple nuclei within exclusion radius",
            "Source_File_Path":              cfg.input_image_name,
        })
    return pd.DataFrame(rows)


nuclei_export_df = build_nuclei_table(halo_df, cfg)
print(nuclei_export_df.shape)
display(nuclei_export_df.head())


In [ ]:
def build_zstack_table(grouped_z_df: pd.DataFrame, best_z_df: pd.DataFrame,
                       tracked_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """One row per Z-plane per nucleus per timepoint — full sweep behind the best-Z pick."""
    cols = ["Nucleus_ID", "Time_Frame", "Slice_ID", "Centroid_XYZ_px",
            "Cross_Sectional_Area_um2", "Is_Selected_Max"]
    if grouped_z_df.empty or best_z_df.empty or tracked_df.empty:
        return pd.DataFrame(columns=cols)

    track_lookup = tracked_df[["t", "nucleus_3d_id", "track_id"]].drop_duplicates()
    best_z_lookup = best_z_df[["t", "nucleus_3d_id", "z"]].rename(columns={"z": "best_z"})

    merged = grouped_z_df.merge(track_lookup, on=["t", "nucleus_3d_id"], how="inner")
    merged = merged.merge(best_z_lookup, on=["t", "nucleus_3d_id"], how="left")

    rows = []
    for row in merged.itertuples(index=False):
        area_um2 = float(row.area_px) * (cfg.pixel_size_um ** 2)
        rows.append({
            "Nucleus_ID":               int(row.track_id),
            "Time_Frame":               int(row.t),
            "Slice_ID":                 _fmt_slice_id(row.z),
            "Centroid_XYZ_px":          _fmt_centroid_xyz(row.centroid_x_px, row.centroid_y_px, row.z),
            "Cross_Sectional_Area_um2": area_um2,
            "Is_Selected_Max":          bool(row.z == row.best_z),
        })
    return pd.DataFrame(rows, columns=cols)


zstack_export_df = build_zstack_table(grouped_z_df, best_z_df, tracked_df, cfg)
print(zstack_export_df.shape)
display(zstack_export_df.head())


In [ ]:
def build_raw_intensities_table(halo_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """Per-halo, per-channel integrated intensities at the selected Z-slice.

    NOTE: only the Mcherry/NLS channel is currently measured by
    measure_fiji_equivalent_halos (Section 18). NPC and Membrane columns,
    and all Background_* columns, are written as NaN until that function
    is extended to loop over channels.
    """
    cols = [
        "Nucleus_ID", "Time_Frame",
        "Halo1_Mcherry", "Halo2_Mcherry", "Halo3_Mcherry", "Halo4_Mcherry",
        "Halo1_NPC", "Halo2_NPC", "Halo3_NPC", "Halo4_NPC",
        "Halo1_Membrane", "Halo2_Membrane", "Halo3_Membrane", "Halo4_Membrane",
        "Background_Mcherry", "Background_NPC", "Background_Membrane",
    ]
    if halo_df.empty:
        return pd.DataFrame(columns=cols)

    rows = []
    for row in halo_df.itertuples(index=False):
        rd = row._asdict()
        rec = {
            "Nucleus_ID": int(rd["track_id"]),
            "Time_Frame": int(rd["t"]),
        }
        for i in range(1, cfg.n_halos + 1):
            rec[f"Halo{i}_Mcherry"] = rd.get(f"ring_{i}_intden", np.nan)
            rec[f"Halo{i}_NPC"] = np.nan       # not measured yet — see docstring
            rec[f"Halo{i}_Membrane"] = np.nan  # not measured yet — see docstring
        rec["Background_Mcherry"] = np.nan
        rec["Background_NPC"] = np.nan
        rec["Background_Membrane"] = np.nan
        rows.append(rec)
    return pd.DataFrame(rows, columns=cols)


raw_intensities_export_df = build_raw_intensities_table(halo_df, cfg)
print(raw_intensities_export_df.shape)
display(raw_intensities_export_df.head())


In [ ]:
def build_radial_profile_table(radial_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """Long-form radial sweep -> schema RadialProfile (one row per ray sample point).

    W_Wall_Proximity_Index, Is_Ridge_Point, and Cluster_ID are written as
    placeholders (NaN / False / blank) -- wall-proximity index and ridge/DBSCAN
    clustering are not computed in v16.
    """
    cols = [
        "Nucleus_ID", "Time_Frame", "Theta_deg", "Rho_Normalized",
        "W_Wall_Proximity_Index", "Channel", "Intensity",
        "Is_Ridge_Point", "Cluster_ID",
    ]
    if radial_df is None or radial_df.empty:
        return pd.DataFrame(columns=cols)

    channel_label_by_index = {
        cfg.membrane_channel_index: "Membrane",
        cfg.nuclear_channel_index:  "Mcherry",
        cfg.npc_channel_index:      "NPC",
    }
    radial_channel_label = channel_label_by_index.get(cfg.radial_channel_index, "Membrane")

    out = pd.DataFrame({
        "Nucleus_ID":             radial_df["track_id"].astype(int),
        "Time_Frame":             radial_df["t"].astype(int),
        "Theta_deg":              np.degrees(radial_df["theta_rad"].astype(float)),
        "Rho_Normalized":         radial_df["distance_norm"].astype(float),
        "W_Wall_Proximity_Index": np.nan,
        "Channel":                radial_channel_label,
        "Intensity":              radial_df["intensity"].astype(float),
        "Is_Ridge_Point":         False,
        "Cluster_ID":             "",
    })
    return out[cols]


radial_profile_export_df = build_radial_profile_table(radial_df, cfg)
print(radial_profile_export_df.shape)
display(radial_profile_export_df.head())


In [ ]:
def build_experimental_cfg_table(cfg: PipelineConfig) -> pd.DataFrame:
    experiment_id = Path(cfg.input_image_name).stem
    channel_labels = {0: "Membrane", 1: "NLS", 2: "NPC"}
    row = {
        "Experiment_ID":                 experiment_id,
        "FOV_ID":                        "FOV01",
        "Droplet_ID":                    "",
        "Date":                          datetime.utcnow().strftime("%Y-%m-%d"),
        "Pixel_Size_um":                 cfg.pixel_size_um,
        "Z_Step_um":                     cfg.z_step_um,
        "Num_Z_Planes":                  int(img_5d.shape[1]) if "img_5d" in dir() else np.nan,
        "Frame_Interval_min":            cfg.minutes_per_tile * cfg.tile_rows * cfg.tile_cols,
        "Channel0_Label":                channel_labels.get(0, ""),
        "Channel1_Label":                channel_labels.get(1, ""),
        "Channel2_Label":                channel_labels.get(2, ""),
        "Microscope":                    "",
        "Objective":                     "",
        "Operator":                      "",
        "Segmentation_Pipeline_Version": PIPELINE_VERSION,
        "Model_Version":                 cfg.model_name,
        "Raw_File_Path":                 str(cfg.input_image_path),
        "Notes":                         "",
    }
    return pd.DataFrame([row])


experimental_cfg_export_df = build_experimental_cfg_table(cfg)
display(experimental_cfg_export_df)


In [ ]:
def export_database_schema_tables(export_dir: Path) -> dict:
    tables = {
        "Nuclei":            nuclei_export_df,
        "NucleusZStack":     zstack_export_df,
        "RawIntensities":    raw_intensities_export_df,
        "RadialProfile":     radial_profile_export_df,
        "Experimental_Cfg":  experimental_cfg_export_df,
    }
    written = {}
    for name, df in tables.items():
        out_path = export_dir / f"{name}.csv"
        df.to_csv(out_path, index=False)
        written[name] = str(out_path)
        print(f"{name}: {len(df)} rows -> {out_path}")
    return written


RUN_DB_SCHEMA_EXPORT = True

if RUN_DB_SCHEMA_EXPORT:
    written_paths = export_database_schema_tables(DB_EXPORT_DIR)


In [ ]:
r = (d.assign(frac=d.nuc_px/d.drop_px)
       .groupby("t")["frac"]
       .describe(percentiles=[.5,.9,.95,.99])[["50%","90%","95%","99%","max"]])
print(r.round(3))

In [ ]:
rows = []
for (t, z), grp in b.groupby(["t", "z"]):          # drop the t>=7 filter
    plane = np.asarray(drop[t, z])
    props = {r.label: r.area for r in measure.regionprops(plane)}
    for r in grp.itertuples():
        did = plane[int(r.centroid_y_px), int(r.centroid_x_px)]
        rows.append({"t": t, "nuc_px": r.area_px,
                     "drop_px": props.get(did, np.nan),
                     "assigned": did > 0})
dall = pd.DataFrame(rows)
dall["frac"] = dall.nuc_px / dall.drop_px
print(dall.groupby("t")["frac"].describe(percentiles=[.5,.9,.95,.99])[["50%","90%","95%","99%","max"]].round(3))
print("\nunassigned (droplet_id=0) by t:")
print(dall.groupby("t")["assigned"].apply(lambda s: 1 - s.mean()).round(3))

In [ ]:
import pandas as pd

tm = pd.read_pickle(cfg.track_dir / "best_z_nuclei_timed.pkl")
tm["area_um2"] = tm.area_px * cfg.pixel_size_um**2

EARLY_MIN, EARLY_MAX = 50.0, 200.0     # true_time_min <= 20
LATE_MIN,  LATE_MAX  = 200.0, 500.0    # true_time_min >  20

early = tm.true_time_min <= 20
keep = ((early  & tm.area_um2.between(EARLY_MIN, EARLY_MAX)) |
        (~early & tm.area_um2.between(LATE_MIN,  LATE_MAX)))

tm_f = tm[keep].copy()
tm_f.to_pickle(cfg.track_dir / "best_z_nuclei_timed_filtered.pkl")

print(f"kept {len(tm_f)} / {len(tm)}  ({100*len(tm_f)/len(tm):.1f}%)")
print("\nretained by frame:")
print(pd.DataFrame({"before": tm.groupby("t").size(),
                    "after":  tm_f.groupby("t").size()}).fillna(0).astype(int))
print("\nmedian area µm² after filter:")
print(tm_f.groupby("t")["area_um2"].median().round(1))

In [ ]:
import pandas as pd

halo_df = pd.read_pickle(cfg.analysis_dir / "halo_analysis.pkl")

# recompute area from px so it always reflects the current calibration
if "nucleus_area_px" in halo_df.columns:
    halo_df["nucleus_area_um2"] = halo_df.nucleus_area_px * cfg.pixel_size_um**2

early = halo_df.true_time_min <= 20
keep = ((early  & halo_df.nucleus_area_um2.between(50, 200)) |
        (~early & halo_df.nucleus_area_um2.between(200, 500)))
hf = halo_df[keep].copy()

print(f"kept {len(hf)} / {len(halo_df)} ({100*len(hf)/len(halo_df):.1f}%)")
print(pd.DataFrame({"before": halo_df.groupby("t").size(),
                    "after":  hf.groupby("t").size()}).fillna(0).astype(int))

plot_area_vs_time(hf)
plot_nc_vs_time(hf)
plot_largest_cross_sectional_area_vs_time(hf, cfg)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, col, lab in [(ax[0], "nucleus_area_um2", "Nuclear area (µm²)"),
                    (ax[1], "nc_ratio_fraction", "N / (N + C)")]:
    g = hf.groupby("true_time_min")[col]
    med, lo, hi = g.median(), g.quantile(.25), g.quantile(.75)
    a.fill_between(med.index, lo, hi, alpha=.25)
    a.plot(med.index, med, lw=2)
    a.set_xlabel("True acquisition time (min)"); a.set_ylabel(lab)
a.figure.suptitle("Median ± IQR, filtered nuclei"); plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd

# nuclei per frame and per true acquisition minute
counts = pd.DataFrame({
    "rows":          hf.groupby("t").size(),
    "unique_tracks": hf.groupby("t")["track_id"].nunique(),
    "unique_droplets": hf.groupby("t")["droplet_id"].nunique() if "droplet_id" in hf else None,
}).fillna(0).astype(int)
counts["dupes"] = counts["rows"] - counts["unique_tracks"]
print(counts)

print("\nper true acquisition minute:")
print(hf.groupby("true_time_min")["track_id"].nunique())

# how persistent are tracks across the timecourse?
span = hf.groupby("track_id")["t"].nunique()
print("\ntimepoints per track:")
print(span.value_counts().sort_index())
print(f"\ntotal unique nuclei (any t): {hf.track_id.nunique()}")
print(f"tracks present at all 10 t:   {(span == hf.t.nunique()).sum()}")

In [ ]:
import numpy as np, tifffile as tiff
from skimage import measure
drop = tiff.memmap(cfg.droplet_instance_hyperstack_path, mode="r")
for t in range(drop.shape[0]):
    z = int(hf[hf.t == t]["z"].median()) if (hf.t == t).any() else drop.shape[1]//2
    n = len([r for r in measure.regionprops(np.asarray(drop[t, z])) if r.area > 5000])
    print(f"t={t}  droplets={n:4d}  nuclei_kept={int((hf.t==t).sum()):4d}")

In [ ]:
import numpy as np, pandas as pd, tifffile as tiff
from skimage import measure

prob = tiff.memmap(cfg.segmentation_probability_hyperstack_path, mode="r")  # (T,Z,C,Y,X)
NUC = cfg.nucleus_class_index

tm = pd.read_pickle(cfg.track_dir / "best_z_nuclei_timed.pkl")
tm["area_um2"] = tm.area_px * cfg.pixel_size_um**2

recs = []
for (t, z), grp in tm.groupby(["t", "z"]):
    lab = np.load(cfg.seg_dir / f"nuclear_mask_t{int(t):03d}_z{int(z):03d}.npy")
    lab = measure.label(lab > 0) if lab.dtype == bool else lab
    pplane = np.asarray(prob[int(t), int(z), NUC]).astype(np.float32)
    tbl = pd.DataFrame(measure.regionprops_table(
        lab, intensity_image=pplane,
        properties=("label", "mean_intensity", "area")))
    lut = dict(zip(tbl.label, tbl.mean_intensity))
    for r in grp.itertuples():
        l = int(lab[int(r.centroid_y_px), int(r.centroid_x_px)])
        recs.append({"t": r.t, "z": r.z, "track_id": getattr(r, "track_id", np.nan),
                     "centroid_x_px": r.centroid_x_px, "centroid_y_px": r.centroid_y_px,
                     "mean_prob": lut.get(l, np.nan)})

pr = pd.DataFrame(recs)
tm = tm.merge(pr, on=["t","z","centroid_x_px","centroid_y_px"], how="left")

print(tm.groupby("t")["mean_prob"].describe(percentiles=[.1,.25,.5,.75,.9])
        [["10%","25%","50%","75%","90%","max"]].round(3))
print("\nfraction with mean_prob > 0.9, by t:")
print(tm.groupby("t")["mean_prob"].apply(lambda s: (s > 0.9).mean()).round(3))

In [ ]:
tmp = tm[tm.mean_prob > 0.7].copy()
print(f"kept {len(tmp)}/{len(tm)}; unique nuclei/t:")
print(tmp.groupby("t")["track_id"].nunique())

plot_largest_cross_sectional_area_vs_time(tmp, cfg)

hf_p = halo_df.merge(tmp[["t","track_id"]].drop_duplicates(), on=["t","track_id"])
plot_nc_vs_time(hf_p)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

radial_df = pd.read_pickle(cfg.analysis_dir / "radial_sweep.pkl")

# --- pick the longest-lived track (or set TRACK manually) ---
span = radial_df.groupby("track_id")["t"].nunique().sort_values(ascending=False)
TRACK = int(span.index[0])
print(f"track {TRACK}: present at t = {sorted(radial_df[radial_df.track_id==TRACK].t.unique())}")

sub = radial_df[radial_df.track_id == TRACK].copy()

# --- radial axis: normalised (nucleus edge -> droplet edge) ---
RBINS = 60
sub["rbin"] = pd.cut(sub.distance_norm, np.linspace(0, 1, RBINS + 1), labels=False)

ts = sorted(sub.t.unique())
grids = {t: (sub[sub.t == t]
             .pivot_table(index="theta_index", columns="rbin",
                          values="intensity", aggfunc="mean")
             .reindex(index=range(cfg.radial_n_angles), columns=range(RBINS)))
         for t in ts}

allv = np.concatenate([g.to_numpy().ravel() for g in grids.values()])
vmin, vmax = np.nanpercentile(allv, [2, 98])

ncol = min(5, len(ts)); nrow = int(np.ceil(len(ts) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1*ncol, 2.9*nrow),
                         squeeze=False, constrained_layout=True)
for ax, t in zip(axes.ravel(), ts):
    im = ax.imshow(grids[t].to_numpy(), aspect="auto", origin="lower",
                   extent=[0, 1, 0, 360], vmin=vmin, vmax=vmax, cmap="inferno")
    ax.set_title(f"t={t}", fontsize=9)
    ax.set_xlabel("r (nucleus edge → droplet edge)", fontsize=7)
    ax.set_ylabel("θ (deg)", fontsize=7)
    ax.tick_params(labelsize=7)
for ax in axes.ravel()[len(ts):]:
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.7, label="membrane intensity")
fig.suptitle(f"Radial membrane profile — track {TRACK}", fontsize=11)
plt.show()

## 38. Recovered Vulcan post-segmentation audits

Run this section against an existing completed run after configuring `cfg` with
that run's paths. No segmentation rerun is required. These diagnostics do not
modify segmentation masks or replace production analysis tables.

Run the cells in order. Audit functions save new timestamped QC outputs.
This population does NOT enforce proximity or multi-nucleus droplet exclusion.
Reviewed T0 examples were cytoplasm artifacts; timepoint alone is not a rejection rule.
Real nuclei in multi-nucleus droplets remain included pending a separate implementation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if 'cfg' not in globals():
    raise RuntimeError('Configure cfg for the existing segmentation run first.')
print('Loading saved objects from:', cfg.obj_dir)
# Deliberately reload from this run, rather than reuse possibly stale kernel tables.
objects_df = pd.read_pickle(cfg.obj_dir / 'plane_objects.pkl')
grouped_z_df = pd.read_pickle(cfg.obj_dir / 'grouped_z_objects.pkl')
best_z_df = pd.read_pickle(cfg.obj_dir / 'best_z_nuclei.pkl')
print('Plane objects:', len(objects_df), '| linked rows:', len(grouped_z_df))


### 38a. Enrichment and solidity filter

Training-like thresholds: enrichment >= 1.20, solidity >= 0.80, at least 50
reference-ring pixels. Support here is the segmented droplet, not the training
circle. Unassessable references are rejected by the gate, not confirmed artifacts.
Definitions are embedded below so this notebook does not depend on external scripts.


In [ ]:
"""Drop-in enrichment/solidity gate for the v18.1 segmented instance dataset.

Execute after segmentation; creates filtered TABLES, not replacement mask files.
Training measurement formulas are preserved. Support comes from segmented
droplet interiors instead of training's fitted/eroded droplet circles.
"""
from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from skimage import filters, measure, morphology
from IPython.display import display


@dataclass(frozen=True)
class ArtifactGateSettings:
    min_enrichment: float = 1.20
    min_solidity: float = 0.80
    ring_gap_um: float = 0.5
    ring_width_um: float = 2.0
    min_ring_pixels: int = 50


def measure_segment_gate(image, candidate, support, raw_foreground, pixel_size_um,
                         settings=ArtifactGateSettings(), solidity=None):
    """Training formula; masks share a crop padded by the outer ring radius."""
    gap=max(1,int(np.ceil(settings.ring_gap_um/pixel_size_um)))
    outer=gap+max(1,int(np.ceil(settings.ring_width_um/pixel_size_um)))
    ring=(morphology.binary_dilation(candidate,morphology.disk(outer)) &
          ~morphology.binary_dilation(candidate,morphology.disk(gap)) &
          support & ~raw_foreground)
    nr=int(ring.sum())
    inside=float(np.mean(image[candidate],dtype=np.float64)) if candidate.any() else np.nan
    outside=float(np.mean(image[ring],dtype=np.float64)) if nr else np.nan
    valid=nr>=settings.min_ring_pixels and np.isfinite(outside) and outside>0 and np.isfinite(inside)
    enrichment=inside/outside if valid else np.nan
    if solidity is None:
        regions=measure.regionprops(candidate.astype(np.uint8))
        solidity=float(regions[0].solidity) if regions else np.nan
    reasons=[]
    if not valid: reasons.append('unassessable_cytoplasm')
    elif enrichment<settings.min_enrichment: reasons.append('enrichment')
    if not np.isfinite(solidity) or solidity<settings.min_solidity: reasons.append('solidity')
    return dict(enrichment=enrichment,solidity=solidity,nucleus_mean=inside,
                cytoplasm_mean=outside,ring_pixels=nr,reference_ok=bool(valid),
                gate_pass=not reasons,gate_reason='keep' if not reasons else ' + '.join(reasons)),ring


def audit_segmented_artifacts(cfg, image=None, objects=None, best_z=None,
                             settings=ArtifactGateSettings(), save=True, examples_per_class=4):
    if cfg.pixel_size_um<=0: raise ValueError('Positive pixel size is required')
    if image is None: image=tifffile.memmap(str(cfg.input_image_path),mode='r')
    nuclei=tifffile.memmap(str(cfg.nucleus_instance_hyperstack_path),mode='r')
    droplets=tifffile.memmap(str(cfg.droplet_instance_hyperstack_path),mode='r')
    if nuclei.shape!=droplets.shape or nuclei.shape!=image.shape[:2]+image.shape[-2:]:
        raise ValueError('Raw image and instance-stack shapes disagree')
    index=pd.read_pickle(cfg.segmentation_index_path)
    included=index['included'].astype(str).str.lower().isin(['true','1'])
    planes=index[included & (index.z>=cfg.focus_min_z)].sort_values(['t','z'])
    if planes.duplicated(['t','z']).any(): raise ValueError('Duplicate plane index')
    if objects is None:
        p=cfg.obj_dir/'plane_objects.pkl'
        if p.exists(): objects=pd.read_pickle(p)
    if best_z is None:
        p=cfg.obj_dir/'best_z_nuclei.pkl'
        if p.exists(): best_z=pd.read_pickle(p)
    gap=max(1,int(np.ceil(settings.ring_gap_um/cfg.pixel_size_um)))
    outer=gap+max(1,int(np.ceil(settings.ring_width_um/cfg.pixel_size_um)))
    rows=[]; examples={True:[],False:[]}; counts={True:0,False:0}
    rng=np.random.default_rng(42)
    for t, sub in planes.groupby('t',sort=True):
        for z in sub.z:
            t,z=int(t),int(z)
            lab=np.asarray(nuclei[t,z]); drop=np.asarray(droplets[t,z])
            raw=np.asarray(image[t,z,cfg.nuclear_channel_index])
            H,W=lab.shape
            drop_regions={r.label:r for r in measure.regionprops(drop)}
            thresholds={}
            for r in measure.regionprops(lab):
                a,b,c,d=r.bbox
                local=lab[a:c,b:d]==r.label
                ids=drop[a:c,b:d][local]; valid_ids=ids[ids>0]
                parent=int(np.bincount(valid_ids).argmax()) if valid_ids.size else 0
                if parent and parent not in thresholds:
                    dr=drop_regions[parent]; sl=dr.slice
                    values=raw[sl][drop[sl]==parent]
                    thresholds[parent]=(float(filters.threshold_otsu(values))
                        if values.size and values.max()>values.min() else np.nan)
                threshold=thresholds.get(parent,np.nan)
                a,b,c,d=max(0,a-outer),max(0,b-outer),min(H,c+outer),min(W,d+outer)
                sl=(slice(a,c),slice(b,d))
                candidate=lab[sl]==r.label
                support=(drop[sl]==parent) if parent else np.zeros(candidate.shape,bool)
                crop=raw[sl]
                foreground=(crop>threshold)&support if np.isfinite(threshold) else np.zeros_like(support)
                rec,ring=measure_segment_gate(crop,candidate,support,foreground,cfg.pixel_size_um,
                                               settings,float(r.solidity))
                if not parent:
                    rec['gate_pass']=False;rec['gate_reason']='no_parent_droplet'
                rec.update(t=t,z=z,label=int(r.label),parent_droplet=parent,
                    area_um2=float(r.area*cfg.pixel_size_um**2),
                    parent_overlap_fraction=float((ids==parent).sum()/r.area) if parent else 0.,
                    otsu_threshold=threshold,centroid_y_px=float(r.centroid[0]),centroid_x_px=float(r.centroid[1]))
                rows.append(rec)
                key=rec['gate_pass']
                # Deterministic reservoir sampling spans all times/planes without keeping full images.
                counts[key]+=1
                if len(examples[key])<examples_per_class:
                    examples[key].append((rec,crop.copy(),candidate.copy(),ring.copy()))
                elif examples_per_class:
                    slot=int(rng.integers(counts[key]))
                    if slot<examples_per_class:
                        examples[key][slot]=(rec,crop.copy(),candidate.copy(),ring.copy())
        print('Artifact audit T%d complete: %d cumulative plane instances' % (t,len(rows)),flush=True)
    result=pd.DataFrame(rows)
    if result.empty: raise ValueError('No segmented instances on included planes')
    summary=result.groupby('t').agg(total=('gate_pass','size'),accepted=('gate_pass','sum'),
        unassessable=('reference_ok',lambda s:int((~s).sum())))
    summary['rejected']=summary.total-summary.accepted
    summary['rejected_fraction']=summary.rejected/summary.total
    display(summary.round(3))
    display(pd.crosstab(result.t,result.gate_reason))
    def attach(table):
        if table is None: return None
        if 'repair_status' in table and table.repair_status.eq('repaired').any():
            raise ValueError('Supply unrepaired rows: this audit measures the saved model instances')
        cols=['t','z','label','enrichment','solidity','ring_pixels','reference_ok','gate_pass','gate_reason']
        merged=table.merge(result[cols],on=['t','z','label'],how='left',validate='many_to_one',indicator=True)
        if not merged['_merge'].eq('both').all():
            raise ValueError('Some supplied object rows do not match the audited plane instances')
        return merged.drop(columns='_merge')
    annotated=attach(objects)
    best=attach(best_z)
    accepted=result[result.gate_pass].copy()
    rejected=result[~result.gate_pass].copy()
    filtered_objects=annotated[annotated.gate_pass].copy() if annotated is not None else accepted.copy()
    filtered_best=best[best.gate_pass].copy() if best is not None else None
    if best is not None:
        print('Selected best-Z nuclei (no fallback to another Z):')
        display(best.groupby('t').gate_pass.agg(['size','sum','mean']).rename(
            columns={'size':'total','sum':'accepted','mean':'accepted_fraction'}).round(3))
    fig,axs=plt.subplots(1,2,figsize=(13,4))
    summary[['accepted','rejected']].plot.bar(stacked=True,ax=axs[0],color=['teal','tomato'])
    axs[0].set_title('All segmented plane instances');axs[0].set_ylabel('Count')
    for passed,color in [(True,'teal'),(False,'tomato')]:
        s=result[result.gate_pass==passed]
        axs[1].scatter(s.enrichment,s.solidity,s=8,alpha=.3,c=color,label='accepted' if passed else 'rejected')
    axs[1].axvline(settings.min_enrichment,c='black',ls='--')
    axs[1].axhline(settings.min_solidity,c='black',ls='--')
    axs[1].set(xlabel='Raw NLS enrichment',ylabel='Solidity',title='Gate measurements (invalid references omitted)')
    axs[1].legend();plt.tight_layout();plt.show()
    samples=examples[False]+examples[True]
    if samples:
        fig,axs=plt.subplots(int(np.ceil(len(samples)/2)),2,figsize=(12,4*np.ceil(len(samples)/2)))
        for ax,(r,crop,mask,ring) in zip(np.atleast_1d(axs).ravel(),samples):
            lo,hi=np.percentile(crop,[1,99.8]);ax.imshow(crop,cmap='gray',vmin=lo,vmax=max(hi,lo+1))
            ax.contour(mask,levels=[.5],colors=['cyan' if r['gate_pass'] else 'red'])
            if ring.any():ax.contour(ring,levels=[.5],colors=['yellow'],linewidths=.5)
            ax.set_title('T%d Z%d label %d: %s\nE=%.2f S=%.2f ring=%d' %
                (r['t'],r['z'],r['label'],r['gate_reason'],r['enrichment'],r['solidity'],r['ring_pixels']))
            ax.axis('off')
        for ax in np.atleast_1d(axs).ravel()[len(samples):]:ax.axis('off')
        plt.tight_layout();plt.show()
    output=None
    if save:
        output=cfg.qc_dir/('enrichment_solidity_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        output.mkdir(parents=True,exist_ok=False)
        result.to_csv(output/'all_instances.csv',index=False)
        summary.to_csv(output/'summary_by_t.csv')
        filtered_objects.to_pickle(output/'accepted_plane_objects.pkl')
        if filtered_best is not None:filtered_best.to_pickle(output/'accepted_best_z.pkl')
        (output/'settings.json').write_text(json.dumps(dict(settings=asdict(settings),
            pixel_size_um=cfg.pixel_size_um,support='segmented droplet interior',
            candidate_source=str(cfg.nucleus_instance_hyperstack_path),
            reference_exclusion='raw Otsu foreground',best_z_policy='filter selected row; no reselection'),indent=2),encoding='utf8')
        print('Saved audit and filtered tables:',output)
    print('Original masks/tables retained. Unassessable references are not proof of an artifact.')
    return dict(all_instances=result,accepted=accepted,rejected=rejected,summary=summary,
        annotated_objects=annotated,filtered_objects=filtered_objects,annotated_best_z=best,
        filtered_best_z=filtered_best,settings=asdict(settings),output_dir=output)



In [ ]:
# Reads the raw image from cfg to avoid stale img_5d from another run.
ARTIFACT_GATE = audit_segmented_artifacts(
    cfg, image=None, objects=objects_df, best_z=best_z_df)


### 38b. Maximum nuclear cross-sectional area before and after filtering

One value per (T, linked nuclear ID): maximum single-component area across Z.
After filtering, take the maximum among passing components; fully rejected IDs
are missing (NaN), not zero. Branched IDs are flagged because a linked ID is not
necessarily one biological nucleus. No best-Z selection or proximity filter is used.


In [ ]:
keys = ['t', 'z', 'label']
identity = grouped_z_df[keys + ['nucleus_3d_id']].copy()
gate_rows = ARTIFACT_GATE['all_instances']
area_planes = identity.merge(gate_rows, on=keys, how='left',
                             validate='one_to_one', indicator=True)
if not area_planes['_merge'].eq('both').all():
    raise ValueError('Grouped objects and artifact gate do not match.')
if not area_planes.gate_pass.isin([True, False]).all():
    raise ValueError('Invalid gate decisions.')
groups = ['t', 'nucleus_3d_id']
before = area_planes.groupby(groups).area_um2.max().rename('before_um2')
after = area_planes.loc[area_planes.gate_pass].groupby(groups).area_um2.max().rename('after_um2')
branches = (area_planes.groupby(groups + ['z']).size().gt(1)
            .groupby(level=[0, 1]).any().rename('multiple_instances_same_z'))
MAX_AREA_FILTER_COMPARISON = pd.concat([before, after, branches], axis=1).reset_index()
comparison = MAX_AREA_FILTER_COMPARISON
display(comparison.groupby('t').agg(
    before_count=('before_um2', 'count'), after_count=('after_um2', 'count'),
    before_median=('before_um2', 'median'), after_median=('after_um2', 'median'),
    branched_ids=('multiple_instances_same_z', 'sum')))
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
rng = np.random.default_rng(0)
times = sorted(comparison.t.unique())
for i, t in enumerate(times):
    sub = comparison[comparison.t == t]
    jitter = rng.uniform(-0.055, 0.055, len(sub))
    for offset, col, color, label in [(-0.15, 'before_um2', 'gray', 'Before'),
                                    (0.15, 'after_um2', 'teal', 'After')]:
        vals = sub[col].to_numpy()
        axs[0].scatter(i + offset + jitter, vals, s=12, alpha=.35,
                       color=color, label=label if i == 0 else None)
        finite = vals[np.isfinite(vals)]
        if len(finite):
            axs[0].plot([i+offset-.09, i+offset+.09], [np.median(finite)]*2,
                        color=color, linewidth=3)
paired = comparison.dropna(subset=['after_um2'])
axs[1].scatter(paired.before_um2, paired.after_um2, s=14, alpha=.4)
limit = max(1., float(comparison.before_um2.max())) if len(comparison) else 1.
axs[1].plot([0, limit], [0, limit], '--', color='gray')
axs[0].set_xticks(range(len(times))); axs[0].set_xticklabels(times)
axs[0].set(xlabel='Timepoint index', ylabel='Maximum nuclear area (um²)',
           title='All linked IDs; points = nuclei, bars = medians')
axs[0].legend()
axs[1].set(xlabel='Before (um²)', ylabel='After (um²)',
           title='Surviving IDs only; identity line shown')
fig.suptitle('Enrichment/solidity only — multi-nucleus droplets NOT excluded')
fig.tight_layout(); plt.show()
if ARTIFACT_GATE.get('output_dir') is not None:
    out = Path(ARTIFACT_GATE['output_dir'])
    comparison.to_csv(out / 'max_area_before_after.csv', index=False)
    fig.savefig(out / 'max_area_before_after.png', dpi=160, bbox_inches='tight')


### 38c. Survivor audit across Z

Default: six area-ranked survivors each from T0, T1 and T2. Cyan = passing
component; red = rejected component. Review sheets are saved for manual labels.
The profile sums components at each Z for branched IDs and flags those IDs;
38b instead reports maximum single-component area. Neither establishes droplet multiplicity.


In [ ]:
"""Drop-in survivor review after ARTIFACT_GATE and grouped_z_df.

T0 and T1/T2 are comparison groups, not truth labels. No new rejection rules.
"""
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from IPython.display import display


def _survivor_run(zs):
    zs=sorted(set(map(int,zs)))
    longest=current=0; previous=None
    for z in zs:
        current=current+1 if previous is not None and z==previous+1 else 1
        longest=max(longest,current);previous=z
    return longest


def survivor_tables(grouped, gate, times):
    keys=['t','z','label']
    ids=grouped[keys+['nucleus_3d_id']].copy()
    if ids.duplicated(keys).any() or gate.duplicated(keys).any():
        raise ValueError('Duplicate instance keys; cannot assign review identities safely')
    planes=ids.merge(gate,on=keys,how='left',validate='one_to_one',indicator=True)
    if not planes['_merge'].eq('both').all():
        raise ValueError('Grouped instances do not match ARTIFACT_GATE; rerun the gate for this dataset')
    planes=planes.drop(columns='_merge')
    if not planes.gate_pass.isin([True,False]).all():raise ValueError('gate_pass must contain booleans')
    planes=planes[planes.t.isin(times)].copy()
    summaries=[]
    for (t,nid),g in planes.groupby(['t','nucleus_3d_id']):
        passed=g[g.gate_pass]
        if passed.empty:continue
        # A linked ID may branch; show summed area per Z and flag that ambiguity.
        areas=g.groupby('z').area_um2.sum().sort_index()
        kept=passed.groupby('z').area_um2.sum().sort_index()
        peak=int(kept.idxmax());all_peak=int(areas.idxmax())
        summaries.append(dict(t=int(t),nucleus_3d_id=int(nid),
            review_id='T%d_N%d'%(t,nid),detected_planes=int(g.z.nunique()),
            passing_planes=int(passed.z.nunique()),consecutive_passing=_survivor_run(passed.z),
            z_lo=int(g.z.min()),z_hi=int(g.z.max()),peak_z=peak,
            max_area_before_um2=float(areas.max()),max_area_after_um2=float(kept.max()),
            enrichment_median=float(passed.enrichment.median()),
            solidity_median=float(passed.solidity.median()),
            peak_at_detected_edge=all_peak in (int(areas.index.min()),int(areas.index.max())),
            multiple_instances_same_z=bool((g.groupby('z').size()>1).any()),
            parent_overlap_median=float(passed.parent_overlap_fraction.median())))
    return planes,pd.DataFrame(summaries)


def audit_survivors(cfg,grouped,gate,image=None,times=(0,1,2),per_t=6,save=True):
    if per_t<1:raise ValueError('per_t must be positive')
    planes,summary=survivor_tables(grouped,gate['all_instances'],times)
    if summary.empty:raise ValueError('No survivors in the requested timepoints')
    if image is None:image=tifffile.memmap(str(cfg.input_image_path),mode='r')
    labels=tifffile.memmap(str(cfg.nucleus_instance_hyperstack_path),mode='r')
    if labels.shape!=image.shape[:2]+image.shape[-2:]:raise ValueError('Image/instance-stack mismatch')
    eligible=pd.read_pickle(cfg.segmentation_index_path)
    included=eligible.included.astype(str).str.lower().isin(['true','1'])
    eligible=eligible[included & (eligible.z>=cfg.focus_min_z)]
    selected=[]
    for t in times:
        g=summary[summary.t==t].sort_values(['max_area_after_um2','nucleus_3d_id'])
        if g.empty:print('No survivors at T%d'%t);continue
        ix=np.unique(np.rint(np.linspace(0,len(g)-1,min(per_t,len(g)))).astype(int))
        selected.append(g.iloc[ix])
    review=pd.concat(selected,ignore_index=True)
    output=None
    if save:
        output=cfg.qc_dir/('survivor_audit_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        output.mkdir(parents=True,exist_ok=False)
    print('All survivors: comparisons are not manually confirmed truth labels.')
    display(summary.groupby('t').agg(nuclei=('review_id','size'),
        area_median=('max_area_after_um2','median'),passing_planes_median=('passing_planes','median'),
        consecutive_median=('consecutive_passing','median'),enrichment_median=('enrichment_median','median'),
        solidity_median=('solidity_median','median'),branched_ids=('multiple_instances_same_z','sum')).round(3))
    display(review.round(3))
    settings=gate['settings']
    for r in review.itertuples():
        g=planes[(planes.t==r.t)&(planes.nucleus_3d_id==r.nucleus_3d_id)]
        allowed=set(eligible[eligible.t==r.t].z.astype(int))
        zs=[z for z in range(max(cfg.focus_min_z,r.z_lo-1),min(image.shape[1],r.z_hi+2)) if z in allowed]
        if not zs:raise ValueError('Track has no eligible planes')
        # Fixed XY window encompasses the complete track on every displayed Z.
        bounds=[]
        for z,sub in g.groupby('z'):
            mask=np.isin(labels[r.t,int(z)],sub.label.to_numpy())
            yy,xx=np.nonzero(mask)
            if not len(yy):raise ValueError('Instance label missing from saved mask')
            bounds.append((yy.min(),xx.min(),yy.max()+1,xx.max()+1))
        pad=max(8,int(np.ceil(3/cfg.pixel_size_um)))
        a=max(0,min(x[0] for x in bounds)-pad);b=max(0,min(x[1] for x in bounds)-pad)
        c=min(labels.shape[-2],max(x[2] for x in bounds)+pad);d=min(labels.shape[-1],max(x[3] for x in bounds)+pad)
        crops=np.stack([np.asarray(image[r.t,z,cfg.nuclear_channel_index,a:c,b:d],np.float32) for z in zs])
        lo,hi=np.percentile(crops,[1,99.8]);hi=max(hi,lo+1e-6)
        fig,axs=plt.subplots(1,3,figsize=(14,3.5))
        all_area=g.groupby('z').area_um2.sum().reindex(zs)
        pass_area=g[g.gate_pass].groupby('z').area_um2.sum().reindex(zs)
        axs[0].plot(zs,all_area,'o-',color='gray',label='all detected')
        axs[0].plot(zs,pass_area,'o-',color='teal',label='passing only')
        axs[0].set_ylabel('Summed instance area (µm²)');axs[0].legend()
        for ax,col,threshold in [(axs[1],'enrichment',settings['min_enrichment']),
                                 (axs[2],'solidity',settings['min_solidity'])]:
            for accepted,color in [(True,'teal'),(False,'tomato')]:
                s=g[g.gate_pass==accepted];ax.scatter(s.z,s[col],c=color,s=28)
            ax.axhline(threshold,c='black',ls='--');ax.set_ylabel(col)
        for ax in axs:ax.set_xlabel('Z plane');ax.set_xticks(zs);ax.grid(alpha=.2)
        fig.suptitle('%s | passing %d/%d planes | longest run %d | branched ID: %s'%
            (r.review_id,r.passing_planes,r.detected_planes,r.consecutive_passing,r.multiple_instances_same_z))
        fig.tight_layout()
        if output:fig.savefig(output/(r.review_id+'_profiles.png'),dpi=140,bbox_inches='tight')
        plt.show();plt.close(fig)
        fig,axs=plt.subplots(int(np.ceil(len(zs)/5)),5,figsize=(16,3.4*np.ceil(len(zs)/5)),squeeze=False)
        for ax,z,crop in zip(axs.ravel(),zs,crops):
            ax.imshow(crop,cmap='gray',vmin=lo,vmax=hi)
            sub=g[g.z==z];local=np.asarray(labels[r.t,z,a:c,b:d])
            for row in sub.itertuples():
                mask=local==row.label
                ax.contour(mask,levels=[.5],colors=['cyan' if row.gate_pass else 'red'],linewidths=1)
            text='no linked detection' if sub.empty else 'pass %d / %d'%(int(sub.gate_pass.sum()),len(sub))
            ax.set_title('Z%d: %s'%(z,text),fontsize=9);ax.axis('off')
        for ax in axs.ravel()[len(zs):]:ax.axis('off')
        fig.suptitle(r.review_id+' | raw NLS, common intensity scale | cyan: pass; red: reject\n'
                     'Missing outlines mean no linked detection, not confirmed biological absence')
        fig.tight_layout()
        if output:fig.savefig(output/(r.review_id+'_z_stack.png'),dpi=140,bbox_inches='tight')
        plt.show();plt.close(fig)
    if output:
        summary.to_csv(output/'all_survivor_features.csv',index=False)
        planes.to_csv(output/'plane_measurements.csv',index=False)
        review=review.copy();review['review_label']='';review['notes']=''
        review.to_csv(output/'review_sheet.csv',index=False)
        (output/'settings.json').write_text(json.dumps(dict(times=list(times),per_t=per_t,
            selection='evenly spaced ranks of maximum passing area within each timepoint',
            gate=settings,source=str(cfg.nucleus_instance_hyperstack_path)),indent=2),encoding='utf8')
        print('Review labels: artifact / real nucleus / segmentation error / uncertain. Saved:',output)
    return dict(features=summary,planes=planes,selected=review,output_dir=output)



In [ ]:
SURVIVOR_AUDIT = audit_survivors(
    cfg, grouped_z_df, ARTIFACT_GATE, image=None,
    times=(0, 1, 2), per_t=6, save=True)


### 38d. Survivor probability audit

Run after 38c. Reads saved full-head probabilities (no second sigmoid), not the
four-channel compatibility TIFF. No segmentation rerun or new filtering is applied.
Metrics cover survivors in SURVIVOR_AUDIT; maps use its selected examples.
Probability thresholds are hypothetical; labels must be manually supplied by review ID.
Multi-nucleus droplets are a separate review category, not segmentation artifacts.


In [ ]:
"""Read-only probability diagnostics for the existing survivor population."""
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from skimage import measure
from scipy.ndimage import binary_erosion
from IPython.display import display


def survivor_probability_metrics(probabilities, mask, core_px=2):
    """Heads: interior, edge, equatorial. Already probabilities, not logits."""
    p = np.asarray(probabilities, dtype=float)
    if p.shape != (3,) + mask.shape or not mask.any():
        raise ValueError('Probability/mask shape mismatch or empty instance')
    if not np.isfinite(p).all() or (p < 0).any() or (p > 1).any():
        raise ValueError('Expected finite decoded probabilities in [0, 1]')
    if core_px < 1:
        raise ValueError('core_px must be positive')
    # Explicit zero padding at the crop edge prevents artificial core retention.
    core = binary_erosion(mask, iterations=core_px, border_value=0)
    boundary = mask & ~binary_erosion(mask, border_value=0)
    result = dict(mask_pixels=int(mask.sum()), core_pixels=int(core.sum()),
                  interior_median=float(np.median(p[0][mask])),
                  interior_p10=float(np.quantile(p[0][mask], .1)),
                  core_interior_median=float(np.median(p[0][core])) if core.any() else np.nan,
                  boundary_edge_median=float(np.median(p[1][boundary])),
                  equatorial_median=float(np.median(p[2][mask])))
    for threshold in (.5, .7, .8, .9, .95):
        result['interior_fraction_ge_%s' % str(threshold).replace('.', '_')] = float(
            np.mean(p[0][mask] >= threshold))
    return result


def audit_survivor_probabilities(cfg, survivors, core_px=2, review_labels=None, save=True):
    """Measure all survivors in the prior audit; plot its selected examples.

    review_labels maps review_id to artifact / real nucleus / multi-nucleus droplet /
    uncertain. No label is inferred from timepoint. No masks or decisions are changed.
    """
    root = Path(cfg.seg_dir)
    meta = json.loads((root / 'vulcan_all_heads.json').read_text(encoding='utf8'))
    names = meta['channels']
    wanted = ['nucleus_interior', 'nucleus_edge', 'nucleus_equatorial']
    if meta.get('axes') != 'TZCYX' or any(x not in names for x in wanted):
        raise ValueError('Unsupported full-head metadata')
    if meta.get('model_sha256') != cfg.model_sha256:
        raise ValueError('Saved probability model hash does not match cfg')
    if json.dumps(meta.get('config'), sort_keys=True) != json.dumps(cfg.to_run_signature(), sort_keys=True):
        raise ValueError('Saved probability configuration does not match cfg')
    heads = tifffile.memmap(root / 'vulcan_all_heads.tif', mode='r')
    labels = tifffile.memmap(cfg.nucleus_instance_hyperstack_path, mode='r')
    raw = tifffile.memmap(cfg.input_image_path, mode='r')
    if heads.shape != labels.shape[:2] + (len(names),) + labels.shape[-2:]:
        raise ValueError('Probability/instance stack mismatch')
    if raw.shape[:2] + raw.shape[-2:] != labels.shape:
        raise ValueError('Raw/instance stack mismatch')
    planes = survivors['planes'].copy()
    features = survivors['features']
    # Prior audit planes also contains fully rejected IDs; remove those here.
    planes = planes.merge(features[['t', 'nucleus_3d_id', 'review_id']],
                          on=['t', 'nucleus_3d_id'], validate='many_to_one')
    if planes.empty or planes.duplicated(['t', 'z', 'label']).any():
        raise ValueError('No survivors or duplicate instance keys')
    eligible = pd.read_pickle(cfg.segmentation_index_path)
    included = eligible.included.astype(str).str.lower().isin(['true', '1'])
    valid = set(map(tuple, eligible.loc[included, ['t', 'z']].to_numpy()))
    channels = [names.index(x) for x in wanted]
    rows = []
    for (t, z), group in planes.groupby(['t', 'z']):
        t, z = int(t), int(z)
        if (t, z) not in valid or z not in meta['included_z'] or z < cfg.focus_min_z:
            raise ValueError('Survivor references an excluded or unfinished plane')
        lab = np.asarray(labels[t, z])
        boxes = {r.label: r.bbox for r in measure.regionprops(lab)}
        for row in group.itertuples():
            if row.label not in boxes:
                raise ValueError('Survivor label missing from saved segmentation')
            a, b, c, d = boxes[row.label]
            mask = lab[a:c, b:d] == row.label
            p = np.stack([heads[t, z, k, a:c, b:d] for k in channels])
            metrics = survivor_probability_metrics(p, mask, core_px)
            rows.append(dict(t=t, z=z, label=row.label, **metrics))
    measurements = planes.merge(pd.DataFrame(rows), on=['t', 'z', 'label'], validate='one_to_one')
    # Diagnostic score: median of per-component core medians over passing components.
    # Branched IDs are retained but explicitly flagged in the review table.
    scores = measurements[measurements.gate_pass].groupby('review_id').agg(
        core_probability_score=('core_interior_median', 'median'),
        interior_probability_score=('interior_median', 'median'),
        measured_components=('label', 'size'),
        components_with_core=('core_interior_median', 'count'))
    summary = features.merge(scores, on='review_id', validate='one_to_one')
    review_labels = {} if review_labels is None else dict(review_labels)
    unknown = set(review_labels) - set(summary.review_id)
    if unknown:
        raise ValueError('Review IDs not in this audit: %s' % sorted(unknown))
    summary['review_label'] = summary.review_id.map(review_labels).fillna('unreviewed')
    display(summary[['review_id', 'core_probability_score', 'interior_probability_score',
                     'components_with_core', 'measured_components', 'multiple_instances_same_z', 'review_label']])
    out = None
    if save:
        out = Path(cfg.qc_dir) / ('survivor_probability_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        out.mkdir(parents=True, exist_ok=False)

    def finish(fig, name):
        fig.tight_layout()
        if out is not None:
            fig.savefig(out / (name + '.png'), dpi=140, bbox_inches='tight')
        plt.show(); plt.close(fig)

    selected = survivors['selected'].review_id.tolist()
    for rid in selected:
        g = measurements[measurements.review_id == rid].sort_values(['z', 'label'])
        if g.empty:
            continue
        # Largest passing component, not an automatically assumed equatorial plane.
        peak = g[g.gate_pass].sort_values('area_um2').iloc[-1]
        t, z, label = int(peak.t), int(peak.z), int(peak.label)
        yy, xx = np.nonzero(labels[t, z] == label)
        pad = max(8, int(np.ceil(3 / cfg.pixel_size_um)))
        a, b = max(0, yy.min()-pad), max(0, xx.min()-pad)
        c, d = min(labels.shape[-2], yy.max()+pad+1), min(labels.shape[-1], xx.max()+pad+1)
        mask = labels[t, z, a:c, b:d] == label
        image = np.asarray(raw[t, z, cfg.nuclear_channel_index, a:c, b:d])
        fig, axs = plt.subplots(1, 4, figsize=(16, 4))
        lo, hi = np.percentile(image, [1, 99.8])
        axs[0].imshow(image, cmap='gray', vmin=lo, vmax=max(hi, lo+1))
        axs[0].set_title('Raw NLS')
        for ax, k, title in zip(axs[1:], channels, wanted):
            im = ax.imshow(heads[t, z, k, a:c, b:d], vmin=0, vmax=1, cmap='viridis')
            ax.set_title(title); fig.colorbar(im, ax=ax, fraction=.046)
        for ax in axs:
            ax.contour(np.arange(-1, mask.shape[1]+1), np.arange(-1, mask.shape[0]+1),
                       np.pad(mask, 1), levels=[.5], colors=['cyan'])
            ax.set_xlim(-.5, mask.shape[1]-.5); ax.set_ylim(mask.shape[0]-.5, -.5); ax.axis('off')
        fig.suptitle('%s Z%d | largest passing component | core P=%.3f | E=%.2f S=%.2f' %
                     (rid, z, peak.core_interior_median, peak.enrichment, peak.solidity))
        finish(fig, rid + '_maps')
        fig, axs = plt.subplots(1, 3, figsize=(15, 4))
        for col in ['interior_median', 'core_interior_median', 'boundary_edge_median', 'equatorial_median']:
            axs[0].plot(g.z, g[col], 'o', label=col)
        axs[0].set_ylim(0, 1); axs[0].legend(fontsize=7); axs[0].set_ylabel('Probability')
        for passed, color in [(True, 'teal'), (False, 'tomato')]:
            sub = g[g.gate_pass == passed]
            axs[1].scatter(sub.z, sub.enrichment, color=color, label='gate pass' if passed else 'gate reject')
            axs[2].scatter(sub.z, sub.solidity, color=color)
        axs[1].set_ylabel('Enrichment'); axs[1].legend(); axs[2].set_ylabel('Solidity')
        for ax in axs:
            ax.set_xlabel('Z plane'); ax.grid(alpha=.2)
        fig.suptitle(rid + ' | all linked components; no probability filter applied')
        finish(fig, rid + '_profiles')
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, group in summary.groupby('review_label'):
        values = np.sort(group.core_probability_score.dropna().to_numpy())
        if len(values):
            ax.step(values, np.arange(1, len(values)+1)/len(values), where='post',
                    label='%s (n=%d)' % (label, len(values)))
    ax.set(xlabel='Median core probability score', ylabel='Cumulative fraction',
           xlim=(0, 1), ylim=(0, 1), title='Manual review categories; not inferred from T')
    if ax.lines: ax.legend()
    finish(fig, 'review_probability_distributions')
    threshold_rows = []
    for label, group in summary.groupby('review_label'):
        finite = group.core_probability_score.dropna()
        for threshold in (.5, .6, .7, .8, .9, .95):
            threshold_rows.append(dict(review_label=label, threshold=threshold, n_total=len(group),
                n_assessable=len(finite), n_unassessable=len(group)-len(finite),
                fraction_below=float((finite < threshold).mean()) if len(finite) else np.nan))
    thresholds = pd.DataFrame(threshold_rows)
    display(thresholds)
    print('Below-threshold fractions are hypothetical, not applied. Only manual labels support artifact-removal/real-loss estimates.')
    print('Equatorial confidence varies with Z. Empty cores are NaN, never automatic rejection.')
    if out is not None:
        measurements.to_csv(out / 'component_probabilities.csv', index=False)
        summary.to_csv(out / 'survivor_probability_review.csv', index=False)
        thresholds.to_csv(out / 'hypothetical_thresholds.csv', index=False)
        (out / 'settings.json').write_text(json.dumps(dict(core_px=core_px,
            heads_path=str(root / 'vulcan_all_heads.tif'), model_sha256=meta['model_sha256'],
            selected_ids=selected, score='median core probability over passing components',
            source_audit=str(survivors.get('output_dir')), review_labels=review_labels), indent=2), encoding='utf8')
        print('Saved:', out)
    return dict(planes=measurements, summary=summary, thresholds=thresholds, output_dir=out)


In [ ]:
# Optional manually confirmed review labels. Never label by timepoint alone.
PROBABILITY_REVIEW_LABELS = {
    # 'T0_N123': 'artifact',
    # 'T1_N456': 'real nucleus',
    # 'T2_N789': 'multi-nucleus droplet',
}
SURVIVOR_PROBABILITY_AUDIT = audit_survivor_probabilities(
    cfg, SURVIVOR_AUDIT, core_px=2,
    review_labels=PROBABILITY_REVIEW_LABELS, save=True)


### 38e. Component decisions and adjacent-Z audit

Run after the manually labeled probability audit. Defaults to confirmed artifacts.
Shows all their linked components, gate decisions, and exact score contributors.
Adjacent-Z overlaps are geometric evidence, not reconstructed historical link decisions.
No filtering changes. Optionally supply review_ids to include real-nucleus controls.


In [ ]:
"""Inspect saved survivor components without changing masks or linking."""
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from IPython.display import display


def component_overlap_metrics(first, second):
    a, b = int(first.sum()), int(second.sum())
    intersection = int(np.count_nonzero(first & second))
    return dict(overlap_pixels=intersection,
                overlap_over_smaller=intersection/min(a, b) if min(a, b) else np.nan,
                iou=intersection/(a+b-intersection) if a+b else np.nan,
                next_over_current_area=b/a if a else np.nan)


def audit_survivor_components(cfg, probability_audit, review_ids=None, save=True):
    """Default to manually labeled artifacts. Additional real controls may be supplied.

    Adjacent-Z pair overlaps are geometric evidence, not a reconstruction of link
    decisions: production links against previous ID unions, not individual pairs.
    """
    planes = probability_audit['planes'].copy()
    summary = probability_audit['summary'].copy()
    if review_ids is None:
        review_ids = summary.loc[summary.review_label.eq('artifact'), 'review_id'].tolist()
    review_ids = list(dict.fromkeys(review_ids))
    if not review_ids:
        raise ValueError('No manually labeled artifacts. Supply review_ids or rerun the labeled probability audit.')
    missing = set(review_ids) - set(summary.review_id)
    if missing:
        raise ValueError('Unknown review IDs: %s' % sorted(missing))
    if planes.duplicated(['t', 'z', 'label']).any():
        raise ValueError('Duplicate component identities')
    labels = tifffile.memmap(cfg.nucleus_instance_hyperstack_path, mode='r')
    raw = tifffile.memmap(cfg.input_image_path, mode='r')
    if raw.shape[:2] + raw.shape[-2:] != labels.shape:
        raise ValueError('Image and mask shapes differ')
    index = pd.read_pickle(cfg.segmentation_index_path)
    valid = set(map(tuple, index.loc[index.included.astype(str).str.lower().isin(['true', '1']), ['t', 'z']].to_numpy()))
    planes = planes[planes.review_id.isin(review_ids)].copy()
    if not planes.gate_pass.isin([True, False]).all():
        raise ValueError('Invalid gate decisions')
    planes['contributes_to_score'] = planes.gate_pass & np.isfinite(planes.core_interior_median)
    planes['score_exclusion_reason'] = np.where(~planes.gate_pass, 'gate rejected',
        np.where(planes.contributes_to_score, 'included', 'no finite core probability'))
    planes['component_area_um2'] = planes.mask_pixels * cfg.pixel_size_um**2
    planes['components_same_z'] = planes.groupby(['review_id', 'z']).label.transform('size')
    out = None
    if save:
        out = Path(cfg.qc_dir) / ('survivor_components_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        out.mkdir(parents=True, exist_ok=False)
    links, features = [], []

    def finish(fig, name):
        fig.tight_layout()
        if out is not None:
            fig.savefig(out / (name + '.png'), dpi=160, bbox_inches='tight')
        plt.show(); plt.close(fig)

    for rid in review_ids:
        g = planes[planes.review_id.eq(rid)].sort_values(['z', 'label'])
        if g.empty or g.t.nunique() != 1:
            raise ValueError('Missing or ambiguous component rows for ' + rid)
        t = int(g.t.iloc[0])
        bounds = []
        for r in g.itertuples():
            if (t, int(r.z)) not in valid or r.z < cfg.focus_min_z:
                raise ValueError('Component references an excluded plane')
            yy, xx = np.nonzero(labels[t, int(r.z)] == int(r.label))
            if len(yy) != r.mask_pixels:
                raise ValueError('Saved masks changed since probability audit: ' + rid)
            bounds.append((yy.min(), xx.min(), yy.max()+1, xx.max()+1))
        pad = max(8, int(np.ceil(3/cfg.pixel_size_um)))
        a = max(0, min(x[0] for x in bounds)-pad)
        b = max(0, min(x[1] for x in bounds)-pad)
        c = min(labels.shape[-2], max(x[2] for x in bounds)+pad)
        d = min(labels.shape[-1], max(x[3] for x in bounds)+pad)
        zs = [z for z in range(max(cfg.focus_min_z, int(g.z.min())-1),
                              min(labels.shape[1], int(g.z.max())+2)) if (t, z) in valid]
        masks = {(int(r.z), int(r.label)): labels[t, int(r.z), a:c, b:d] == int(r.label)
                 for r in g.itertuples()}
        for r in g.itertuples():
            # Include zero-overlap pairs to reveal discontinuous/branched ID membership.
            for s in g[g.z.eq(r.z+1)].itertuples():
                links.append(dict(review_id=rid, t=t, z=int(r.z), label=int(r.label),
                    next_z=int(s.z), next_label=int(s.label),
                    current_gate_pass=bool(r.gate_pass), next_gate_pass=bool(s.gate_pass),
                    **component_overlap_metrics(masks[(int(r.z), int(r.label))],
                                                masks[(int(s.z), int(s.label))])))
        scores = g.loc[g.contributes_to_score, 'core_interior_median']
        recomputed = float(scores.median()) if len(scores) else np.nan
        recorded = float(summary.set_index('review_id').loc[rid, 'core_probability_score'])
        if not np.isclose(recomputed, recorded, equal_nan=True):
            raise ValueError('Probability score could not be reproduced for ' + rid)
        features.append(dict(review_id=rid, t=t, components=len(g),
            passing_components=int(g.gate_pass.sum()), score_contributors=len(scores),
            reproduced_score=recomputed, min_pixels=int(g.mask_pixels.min()),
            max_pixels=int(g.mask_pixels.max()), branches=bool(g.components_same_z.gt(1).any())))
        print('\n', rid, '| reproduced probability score:', recomputed)
        display(g[['review_id', 'z', 'label', 'mask_pixels', 'component_area_um2',
                   'core_pixels', 'gate_pass', 'gate_reason', 'enrichment', 'solidity',
                   'core_interior_median', 'contributes_to_score', 'score_exclusion_reason',
                   'components_same_z']])
        crops = [np.asarray(raw[t, z, cfg.nuclear_channel_index, a:c, b:d]) for z in zs]
        # Sample only for display scaling; metrics above use full-resolution masks.
        sampled = np.concatenate([x[::4, ::4].ravel() for x in crops])
        lo, hi = np.percentile(sampled, [1, 99.8]); hi = max(hi, lo+1e-6)
        for start in range(0, len(zs), 8):
            page = zs[start:start+8]
            fig, axs = plt.subplots(int(np.ceil(len(page)/4)), 4,
                                    figsize=(18, 4.5*np.ceil(len(page)/4)), squeeze=False)
            for ax, z in zip(axs.ravel(), page):
                ax.imshow(crops[zs.index(z)], cmap='gray', vmin=lo, vmax=hi, interpolation='nearest')
                sub = g[g.z.eq(z)]
                for r in sub.itertuples():
                    mask = masks[(int(z), int(r.label))]
                    color = 'cyan' if r.gate_pass else 'red'
                    ax.contour(np.arange(-1, mask.shape[1]+1), np.arange(-1, mask.shape[0]+1),
                               np.pad(mask, 1), levels=[.5], colors=[color], linewidths=.8)
                    yy, xx = np.nonzero(mask)
                    ax.text(xx.mean(), yy.mean(), '%d%s (%dpx)' %
                            (r.label, '*' if r.contributes_to_score else '', r.mask_pixels),
                            color='yellow', fontsize=8)
                    if r.mask_pixels <= 4:
                        ax.scatter(xx, yy, s=80, facecolors='none', edgecolors=color)
                ax.set_title('Z%d | %d linked components' % (z, len(sub)))
                ax.set_xlim(-.5, d-b-.5); ax.set_ylim(c-a-.5, -.5); ax.axis('off')
            for ax in axs.ravel()[len(page):]: ax.axis('off')
            fig.suptitle(rid + ' | cyan: gate pass; red: reject; *: contributes to score\n'
                         'Fixed XY window/intensity scale. Circles highlight <=4px masks (not their size).')
            finish(fig, rid + '_z_page_%02d' % (start//8+1))
        fig, axs = plt.subplots(1, 2, figsize=(12, 4))
        for passed, color in [(True, 'teal'), (False, 'tomato')]:
            sub = g[g.gate_pass.eq(passed)]
            axs[0].scatter(sub.z, sub.mask_pixels, color=color, label='pass' if passed else 'reject')
        axs[0].set_yscale('log'); axs[0].set_ylabel('Component area (pixels; log scale)'); axs[0].legend()
        axs[1].scatter(g.z, g.core_interior_median, color='gray', label='all finite core scores')
        contributing = g[g.contributes_to_score]
        axs[1].scatter(contributing.z, contributing.core_interior_median, marker='*', s=100,
                       color='teal', label='score contributors')
        if np.isfinite(recomputed): axs[1].axhline(recomputed, ls='--', color='teal')
        axs[1].set_ylim(0, 1); axs[1].set_ylabel('Core probability'); axs[1].legend(fontsize=8)
        for ax in axs: ax.set_xlabel('Z'); ax.grid(alpha=.2)
        fig.suptitle(rid); finish(fig, rid + '_components')
    overlaps = pd.DataFrame(links, columns=['review_id', 't', 'z', 'label', 'next_z', 'next_label',
        'current_gate_pass', 'next_gate_pass', 'overlap_pixels', 'overlap_over_smaller', 'iou',
        'next_over_current_area'])
    features = pd.DataFrame(features)
    display(features); display(overlaps)
    print('Pair overlaps are not inferred link edges; production compares previous ID unions.')
    print('No area cutoff, probability cutoff, relinking, or multi-nucleus exclusion applied.')
    if out is not None:
        planes.to_csv(out / 'component_decisions.csv', index=False)
        overlaps.to_csv(out / 'adjacent_z_pair_overlaps.csv', index=False)
        features.to_csv(out / 'score_reproduction.csv', index=False)
        (out / 'settings.json').write_text(json.dumps(dict(review_ids=review_ids,
            probability_audit=str(probability_audit.get('output_dir')),
            masks=str(cfg.nucleus_instance_hyperstack_path), pixel_size_um=cfg.pixel_size_um,
            overlap_scope='same-ID component pairs on adjacent Z; not historical link edges'), indent=2), encoding='utf8')
        print('Saved:', out)
    return dict(components=planes, overlaps=overlaps, summary=features, output_dir=out)


In [ ]:
SURVIVOR_COMPONENT_AUDIT = audit_survivor_components(
    cfg, SURVIVOR_PROBABILITY_AUDIT, save=True)


### 38f. One-pixel sensitivity and raw envelope evidence

Run after the manually labeled probability audit. Tests exclusion of exactly one-pixel
components across the full gate dataset without changing the original decisions or masks.
Compares remaining reviewed artifacts and real nuclei using raw NLS/NPC and parent
droplet geometry. NPC contrast is exploratory, not proof of a closed envelope.
Parent droplet IDs are plane-local. No multi-nucleus exclusion or relinking is applied.


In [ ]:
"""One-pixel sensitivity test and raw-channel evidence; never rewrites masks."""
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
import tifffile
from IPython.display import display


def test_one_pixel_exclusion(gate, grouped, pixel_size_um):
    data = gate['all_instances'].copy()
    keys = ['t', 'z', 'label']
    # area_um2 was measured directly from the saved instance mask by the gate.
    pixels = data.area_um2.to_numpy() / pixel_size_um**2
    if not np.allclose(pixels, np.rint(pixels), atol=1e-5, rtol=0):
        raise ValueError('Gate areas disagree with the current pixel calibration')
    data['area_pixels'] = np.rint(pixels).astype(int)
    data['test_gate_pass'] = data.gate_pass & data.area_pixels.gt(1)
    data['one_pixel_removed'] = data.gate_pass & ~data.test_gate_pass
    ids = grouped[keys + ['nucleus_3d_id']]
    linked = ids.merge(data, on=keys, validate='one_to_one', how='outer', indicator=True)
    if not linked['_merge'].eq('both').all():
        raise ValueError('Grouped objects and gate must cover identical instance keys')
    tracks = linked.groupby(['t', 'nucleus_3d_id']).agg(
        before=('gate_pass', 'any'), after=('test_gate_pass', 'any')).reset_index()
    tracks['lost'] = tracks.before & ~tracks.after
    counts = data.groupby('t').agg(before_components=('gate_pass', 'sum'),
        after_components=('test_gate_pass', 'sum'), removed_one_pixel=('one_pixel_removed', 'sum'))
    counts = counts.join(tracks.groupby('t').agg(before_ids=('before', 'sum'),
        after_ids=('after', 'sum'), lost_ids=('lost', 'sum')))
    return data, tracks, counts


def envelope_measurements(npc, nucleus, parent, other_nuclei, pixel_size_um):
    # The caller pads the crop beyond these distances; parent limits reference support.
    inside = distance_transform_edt(nucleus) * pixel_size_um
    outside = distance_transform_edt(~nucleus) * pixel_size_um
    boundary = (nucleus & (inside <= .5)) | (~nucleus & parent & ~other_nuclei & (outside <= .5))
    reference = parent & ~nucleus & ~other_nuclei & (outside >= 1.) & (outside <= 2.)
    b, r = npc[boundary], npc[reference]
    valid = len(b) > 0 and len(r) >= 50 and np.isfinite(b).all() and np.isfinite(r).all()
    bmean = float(b.mean()) if len(b) else np.nan
    rmean = float(r.mean()) if len(r) else np.nan
    ratio = bmean/rmean if valid and rmean > 0 else np.nan
    return dict(npc_boundary_mean=bmean, npc_reference_mean=rmean,
                npc_boundary_reference_ratio=ratio, npc_boundary_pixels=len(b),
                npc_reference_pixels=len(r), npc_reference_ok=bool(valid and rmean > 0)), boundary, reference


def audit_survivor_envelopes(cfg, gate, grouped, probability_audit, save=True):
    candidate, tracks, counts = test_one_pixel_exclusion(gate, grouped, cfg.pixel_size_um)
    print('Hypothetical one-pixel exclusion across the gate dataset; production decisions unchanged:')
    display(counts)
    summary = probability_audit['summary']
    reviewed = summary[summary.review_label.isin(['artifact', 'real nucleus'])].copy()
    if reviewed.empty:
        raise ValueError('Run the manually labeled probability audit first')
    reviewed = reviewed.merge(tracks, on=['t', 'nucleus_3d_id'], validate='one_to_one')
    display(reviewed.groupby('review_label').agg(reviewed=('review_id', 'size'),
        removed_by_one_pixel_test=('lost', 'sum')))
    raw = tifffile.memmap(cfg.input_image_path, mode='r')
    nuclei = tifffile.memmap(cfg.nucleus_instance_hyperstack_path, mode='r')
    droplets = tifffile.memmap(cfg.droplet_instance_hyperstack_path, mode='r')
    if raw.shape[:2] + raw.shape[-2:] != nuclei.shape or droplets.shape != nuclei.shape:
        raise ValueError('Raw/nuclear/droplet stack mismatch')
    measured = probability_audit['planes'].merge(
        candidate[['t', 'z', 'label', 'test_gate_pass']], on=['t', 'z', 'label'],
        validate='one_to_one', how='left')
    if measured.test_gate_pass.isna().any():
        raise ValueError('Probability audit references missing gate instances')
    out = None
    if save:
        out = Path(cfg.qc_dir)/('survivor_envelope_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        out.mkdir(parents=True, exist_ok=False)
    rows = []
    for review in reviewed[reviewed.after].itertuples():
        g = measured[measured.review_id.eq(review.review_id) & measured.test_gate_pass]
        peak_index = g.area_um2.idxmax()
        for ix, row in g.iterrows():
            t, z, label = int(row.t), int(row.z), int(row.label)
            plane = np.asarray(nuclei[t, z]); drop = np.asarray(droplets[t, z])
            yy, xx = np.nonzero(plane == label)
            if len(yy) != row.mask_pixels:
                raise ValueError('Masks changed since probability audit')
            values = drop[yy, xx]; values = values[values > 0]
            parent_id = int(np.bincount(values).argmax()) if len(values) else 0
            if parent_id != int(row.parent_droplet):
                raise ValueError('Parent droplet changed since gate audit')
            parent_area = int(np.count_nonzero(drop == parent_id)) if parent_id else 0
            pad = max(8, int(np.ceil(3/cfg.pixel_size_um)))
            a,b = max(0, yy.min()-pad), max(0, xx.min()-pad)
            c,d = min(plane.shape[0], yy.max()+pad+1), min(plane.shape[1], xx.max()+pad+1)
            mask = plane[a:c,b:d] == label
            parent = (drop[a:c,b:d] == parent_id) if parent_id else np.zeros(mask.shape, bool)
            npc = np.asarray(raw[t,z,cfg.npc_channel_index,a:c,b:d], dtype=float)
            other = (plane[a:c,b:d] > 0) & ~mask
            metrics, band, reference = envelope_measurements(npc, mask, parent, other, cfg.pixel_size_um)
            overlap = int(np.count_nonzero(parent & mask))
            rec = dict(review_id=review.review_id, review_label=review.review_label,
                t=t, z=z, label=label, area_pixels=len(yy), parent_droplet=parent_id,
                parent_area_pixels=parent_area, parent_overlap_fraction=overlap/len(yy),
                nucleus_to_parent_area=len(yy)/parent_area if parent_area else np.nan,
                parent_occupied_fraction=overlap/parent_area if parent_area else np.nan,
                image_border=bool(yy.min()==0 or xx.min()==0 or yy.max()==plane.shape[0]-1 or xx.max()==plane.shape[1]-1),
                **metrics)
            rows.append(rec)
            if ix != peak_index:
                continue
            fig, axs = plt.subplots(1,3,figsize=(16,5))
            nls = np.asarray(raw[t,z,cfg.nuclear_channel_index,a:c,b:d])
            for ax, image, title in [(axs[0],nls,'Raw NLS'),(axs[1],npc,'Raw NPC'),(axs[2],npc,'NPC measurement regions')]:
                lo,hi=np.percentile(image,[1,99.8])
                ax.imshow(image,cmap='gray',vmin=lo,vmax=max(hi,lo+1e-6),interpolation='nearest')
                ax.contour(mask,levels=[.5],colors=['cyan'])
                if parent.any() and not parent.all(): ax.contour(parent,levels=[.5],colors=['yellow'])
                ax.set_title(title); ax.axis('off')
            for region,color in [(band,'magenta'),(reference,'lime')]:
                if region.any() and not region.all(): axs[2].contour(region,levels=[.5],colors=[color],linewidths=.6)
            fig.suptitle('%s | %s | Z%d | nuclear/parent area=%.3f | NPC ratio=%.3f\n'
                'Largest remaining passing component; cyan=nucleus, yellow=parent, magenta=boundary band, green=reference\n'
                'Channels scaled separately per crop; use raw-intensity metrics for comparison' %
                (review.review_id,review.review_label,z,rec['nucleus_to_parent_area'],rec['npc_boundary_reference_ratio']))
            fig.tight_layout()
            if out: fig.savefig(out/(review.review_id+'_raw_envelope.png'),dpi=150,bbox_inches='tight')
            plt.show(); plt.close(fig)
    metrics = pd.DataFrame(rows)
    if not metrics.empty:
        display(metrics.groupby(['review_id','review_label']).agg(
            components=('label','size'), median_npc_ratio=('npc_boundary_reference_ratio','median'),
            assessable_npc_components=('npc_reference_ok','sum'),
            max_nucleus_parent_ratio=('nucleus_to_parent_area','max')))
        fig,ax=plt.subplots(figsize=(8,5))
        for category,g in metrics.groupby('review_label'):
            ax.scatter(g.nucleus_to_parent_area,g.npc_boundary_reference_ratio,label=category,alpha=.6)
        ax.set(xlabel='Nuclear component area / parent droplet area',ylabel='Raw NPC boundary / local reference',
               title='Remaining passing components; exploratory, no threshold applied')
        ax.legend();fig.tight_layout()
        if out: fig.savefig(out/'envelope_comparison.png',dpi=150,bbox_inches='tight')
        plt.show();plt.close(fig)
    if out:
        candidate.to_csv(out/'hypothetical_component_decisions.csv',index=False)
        tracks.to_csv(out/'hypothetical_track_survival.csv',index=False)
        counts.to_csv(out/'one_pixel_counts_by_t.csv')
        reviewed.to_csv(out/'reviewed_one_pixel_effect.csv',index=False)
        metrics.to_csv(out/'raw_envelope_metrics.csv',index=False)
        (out/'settings.json').write_text(json.dumps(dict(pixel_size_um=cfg.pixel_size_um,
            test='remove gate-passing components with exactly one pixel; no relinking',
            npc_boundary_um=.5, npc_reference_um=[1,2], min_reference_pixels=50,
            gate_source=str(gate.get('output_dir')),probability_source=str(probability_audit.get('output_dir')),
            masks=str(cfg.nucleus_instance_hyperstack_path)),indent=2),encoding='utf8')
        print('Saved:',out)
    print('NPC ratios are exploratory, not envelope-closure measurements. Parent labels are plane-local.')
    print('Original gate and masks unchanged; no multi-nucleus droplet exclusion applied.')
    return dict(candidate_components=candidate,track_survival=tracks,counts=counts,
                reviewed_effect=reviewed,envelope_metrics=metrics,output_dir=out)


In [ ]:
SURVIVOR_ENVELOPE_AUDIT = audit_survivor_envelopes(
    cfg, ARTIFACT_GATE, grouped_z_df, SURVIVOR_PROBABILITY_AUDIT, save=True)


### 38g. Training RANSAC droplet geometry comparison

Uses the active training notebook's geometry functions and declared geometry defaults
in a separate namespace; does not execute its training/setup cells or mutate cfg.
Pixel calibration and NPC channel come from the current inference configuration.
Inventory at Z15; geometry scans from Z3; nuclear evaluation stays at Z>=6.
The empirical 2.18 um geometry step is retained, not substituted for acquisition spacing.

This can take substantial CPU time and memory; no neural inference is run.
Completed per-timepoint geometry is cached with code/config/input provenance.
Observed inlier circles and sphere-predicted support are distinguished; predicted
support is not asserted to be a validated droplet label. Manual gold ROI corrections
are not available in this inference comparison.

Original nuclear masks, model droplet masks, and ARTIFACT_GATE are unchanged.
Missing/ambiguous parents are unassessable, not biological negatives.
`geometry_gate_pass` isolates the support change; `contained_gate_pass` adds a
separate exploratory 85% containment requirement. No single-pixel or multiplicity
rule is added here. Inspect coverage and circle overlays before adopting results.


In [ ]:
"""Training-derived droplet support audit. No inference or production-mask writes."""
import ast
import copy
import hashlib
import itertools
import json
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage import filters, morphology, measure, segmentation
from skimage.morphology import h_maxima
from IPython.display import display


def load_training_geometry(training_notebook, cfg):
    """Execute only the named geometry definitions, never training notebook cells."""
    source = Path(training_notebook).read_text(encoding='utf8')
    functions, config = {}, None
    for cell in json.loads(source)['cells']:
        if cell['cell_type'] != 'code': continue
        text = ''.join(cell['source'])
        try: tree = ast.parse(text)
        except SyntaxError: continue
        for node in tree.body:
            if isinstance(node, ast.FunctionDef): functions[node.name] = ast.get_source_segment(text, node)
            if isinstance(node, ast.ClassDef) and node.name == 'PipelineConfig': config = node
    fields = {n.target.id: n.value for n in config.body if isinstance(n, ast.AnnAssign)}
    geom_fields = next(ast.literal_eval(n.value) for n in config.body
                      if isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and t.id == '_GEOM_FIELDS' for t in n.targets))
    settings = {k: ast.literal_eval(fields[k]) for k in tuple(geom_fields) + ('erosion_px',)}
    settings.update(pixel_size_um=float(cfg.pixel_size_um), npc_channel_idx=int(cfg.npc_channel_index))
    gc = SimpleNamespace(**settings)
    gc.min_droplet_area_px = lambda: gc.min_droplet_area_um2 / gc.pixel_size_um**2
    names = ['extract_plane', 'clip_histogram', '_circularity', 'detect_droplets_npc_watershed',
             '_circle_from_3', '_fit_circle_kasa', 'fit_circle_ransac', 'smooth_npc_plane',
             'wall_points_on_smoothed', 'fit_droplet_circle_on_smoothed', '_sphere_lsq',
             'fit_sphere_profile', 'fit_sphere_ransac', 'compute_droplet_geometry',
             'sphere_radius_px', 'predicted_circle', 'p2_circle']
    selected = '\n\n'.join(functions[name] for name in names)
    namespace = dict(np=np, filters=filters, morphology=morphology, measure=measure,
                     segmentation=segmentation, ndimage=ndimage, h_maxima=h_maxima,
                     copy=copy, itertools=itertools, cfg=gc)
    exec(compile(selected, str(training_notebook) + ':geometry-only', 'exec'), namespace)
    return namespace, gc, settings, hashlib.sha256(selected.encode()).hexdigest()


def circle_crop(cx, cy, radius, shape):
    h,w = shape
    a,b = max(0,int(np.floor(cy-radius))), max(0,int(np.floor(cx-radius)))
    c,d = min(h,int(np.ceil(cy+radius))+1), min(w,int(np.ceil(cx+radius))+1)
    a,b,c,d = min(a,h),min(b,w),max(c,0),max(d,0)
    yy,xx = np.ogrid[a:c,b:d]
    return (a,b,c,d), (xx-cx)**2+(yy-cy)**2 <= radius**2


def fitted_plane_support(geometry, z, shape, gc, namespace):
    """Training p2_circle policy: valid observed circle, else sphere prediction."""
    circles = []
    for did,g in geometry.items():
        selected = namespace['p2_circle'](g,z,gc)
        if selected is None: continue
        kind,circ = selected
        if kind not in ('fitted','predicted'):
            raise ValueError('Unsupported training p2_circle contract')
        observed = kind == 'fitted'
        if circ[2] <= gc.erosion_px: continue
        cx,cy,r = map(float,circ)
        box,mask = circle_crop(cx,cy,r-gc.erosion_px,shape)
        if mask.any(): circles.append(dict(parent=int(did)+1, cx=cx,cy=cy,r=r,
            source='observed_inlier' if observed else 'sphere_predicted',box=box,mask=mask))
    return circles


def audit_ransac_droplets(cfg, gate, training_notebook, timepoints=(0,1,2),
                          cache_dir=None, min_parent_overlap=.85, save=True):
    """Separate comparison: geometry-only gate, plus explicitly reported containment test.

    No fitted support is not evidence of an artifact. Ambiguous/missing support is
    unassessable. Nuclear shapes and linked IDs are never changed. Cache is optional
    and validated against sources/config; partially completed timepoints are reusable.
    """
    from segmented_artifact_gate import ArtifactGateSettings, measure_segment_gate
    ns,gc,settings,code_hash = load_training_geometry(training_notebook,cfg)
    if not 0 < min_parent_overlap <= 1: raise ValueError('Invalid containment fraction')
    hs = tifffile.memmap(cfg.input_image_path, mode='r')
    labels = tifffile.memmap(cfg.nucleus_instance_hyperstack_path, mode='r')
    if hs.shape[:2]+hs.shape[-2:] != labels.shape: raise ValueError('Raw/mask shape mismatch')
    if not 0 <= gc.inventory_ref_z < hs.shape[1]: raise ValueError('Training inventory_ref_z outside stack')
    ts = sorted(set(map(int,timepoints)))
    if not ts or any(t < 0 or t >= hs.shape[0] for t in ts): raise ValueError('Invalid timepoints')
    old = gate['all_instances'].loc[lambda d:d.t.isin(ts)].copy()
    if set(old.t) != set(ts): raise ValueError('Requested timepoints absent from original gate')
    if old.duplicated(['t','z','label']).any(): raise ValueError('Duplicate gate keys')
    index = pd.read_pickle(cfg.segmentation_index_path)
    valid = set(map(tuple,index.loc[index.included.astype(str).str.lower().isin(['true','1']),['t','z']].to_numpy()))
    for t,z in old[['t','z']].drop_duplicates().itertuples(index=False,name=None):
        if (t,z) not in valid or z < cfg.focus_min_z: raise ValueError('Gate includes invalid nuclear plane')
    params = ArtifactGateSettings(**gate['settings'])
    st = Path(cfg.input_image_path).stat()
    signature = dict(settings=settings,code_sha256=code_hash,
                     image=[str(Path(cfg.input_image_path).resolve()),st.st_size,st.st_mtime_ns])
    out = None
    if save:
        out=Path(cfg.qc_dir)/('ransac_droplet_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        out.mkdir(parents=True,exist_ok=False)
    cache = Path(cache_dir) if cache_dir is not None else out
    if cache is not None:
        manifest=cache/'geometry_signature.json'
        if manifest.exists():
            if json.loads(manifest.read_text()) != signature: raise ValueError('Geometry cache provenance mismatch')
        else:
            if cache.exists() and any(cache.iterdir()): raise ValueError('Cache lacks provenance; use a fresh directory')
            cache.mkdir(parents=True,exist_ok=True)
            manifest.write_text(json.dumps(signature,indent=2),encoding='utf8')
        print('Geometry cache (reusable after interruption):',cache,flush=True)
    rows,circle_rows,coverage=[],[],[]
    for t in ts:
        cp=cache/('geometry_t%03d.pkl'%t) if cache else None
        if cp is not None and cp.exists():
            saved=pd.read_pickle(cp); geometry=saved['geometry']; n_inventory=saved['n_inventory']
            print('Loaded validated geometry T%d'%t,flush=True)
        else:
            inventory=ns['detect_droplets_npc_watershed'](ns['extract_plane'](hs,t,gc.inventory_ref_z,gc.npc_channel_idx),gc,compact=True)
            n_inventory=len(inventory)
            if not inventory: raise RuntimeError('T%d: no inventory; refusing fallback to model droplets'%t)
            geometry=ns['compute_droplet_geometry'](hs,t,inventory,gc,progress=True)
            if cp is not None:
                tmp=cp.with_suffix('.tmp');pd.to_pickle(dict(geometry=geometry,n_inventory=n_inventory),tmp);tmp.replace(cp)
        coverage.append(dict(t=t,inventory=n_inventory,validated_geometry=len(geometry)))
        for z,group in old[old.t.eq(t)].groupby('z'):
            z=int(z);lab=np.asarray(labels[t,z]);raw=np.asarray(hs[t,z,cfg.nuclear_channel_index])
            circles=fitted_plane_support(geometry,z,lab.shape,gc,ns)
            assignments={}; support_cache={}
            for circle in circles:
                did=circle['parent'];a,b,c,d=circle['box'];mask=circle['mask']
                ids,counts=np.unique(lab[a:c,b:d][mask],return_counts=True)
                for label,count in zip(ids,counts):
                    if label: assignments.setdefault(int(label),[]).append((int(count),did))
                values=raw[a:c,b:d][mask]
                threshold=float(filters.threshold_otsu(values)) if values.max()>values.min() else np.nan
                support_cache[did]=(circle,threshold)
                circle_rows.append(dict(t=t,z=z,parent=did,cx=circle['cx'],cy=circle['cy'],
                    radius_px=circle['r'],support_pixels=int(mask.sum()),source=circle['source']))
            regions={r.label:r for r in measure.regionprops(lab)}
            for oldrow in group.itertuples():
                label=int(oldrow.label);region=regions.get(label)
                if region is None or not np.isclose(region.area*cfg.pixel_size_um**2,oldrow.area_um2):
                    raise ValueError('Original gate disagrees with saved nuclear masks')
                candidates=sorted(assignments.get(label,[]),reverse=True)
                eligible=[(n,did) for n,did in candidates if n/region.area>=min_parent_overlap]
                rec=dict(t=t,z=z,label=label,old_parent=oldrow.parent_droplet,
                    old_enrichment=oldrow.enrichment,old_gate_pass=bool(oldrow.gate_pass),
                    area_pixels=int(region.area),parent_candidates=len(candidates),
                    parent=0,parent_source='none',parent_overlap_fraction=0.,
                    geometry_status='missing_parent',geometry_gate_pass=None,contained_gate_pass=None,
                    enrichment=np.nan,solidity=float(region.solidity),gate_reason='unassessable geometry')
                # Any overlap with >1 circle is reported; >1 qualifying parents is ambiguous.
                tied = not eligible and len(candidates)>1 and candidates[0][0]==candidates[1][0]
                if candidates and len(eligible)<=1 and not tied:
                    n,did=eligible[0] if eligible else candidates[0]
                    circle,threshold=support_cache[did]
                    overlap=n/region.area
                    gap=max(1,int(np.ceil(params.ring_gap_um/cfg.pixel_size_um)))
                    pad=gap+max(1,int(np.ceil(params.ring_width_um/cfg.pixel_size_um)))
                    a,b,c,d=region.bbox;a,b,c,d=max(0,a-pad),max(0,b-pad),min(lab.shape[0],c+pad),min(lab.shape[1],d+pad)
                    yy,xx=np.ogrid[a:c,b:d]
                    support=(xx-circle['cx'])**2+(yy-circle['cy'])**2 <= (circle['r']-gc.erosion_px)**2
                    crop=raw[a:c,b:d];candidate=lab[a:c,b:d]==label
                    foreground=(crop>threshold)&support if np.isfinite(threshold) else np.zeros_like(support)
                    metric,_=measure_segment_gate(crop,candidate,support,foreground,cfg.pixel_size_um,params,float(region.solidity))
                    rec.update(metric)
                    rec.update(parent=did,parent_source=circle['source'],parent_overlap_fraction=overlap,
                        geometry_status='assigned' if overlap>=min_parent_overlap else 'partial_parent',
                        geometry_gate_pass=bool(metric['gate_pass']),
                        contained_gate_pass=bool(metric['gate_pass'] and overlap>=min_parent_overlap))
                elif len(eligible)>1 or tied: rec['geometry_status']='ambiguous_parent'
                rows.append(rec)
            print('Audited T%d Z%d: %d fitted/predicted supports, %d nuclei'%(t,z,len(circles),len(group)),flush=True)
            # Full-field raw NPC and fitted circles, not model droplet labels.
            if z==int(group.z.iloc[0]) and z==int(old[old.t.eq(t)].z.min()):
                fig,ax=plt.subplots(figsize=(12,8));im=np.asarray(hs[t,z,gc.npc_channel_idx]);lo,hi=np.percentile(im,[1,99.8])
                ax.imshow(im,cmap='gray',vmin=lo,vmax=max(hi,lo+1e-6))
                for circle in circles:
                    ax.add_patch(plt.Circle((circle['cx'],circle['cy']),circle['r'],fill=False,
                        color='cyan' if circle['source']=='observed_inlier' else 'orange',linewidth=.6))
                    ax.text(circle['cx'],circle['cy'],str(circle['parent']),color='yellow',fontsize=5)
                ax.set_title('T%d Z%d raw NPC: cyan observed fits; orange sphere predictions'%(t,z));ax.axis('off');fig.tight_layout()
                if out:fig.savefig(out/('geometry_T%d.png'%t),dpi=180,bbox_inches='tight')
                plt.show();plt.close(fig)
        if out:pd.DataFrame(rows).to_csv(out/'component_comparison_partial.csv',index=False)
    result=pd.DataFrame(rows)
    result['changed_assessable']=result.geometry_gate_pass.notna() & result.geometry_gate_pass.ne(result.old_gate_pass)
    display(pd.DataFrame(coverage));display(pd.crosstab(result.t,result.geometry_status))
    display(result.groupby('t').agg(old_pass=('old_gate_pass','sum'),
        geometry_assessable=('geometry_gate_pass','count'),geometry_pass=('geometry_gate_pass','sum'),
        containment_test_pass=('contained_gate_pass','sum'),changed_assessable=('changed_assessable','sum')))
    if out:
        result.to_csv(out/'component_comparison.csv',index=False)
        pd.DataFrame(circle_rows).to_csv(out/'droplet_circles.csv',index=False)
        pd.DataFrame(coverage).to_csv(out/'geometry_coverage.csv',index=False)
        (out/'comparison_settings.json').write_text(json.dumps(dict(timepoints=ts,min_parent_overlap=min_parent_overlap,
            geometry_signature=signature,gate_settings=gate['settings'],old_gate_source=str(gate.get('output_dir')),
            masks=str(cfg.nucleus_instance_hyperstack_path)),indent=2),encoding='utf8')
    print('No original masks or decisions changed. Missing/ambiguous geometry is unassessable, not confirmed artifact.')
    return dict(comparison=result,circles=pd.DataFrame(circle_rows),coverage=pd.DataFrame(coverage),
                output_dir=out,cache_dir=cache,settings=settings)


In [ ]:
from pathlib import Path
import sys
RANSAC_MODULE_DIR = Path('/home/tdeibert/Projects/Nuclear_Scaling/Code_Repository/Python/Model_Training')
if str(RANSAC_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(RANSAC_MODULE_DIR))
# Start with the reviewed timepoints. Use tuple(range(img_5d.shape[0])) for all T later.
RANSAC_TIMEPOINTS = (0, 1, 2)
# For a resumed run, set this to the printed geometry cache directory.
RANSAC_CACHE_DIR = None
RANSAC_DROPLET_AUDIT = audit_ransac_droplets(
    cfg, ARTIFACT_GATE,
    training_notebook=RANSAC_MODULE_DIR / 'vulcan_training_2_5_3.ipynb',
    timepoints=RANSAC_TIMEPOINTS, cache_dir=RANSAC_CACHE_DIR, save=True)
